# Acute Encephalopathy Export: Frozen Dataset v1.2 (build notebook)

This notebook builds the frozen, versioned research dataset from the two research-platform exports of Wuhan
Children's Hospital (the main export, with sheet `病案首页基本信息`, and the supplementary export, with sheet
`病案首页其他信息`). Every table in the frozen package is produced here from the raw exports, so the package can be
rebuilt and audited at any time. Downstream analyses read the frozen tables (seconds) instead of the raw workbooks.
From v1.2 the build also reads the clinician adjudication of discharges against medical advice (DAMA): the review
sheet of the previous release with a decision in every row (any `.xlsx` with sheet `dama_review`, e.g.
`dama_review_adjudicated.xlsx`). It changes only the adjudicated outcome labels.

**How to run.** Put this notebook in the folder that holds the raw exports (`.xlsx`; split exports such as one
workbook per year are accepted) and the adjudication workbook, and choose *Restart Kernel and Run All Cells*. To document
what changed, also place the previous release (`AE_freeze_v1.1/`) in the folder. The first run parses the workbooks
(about 30 s with `python-calamine`, 2-3 min with `openpyxl`) and caches them in `./cache_freeze_build/`.
Requirements: Python 3.9 or later, pandas 2.0 or later, NumPy and openpyxl (`python-calamine` is used when installed).
The tables are byte-identical across the tested environments (see `manifest.json` and the README).

**A released freeze is never overwritten.** If `./AE_freeze_v1.2/manifest.json` already exists, the notebook builds
into `./AE_freeze_v1.2_rebuild_<date-time>/` instead and reports, table by table, whether the rebuild reproduces the
checksums of the released freeze.

**Output** (`./AE_freeze_v1.2/`):

| Item | Contents |
|---|---|
| `README.md`, `manifest.json` | Overview and key counts; MD5 checksums of the raw exports and of every output file, software versions |
| `tables/` | 18 tables as UTF-8 CSV (research IDs and dates: internal use only) |
| `docs/` | `data_dictionary.xlsx` (every column), `definitions.md`, `qc_report.md`, `reconciliation_v1.md`, `reconciliation_v1.1.md` |
| `adjudication_INTERNAL/` | Review sheet for discharge against medical advice with the clinician decisions (contains short text excerpts and reviewer notes; never share) |
| `code/` | A copy of this notebook and `load_frozen.py` (typed loading and checksum verification) |

**Rules that apply to every table.**
1. Nothing is imputed or invented. A value that cannot be read, or that is physiologically impossible, is set to
   missing and the reason is recorded; the raw value stays in a `*_raw` column or in a cleaning log.
2. Unknown is coded as missing, never as "no". Variables that come only from the supplementary export (discharge
   disposition, procedures, non-drug orders, ICU transfer times, NT-proBNP and other supplement-only tests) are
   missing for admissions that the supplementary export does not cover (`in_supplement = False`).
3. Timestamps are kept as recorded. Laboratory results carry the report time only: the platform has no specimen
   collection time (confirmed with the hospital information department, October 2026).
4. Free text (discharge summaries, admission notes, imaging, EEG, ECG and echocardiography narratives) is not copied
   into the frozen tables. It is read inside this notebook only to derive documented flags; excerpts appear only in
   the internal adjudication sheet.
5. Duplicate copies of the same record in the two exports are collapsed; repeated identical rows within one export
   are collapsed with their multiplicity kept in `n_source_rows`.

In [ ]:
# Setup.
import os, re, sys, json, time, hashlib, platform, warnings, collections, shutil
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 80)

if int(pd.__version__.split('.')[0]) < 2:
    raise RuntimeError(f'pandas 2.0 or later is required (found {pd.__version__}).')

FREEZE_VERSION = 'v1.2'
NB_NAME = f"AE_freeze_build_{FREEZE_VERSION.replace('.', '_')}.ipynb"
HERE = Path.cwd()
OUT = HERE / f'AE_freeze_{FREEZE_VERSION}'
REFERENCE = None   # manifest of an already released freeze of this version, if present
if (OUT / 'manifest.json').exists():
    REFERENCE = json.loads((OUT / 'manifest.json').read_text(encoding='utf-8'))
    OUT = HERE / f'AE_freeze_{FREEZE_VERSION}_rebuild_{time.strftime("%Y%m%d_%H%M%S")}'
    print(f'A released freeze {FREEZE_VERSION} exists and is left untouched; building into {OUT.name} for comparison.')
TAB, DOC, ADJ, CODE = OUT / 'tables', OUT / 'docs', OUT / 'adjudication_INTERNAL', OUT / 'code'
CACHE = HERE / 'cache_freeze_build'
for p in (OUT, TAB, DOC, ADJ, CODE, CACHE):
    p.mkdir(parents=True, exist_ok=True)
T0 = time.time()
QC = []            # (section, check, value, expected/status, note)
TABLES = {}        # name -> DataFrame written to tables/

def qc(section, check, value, status='info', note=''):
    QC.append({'section': section, 'check': check, 'value': value, 'status': status, 'note': note})

def log(*a):
    print(*a, flush=True)

log('pandas', pd.__version__, '| numpy', np.__version__, '| python', platform.python_version())

In [ ]:
# Locate the raw exports, record their checksums, parse every sheet (cached), and stack split workbooks.
def md5_file(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

def excel_engine():
    try:
        import python_calamine  # noqa: F401
        return 'calamine'
    except ImportError:
        return 'openpyxl'

ENGINE = os.environ.get('FREEZE_EXCEL_ENGINE') or excel_engine()   # optional override, e.g. FREEZE_EXCEL_ENGINE=openpyxl

def sheet_names(path):
    return list(pd.ExcelFile(path, engine=ENGINE).sheet_names)

def read_book(path, md5):
    f = CACHE / f'{md5}_{ENGINE}_pd{pd.__version__}.pkl'
    if f.exists():
        try:
            return pd.read_pickle(f)
        except Exception:
            pass
    book = pd.read_excel(path, sheet_name=None, dtype=object, engine=ENGINE)
    pd.to_pickle(book, f)
    return book

WORKBOOKS = []
ADJ_INPUTS = []    # clinician adjudication workbook (sheet dama_review), from v1.2
ADJ_DECISION_COL = 'reviewer_decision (critical / not_critical / unsure)'
for p in sorted(HERE.glob('*.xlsx')):
    if p.name.startswith('~$'):
        continue
    try:
        names = sheet_names(p)
    except Exception:
        continue
    role = 'supplement' if '病案首页其他信息' in names else ('main' if '病案首页基本信息' in names else None)
    if role is None:
        if 'dama_review' in names:
            ADJ_INPUTS.append({'file': p.name, 'role': 'adjudication', 'bytes': p.stat().st_size, 'md5': md5_file(p), 'path': p})
        continue
    WORKBOOKS.append({'file': p.name, 'role': role, 'bytes': p.stat().st_size, 'md5': md5_file(p), 'path': p})
if not any(w['role'] == 'main' for w in WORKBOOKS) or not any(w['role'] == 'supplement' for w in WORKBOOKS):
    raise FileNotFoundError('Put the main export (sheet 病案首页基本信息) and the supplementary export '
                            '(sheet 病案首页其他信息) in this folder.')

def base_sheet(name, known):
    # Split exports may suffix a sheet name with a part number, e.g. "住院用药医嘱(2)" or "住院用药医嘱_2".
    m = re.match(r'^(.*?)[\s_\-]*[\(（]?\d{1,2}[\)）]?$', name)
    if m and m.group(1) in known:
        return m.group(1)
    return name

RAW = {'main': collections.defaultdict(list), 'supplement': collections.defaultdict(list)}
for w in WORKBOOKS:
    t = time.time()
    book = read_book(w['path'], w['md5'])
    known = set(book)
    for sh, df in book.items():
        df = df.copy()
        df.columns = [str(c).strip() for c in df.columns]
        df['_file'] = w['file']
        RAW[w['role']][base_sheet(sh, known)].append(df)
    log(f"{w['role']:<10} {w['file']}: {len(book)} sheets, {round(time.time() - t, 1)} s ({ENGINE})")

SHEETS_READ = set()   # (role, sheet) pairs read by the build; unread sheets are reported in the QC report
def sheet(role, name):
    SHEETS_READ.add((role, name))
    parts = RAW[role].get(name, [])
    if not parts:
        return pd.DataFrame()
    return pd.concat(parts, ignore_index=True, sort=False)

for w in WORKBOOKS:
    qc('0 Sources', f"{w['role']} export {w['file']}", f"{w['bytes']} bytes, MD5 {w['md5']}")
if len(ADJ_INPUTS) > 1:
    raise RuntimeError('More than one adjudication workbook (sheet dama_review) in this folder: ' + ', '.join(a['file'] for a in ADJ_INPUTS))
for a in ADJ_INPUTS:
    qc('0 Sources', f"adjudication workbook {a['file']}", f"{a['bytes']} bytes, MD5 {a['md5']}")
if not ADJ_INPUTS:
    qc('0 Sources', 'Adjudication workbook', 'none found: the final critical-DAMA label equals the rule', 'warn')

In [ ]:
# Parsing helpers (engine-independent: values may arrive as str, int, float or datetime).
FULLWIDTH = str.maketrans('０１２３４５６７８９．－＋：，（）＜＞％', '0123456789.-+:,()<>%')

def is_missing(x):
    return x is None or (isinstance(x, float) and np.isnan(x)) or x is pd.NaT or (isinstance(x, str) and x.strip() == '')

def norm_id(s):
    def f(x):
        if is_missing(x):
            return None
        if isinstance(x, (int, np.integer)):
            return str(int(x))
        if isinstance(x, (float, np.floating)):
            return str(int(x)) if float(x).is_integer() else repr(float(x))
        x = str(x).strip()
        return x[:-2] if re.fullmatch(r'\d+\.0', x) else x
    return pd.Series(s, copy=False).map(f).astype(object)

def norm_text(s):
    def f(x):
        if is_missing(x):
            return None
        return str(x).replace('　', ' ').strip() or None
    return pd.Series(s, copy=False).map(f).astype(object)

INVALID_DATES = collections.Counter()
def to_dt(s):
    """Parse timestamps; dates outside 1990-2027 (data-entry errors such as year 0119) are set to missing and counted."""
    s = pd.Series(s, copy=False)
    def f(x):
        if is_missing(x):
            return None
        x = x.strftime('%Y-%m-%d %H:%M:%S') if hasattr(x, 'strftime') else str(x).strip()
        m = re.match(r'^(\d{1,4})-', x)
        if m and not 1990 <= int(m.group(1)) <= 2027:
            INVALID_DATES['out_of_range'] += 1
            return None
        return x
    st = s.map(f)
    out = pd.to_datetime(st, format='%Y-%m-%d %H:%M:%S', errors='coerce')
    rest = out.isna() & st.notna()
    if rest.any():
        out.loc[rest] = pd.to_datetime(st[rest], format='%Y-%m-%d', errors='coerce')
    rest = out.isna() & st.notna()
    if rest.any():
        extra = {'format': 'mixed'} if int(pd.__version__.split('.')[0]) >= 2 else {}
        out.loc[rest] = pd.to_datetime(st[rest], errors='coerce', **extra)
    return out

def to_num(s):
    def f(x):
        if is_missing(x):
            return np.nan
        if isinstance(x, (int, float, np.integer, np.floating)):
            return float(x)
        x = str(x).translate(FULLWIDTH).strip()
        try:
            return float(x)
        except ValueError:
            return np.nan
    return pd.Series(s, copy=False).map(f).astype(float)

def midnight(t):
    return t.notna() & (t.dt.hour == 0) & (t.dt.minute == 0) & (t.dt.second == 0)

def row_keys(df, cols):
    """One string key per row (missing values as empty strings); identical across pandas versions."""
    key = None
    for c in cols:
        s = df[c]
        if pd.api.types.is_datetime64_any_dtype(s):
            part = s.dt.strftime('%Y-%m-%d %H:%M:%S').astype(object).where(s.notna(), '')
        else:
            part = s.map(lambda x: '' if is_missing(x) else str(x)).astype(object)
        key = part if key is None else key + '\x1f' + part
    return key

def collapse_identical(df, cols):
    """Collapse identical rows (on cols) and keep their multiplicity in n_source_rows."""
    k = row_keys(df, cols)
    n = k.map(k.value_counts())
    keep = ~k.duplicated()
    out = df.loc[keep].copy()
    out['n_source_rows'] = n[keep].astype(int).values
    return out

In [ ]:
# Wards: PICU units and other ward classes (ward names are kept in Chinese as recorded).
PICU_UNITS = [('picu_main', r'^重症医学科(?:病区)?$'), ('picu_west', r'^儿童重症监护室\s*[（(]\s*西院\s*[)）]$')]
OTHER_ICU = r'GICU|SICU|外科重症|重症监护'

def ward_unit(w):
    if is_missing(w):
        return None
    w = str(w).strip()
    for unit, pat in PICU_UNITS:
        if re.search(pat, w):
            return unit
    if re.search(OTHER_ICU, w, flags=re.I):
        return 'other_icu'
    if '新生儿' in w:
        return 'neonatal_ward'
    return 'ward'

def is_icu(unit):
    return unit in ('picu_main', 'picu_west', 'other_icu')

def tri(values, known):
    """Nullable boolean: values where known is True, missing (NA) elsewhere."""
    k = pd.Series(known).fillna(False).astype(bool)
    return pd.Series(np.asarray(values, dtype=bool), index=k.index, dtype='boolean').mask(~k)

def S(x):
    """Scalar text or None (pandas 3 stores missing strings as NaN)."""
    return None if is_missing(x) else str(x).strip()

In [ ]:
# Patients and inpatient admissions.
fp = sheet('main', '病案首页基本信息')
fp['visit_id'], fp['patient_id'] = norm_id(fp['科研就诊编号']), norm_id(fp['科研患者编号'])
dup_fp = int(fp.duplicated('visit_id').sum())
fp = fp.drop_duplicates('visit_id')
oth = sheet('supplement', '病案首页其他信息'); oth['visit_id'] = norm_id(oth['科研就诊编号'])
fp2 = sheet('supplement', '病案首页基本信息'); fp2['visit_id'] = norm_id(fp2['科研就诊编号'])
vinfo = sheet('supplement', '患者就诊信息'); vinfo['visit_id'] = norm_id(vinfo['科研就诊编号']); vinfo['patient_id'] = norm_id(vinfo['科研患者编号'])
qc('1 Admissions', 'Duplicate admission rows in 病案首页基本信息 (collapsed)', dup_fp, 'pass' if dup_fp == 0 else 'warn')
qc('1 Admissions', 'Duplicate rows in 病案首页其他信息', int(oth.duplicated('visit_id').sum()), 'pass' if not oth.duplicated('visit_id').any() else 'warn')
oth, fp2 = oth.drop_duplicates('visit_id'), fp2.drop_duplicates('visit_id')

SEX = {'男性': 'male', '女性': 'female', '男': 'male', '女': 'female'}
SOURCE = {'门诊': 'outpatient_clinic', '急诊': 'emergency', '其他医疗机构转入': 'transfer_other_hospital', '其他': 'other'}
DISPOSITION = {'医嘱离院': 'discharged', '非医嘱离院': 'dama', '医嘱转院': 'transfer_hospital',
               '医嘱转社区卫生服务机构/乡镇卫生院': 'transfer_community', '死亡': 'death', '其他': 'other'}
OUTCOME = {'治愈': 'cured', '好转': 'improved', '未愈': 'not_improved', '死亡': 'death', '其他': 'other'}

V = pd.DataFrame({'visit_id': fp.visit_id, 'patient_id': fp.patient_id})
V['in_supplement'] = V.visit_id.isin(set(oth.visit_id))
V['admit_time'] = to_dt(fp['入院日期']).values
V['discharge_time'] = to_dt(fp['出院日期']).values
V['discharge_time_missing'] = V.discharge_time.isna()
V['sex'] = norm_text(fp['性别']).map(SEX).values
V['age_years'] = to_num(fp['年龄（岁）']).values
V['age_months'] = to_num(fp['年龄（月）']).values
V['age_days'] = to_num(fp['年龄（天）']).values
V['admit_ward'] = norm_text(fp['入院科别']).values
V['discharge_ward'] = norm_text(fp['出院科别']).values
V['transfer_path'] = norm_text(fp['转科科别']).values
V['admit_unit'] = V.admit_ward.map(ward_unit)
V['discharge_unit'] = V.discharge_ward.map(ward_unit)
m = V.merge(fp2[['visit_id', '入院途径', '实际住院天数']], on='visit_id', how='left') \
     .merge(oth[['visit_id', '离院方式', '是否抢救', '抢救次数', '成功次数']], on='visit_id', how='left') \
     .merge(vinfo[['visit_id', '转归情况', '入院时间', '出院时间']], on='visit_id', how='left')
V['admission_source_raw'] = norm_text(m['入院途径']).values
V['admission_source'] = V.admission_source_raw.map(SOURCE)
V['disposition_raw'] = norm_text(m['离院方式']).values
V['disposition'] = V.disposition_raw.map(DISPOSITION)
V['outcome_raw'] = norm_text(m['转归情况']).values
V['outcome'] = V.outcome_raw.map(OUTCOME)
V['los_days_recorded'] = to_num(m['实际住院天数']).values
V['los_hours'] = ((V.discharge_time - V.admit_time).dt.total_seconds() / 3600).round(2)
resc = norm_text(m['是否抢救'])
V['rescue_any'] = resc.map({'是': True, '否': False}).astype('boolean').values
V['rescue_count'] = to_num(m['抢救次数']).values
V['rescue_success_count'] = to_num(m['成功次数']).values
V['admit_year'] = V.admit_time.dt.year
# Supplement admission/discharge times must agree with the main export.
d_adm = (to_dt(m['入院时间']) - V.admit_time).dt.total_seconds().abs()
d_dis = (to_dt(m['出院时间']) - V.discharge_time).dt.total_seconds().abs()
qc('1 Admissions', 'Inpatient admissions (main export)', len(V))
qc('1 Admissions', 'Admissions covered by the supplementary export', int(V.in_supplement.sum()))
qc('1 Admissions', 'Supplement admission time differs from main export', int((d_adm > 60).sum()), 'pass' if (d_adm > 60).sum() == 0 else 'fail')
qc('1 Admissions', 'Supplement discharge time differs from main export', int((d_dis > 60).sum()), 'pass' if (d_dis > 60).sum() == 0 else 'fail')
qc('1 Admissions', 'Admissions without a discharge time (still in hospital at export or missing)', int(V.discharge_time_missing.sum()), 'info')
qc('1 Admissions', 'Discharge before admission', int((V.discharge_time < V.admit_time).sum()), 'pass' if (V.discharge_time < V.admit_time).sum() == 0 else 'fail')
qc('1 Admissions', 'Unmapped values: sex / admission source / disposition / outcome',
   f"{int(V.sex.isna().sum())} / {int((V.admission_source_raw.notna() & V.admission_source.isna()).sum())} / "
   f"{int((V.disposition_raw.notna() & V.disposition.isna()).sum())} / {int((V.outcome_raw.notna() & V.outcome.isna()).sum())}",
   'pass' if V.sex.notna().all() and not (V.disposition_raw.notna() & V.disposition.isna()).any() else 'warn')
qc('1 Admissions', 'Disposition recorded (supplement admissions)', f"{int(V.disposition.notna().sum())} of {int(V.in_supplement.sum())}")
qc('1 Admissions', 'Age in days vs years disagree by > 0.05 y', int(((V.age_days / 365.25 - V.age_years).abs() > 0.05).sum()), 'info')

# Patients
pm = sheet('main', '患者基本信息'); pm['patient_id'] = norm_id(pm['科研患者编号'])
ps = sheet('supplement', '患者基本信息'); ps['patient_id'] = norm_id(ps['科研患者编号'])
WUHAN_DISTRICTS = ['江岸区', '江汉区', '硚口区', '汉阳区', '武昌区', '青山区', '洪山区', '东西湖区', '汉南区', '蔡甸区', '江夏区', '黄陂区', '新洲区']
P = pd.DataFrame({'patient_id': sorted(set(V.patient_id) | set(pm.patient_id) | set(ps.patient_id))})
sex_by_patient = V.groupby('patient_id').sex.agg(lambda s: '|'.join(sorted(set(s.dropna()))))
P['sex'] = P.patient_id.map(sex_by_patient).replace('', None)
P['ethnicity_cn'] = P.patient_id.map(ps.drop_duplicates('patient_id').set_index('patient_id')['民族'].pipe(norm_text))
county = pd.concat([pm.set_index('patient_id')['户口地址-县'], ps.set_index('patient_id')['户口地址-县']]).pipe(norm_text)
county = county[~county.index.duplicated()]
P['home_county_cn'] = P.patient_id.map(county)
P['residence_wuhan'] = P.home_county_cn.map(lambda c: None if is_missing(c) else c in WUHAN_DISTRICTS).astype('boolean')
P['n_admissions'] = P.patient_id.map(V.groupby('patient_id').size()).fillna(0).astype(int)
P['first_admit_time'] = P.patient_id.map(V.groupby('patient_id').admit_time.min())
qc('1 Admissions', 'Patients', len(P))
qc('1 Admissions', 'Patients with conflicting sex across admissions', int(P.sex.fillna('').str.contains(r'\|').sum()), 'pass' if not P.sex.fillna('').str.contains(r'\|').any() else 'fail')

# Outpatient and emergency visits listed in the supplementary export
VO = vinfo[vinfo['入院时间'].isna()].copy()
VO = pd.DataFrame({'visit_id': VO.visit_id.values, 'patient_id': VO.patient_id.values,
                   'visit_time': to_dt(VO['门诊就诊时间']).values, 'department_cn': norm_text(VO['门诊就诊科室']).values,
                   'age_years': to_num(VO['年龄（岁）']).values, 'age_days': to_num(VO['年龄（天）']).values})
VO = VO.drop_duplicates('visit_id')
qc('1 Admissions', 'Outpatient/emergency visits listed', len(VO))
log('Admissions:', len(V), '| supplement-covered:', int(V.in_supplement.sum()), '| patients:', len(P), '| outpatient visits:', len(VO))

In [ ]:
# Transfers between wards and ICU stays.
tr = sheet('supplement', '患者转科信息')
TR = pd.DataFrame({'visit_id': norm_id(tr['科研就诊编号']), 'patient_id': norm_id(tr['科研患者编号']),
                   'transfer_time': to_dt(tr['转科时间']), 'from_ward': norm_text(tr['转出科室']), 'to_ward': norm_text(tr['转入科室'])})
n_raw = len(TR)
TR = collapse_identical(TR, ['visit_id', 'transfer_time', 'from_ward', 'to_ward'])
TR = TR.merge(V[['visit_id', 'admit_time', 'discharge_time', 'admit_ward']], on='visit_id', how='left')
TR = TR.sort_values(['visit_id', 'transfer_time', 'from_ward', 'to_ward'], na_position='last').reset_index(drop=True)
TR['transfer_seq'] = TR.groupby('visit_id').cumcount() + 1
TR['from_unit'] = TR.from_ward.map(ward_unit)
TR['to_unit'] = TR.to_ward.map(ward_unit)
TR['flag_unknown_visit'] = TR.admit_time.isna()
TR['minutes_before_admission'] = ((TR.admit_time - TR.transfer_time).dt.total_seconds() / 60).where(TR.transfer_time < TR.admit_time)
TR['flag_after_discharge'] = TR.transfer_time > TR.discharge_time
TR['flag_before_admission'] = TR.transfer_time < TR.admit_time
TR['flag_to_ward_missing'] = TR.to_ward.isna()
TR['flag_missing_time'] = TR.transfer_time.isna()
# Transfers recorded up to 24 h before the admission time are treated as at admission; later than discharge are not used.
TR['used_in_timeline'] = ~TR.flag_unknown_visit & ~TR.flag_missing_time & ~TR.flag_after_discharge & \
                         ~(TR.flag_before_admission & (TR.minutes_before_admission > 24 * 60))
TR['timeline_time'] = TR.transfer_time.where(~TR.flag_before_admission, TR.admit_time)
qc('2 Transfers', 'Transfer records (raw / after collapsing identical rows)', f'{n_raw} / {len(TR)}')
qc('2 Transfers', 'Transfers for admissions not in the main export', int(TR.flag_unknown_visit.sum()), 'warn' if TR.flag_unknown_visit.any() else 'pass')
qc('2 Transfers', 'Transfers after hospital discharge (not used)', int(TR.flag_after_discharge.sum()), 'info',
   'Mostly paired records on the discharge day; excluded from the ward timeline.')
qc('2 Transfers', 'Transfers before admission (<= 24 h: set to admission time)', int(TR.flag_before_admission.sum()), 'info')
qc('2 Transfers', 'Transfers with missing destination ward (destination treated as unknown)', int(TR.flag_to_ward_missing.sum()), 'info')

# Ward timeline per admission -> ICU segments
GENERIC = {'内科', '外科'}
stays, mismatch, mismatch_dc = [], 0, 0
TRg = {k: g for k, g in TR[TR.used_in_timeline].groupby('visit_id')}
for v in V.itertuples(index=False):
    events = [(v.admit_time, S(v.admit_ward), 'hospital_admission')]
    g = TRg.get(v.visit_id)
    if g is not None:
        seen = set()
        for e in g.itertuples(index=False):
            tw, fw = S(e.to_ward), S(e.from_ward)
            events.append((e.timeline_time, tw if tw is not None else '__unknown__', 'transfer'))
            prev = events[-2][1]
            if fw is not None and prev not in ('__unknown__', None) and fw != prev and fw not in GENERIC:
                mismatch += 1
                mismatch_dc += int(e.timeline_time == v.discharge_time and (fw, tw) in seen)
            seen.add((fw, tw))
    segs, cur_in, cur_unit, cur_ward, cur_src = [], None, None, None, None
    for t, w, src in events:
        u = ward_unit(w) if w not in ('__unknown__', None) else None
        if cur_in is not None and (not is_icu(u) or u != cur_unit):
            segs.append({'unit': cur_unit, 'ward_cn': cur_ward, 'in_time': cur_in, 'in_source': cur_src, 'out_time': t, 'out_source': 'transfer'})
            cur_in = None
        if is_icu(u) and cur_in is None:
            cur_in, cur_unit, cur_ward, cur_src = t, u, w, src
    if cur_in is not None:
        if S(v.discharge_unit) == cur_unit and not pd.isna(v.discharge_time):
            segs.append({'unit': cur_unit, 'ward_cn': cur_ward, 'in_time': cur_in, 'in_source': cur_src, 'out_time': v.discharge_time, 'out_source': 'hospital_discharge'})
        else:
            segs.append({'unit': cur_unit, 'ward_cn': cur_ward, 'in_time': cur_in, 'in_source': cur_src, 'out_time': pd.NaT, 'out_source': 'unknown',
                         'out_time_upper_bound': v.discharge_time})
    elif is_icu(S(v.discharge_unit)):
        # Discharged from an ICU without a recorded transfer into it.
        segs.append({'unit': S(v.discharge_unit), 'ward_cn': S(v.discharge_ward), 'in_time': pd.NaT, 'in_source': 'unknown',
                     'in_time_lower_bound': events[-1][0], 'out_time': v.discharge_time, 'out_source': 'hospital_discharge'})
    for s in segs:
        s['visit_id'] = v.visit_id
        s['source'] = 'transfer_records' if v.in_supplement else 'ward_fields_only'
    tp = S(v.transfer_path)
    if not segs and tp and re.search(r'重症医学科|儿童重症监护室', tp):
        segs.append({'visit_id': v.visit_id, 'unit': 'picu_west' if '儿童重症监护室' in tp else 'picu_main',
                     'ward_cn': None, 'in_time': pd.NaT, 'in_source': 'unknown', 'out_time': pd.NaT, 'out_source': 'unknown',
                     'source': 'transfer_path_text_only'})
    stays.extend(segs)
IS = pd.DataFrame(stays)
for c in ['in_time_lower_bound', 'out_time_upper_bound']:
    if c not in IS:
        IS[c] = pd.NaT
IS = IS.merge(V[['visit_id', 'patient_id', 'admit_time', 'discharge_time', 'age_days', 'in_supplement']], on='visit_id', how='left')
# For non-supplement admissions the timeline has no transfer times: an ICU stay that starts at admission and would end at
# discharge ends at an unknown time when the ward path text shows the child leaving an ICU. (A stay that ends with
# discharge from an ICU but whose entry is unknown keeps the discharge time as its exit.)
nos = ~IS.in_supplement & IS.source.eq('ward_fields_only')
path = IS.visit_id.map(V.set_index('visit_id').transfer_path)
left_icu = path.fillna('').str.contains(r'(?:重症医学科|儿童重症监护室)[^>]*->')
fix = nos & IS.out_source.eq('hospital_discharge') & IS.in_source.eq('hospital_admission') & left_icu
IS.loc[fix, 'out_time_upper_bound'] = IS.loc[fix, 'out_time']
IS.loc[fix, ['out_time']] = pd.NaT
IS.loc[fix, 'out_source'] = 'unknown'
IS = IS.sort_values(['visit_id', 'in_time'], na_position='last').reset_index(drop=True)
IS['stay_seq'] = IS.groupby('visit_id').cumcount() + 1
IS['timing_complete'] = IS.in_time.notna() & IS.out_time.notna()
IS['los_hours'] = ((IS.out_time - IS.in_time).dt.total_seconds() / 3600).round(2)
IS['hours_admission_to_icu'] = ((IS.in_time - IS.admit_time).dt.total_seconds() / 3600).round(2)
IS['age_days_at_icu_in'] = (IS.age_days + (IS.in_time - IS.admit_time).dt.total_seconds() / 86400).round(1)
# ICU episodes: contiguous segments in different ICU units (transfer between ICUs) form one episode.
IS['icu_episode'] = 1
for vid, g in IS.groupby('visit_id'):
    ep = 1
    prev_out = None
    for i in g.index:
        if prev_out is not None and not pd.isna(IS.at[i, 'in_time']) and IS.at[i, 'in_time'] != prev_out:
            ep += 1
        IS.at[i, 'icu_episode'] = ep
        prev_out = IS.at[i, 'out_time']
IS = IS.drop(columns=['admit_time', 'discharge_time', 'age_days'])
qc('2 Transfers', 'Transfers whose origin ward differs from the ward in the timeline (generic 内科/外科 excluded) / of these, repeats of an earlier transfer of the admission recorded at the discharge time',
   f'{mismatch} / {mismatch_dc}', 'info', 'The origin ward is not used to build the timeline.')
qc('3 ICU stays', 'ICU stays (all units) / admissions with any ICU stay', f'{len(IS)} / {IS.visit_id.nunique()}')
qc('3 ICU stays', 'ICU stays by unit', IS.unit.value_counts().to_dict())
qc('3 ICU stays', 'ICU stays by source', IS.source.value_counts().to_dict())
qc('3 ICU stays', 'Stays with known entry and exit times', f"{int(IS.timing_complete.sum())} of {len(IS)}")
qc('3 ICU stays', 'Exit time unknown (no transfer out recorded; upper bound = discharge)', int(IS.out_source.eq('unknown').sum()), 'info')
qc('3 ICU stays', 'Exit before entry', int((IS.los_hours <= 0).sum()), 'pass' if (IS.los_hours <= 0).sum() == 0 else 'fail')
log('ICU stays:', len(IS), '| admissions:', IS.visit_id.nunique(), '| complete timing:', int(IS.timing_complete.sum()))

In [ ]:
# Diagnoses: coded discharge diagnoses (supplement) and diagnosis lists from the discharge records and admission notes.
dc = sheet('supplement', '病案出院诊断归一')
DXC = pd.DataFrame({'visit_id': norm_id(dc['科研就诊编号']), 'diagnosis_cn': norm_text(dc['诊断疾病名称']),
                    'icd10_code': norm_text(dc['诊断疾病编码'])})
DXC['_order'] = np.arange(len(DXC))
n0 = len(DXC)
DXC = DXC.drop_duplicates(['visit_id', 'diagnosis_cn', 'icd10_code']).sort_values(['visit_id', '_order'])
DXC['dx_seq'] = DXC.groupby('visit_id').cumcount() + 1
DXC['icd10_3char'] = DXC.icd10_code.fillna('').str.upper().str.extract(r'^([A-Z]\d{2})', expand=False)
DXC = DXC.drop(columns='_order').reset_index(drop=True)
qc('4 Diagnoses', 'Coded discharge diagnoses (raw / unique per admission)', f'{n0} / {len(DXC)}')
qc('4 Diagnoses', 'Coded diagnoses without a valid ICD-10 category', int(DXC.icd10_3char.isna().sum()), 'info')

NARRATIVE = re.compile(r'入院后|患者因|患儿因|患儿|入院查体|(?<!健康)查体|完善相关检查|主诉|现病史|。|\(10\^|因[“"]|\d+岁|入院')   # 健康查体 is a diagnosis
def split_top(s):
    """Split on commas and semicolons that are not inside brackets."""
    out, buf, depth = [], '', 0
    for ch in s:
        if ch in '(（[［':
            depth += 1
        elif ch in ')）]］':
            depth = max(0, depth - 1)
        if ch in ',，;；' and depth == 0:
            out.append(buf); buf = ''
        else:
            buf += ch
    out.append(buf)
    return out

def dx_items(text):
    """Split a diagnosis list such as '入院诊断1:急性胃肠炎\\n入院诊断2:不明原因抽搐' into items. Narrative text that sometimes follows the
    list in the same field (hospital course, examination) is cut off at the first narrative marker and flagged."""
    if is_missing(text):
        return []
    s = str(text).replace('\r', '\n').replace('\u3000', ' ')
    s = re.sub(r'(入院诊断|出院诊断|初步诊断|修正诊断|补充诊断)\s*\d*\s*[:：]', '\n§DX§', s)
    s = re.sub(r'诊断备注\s*\d*\s*[:：]', '\n§REMARK§', s)
    items, stop = [], False
    for line in s.split('\n'):
        if stop:
            break
        kind = 'remark' if line.startswith('§REMARK§') else 'diagnosis'
        line = line.replace('§DX§', '').replace('§REMARK§', '').strip()
        for part in re.split(r'(?:(?<=^)|(?<=[^\d\.]))\d{1,2}\s*[\.．、]\s*(?=\D)', line):
            for q in split_top(part):
                q = q.strip(' ;；,，。')
                truncated = False
                m = NARRATIVE.search(q)
                if m:
                    q, truncated, stop = q[:m.start()].strip(' ;；,，。：:'), True, True
                if q:
                    items.append((kind, q, truncated))
                if stop:
                    break
            if stop:
                break
    return items

rows = []
# Discharge records: the main export holds them for all admissions; the supplementary copies are identical except for admissions whose
# record exists only in the supplementary export (those records are added; the supplementary sheet has no admission-diagnosis column).
_drm, _drs = sheet('main', '住院病历出院记录'), sheet('supplement', '住院病历出院记录')
_common = [c for c in _drs.columns if c in _drm.columns and c != '_file']
_ks = row_keys(_drs, _common)
_extra = _drs[~_ks.isin(set(row_keys(_drm, _common)))]
DR = pd.concat([_drm.assign(_src='main'), _extra.assign(_src='supplement')], ignore_index=True, sort=False)
DR['visit_id'] = norm_id(DR['科研就诊编号'])
DR_EXTRA = len(_extra)
qc('4 Diagnoses', 'Discharge records: main export / supplementary records identical to a main record / supplementary records added (no main record)',
   f'{len(_drm)} / {len(_drs) - len(_extra)} / {len(_extra)}', 'info')
dr = DR.copy(); dr['_rec'] = dr.groupby('visit_id').cumcount() + 1
for col, src in [('入院诊断', 'discharge_record_admission_dx'), ('出院诊断', 'discharge_record_discharge_dx')]:
    if col in dr:
        for vid, rec, text in zip(dr.visit_id, dr._rec, dr[col]):
            for i, (kind, item, trunc) in enumerate(dx_items(text), 1):
                rows.append((vid, src, rec, i, kind, item, trunc))
an = sheet('supplement', '儿科入院记录')
if len(an):
    an['visit_id'] = norm_id(an['科研就诊编号']); an['_rec'] = an.groupby('visit_id').cumcount() + 1
    for vid, rec, text in zip(an.visit_id, an._rec, an['初步诊断']):
        for i, (kind, item, trunc) in enumerate(dx_items(text), 1):
            rows.append((vid, 'admission_note_preliminary_dx', rec, i, kind, item, trunc))
# Outpatient and emergency records (门急诊病历, supplement): preliminary diagnoses only; visit_id is the outpatient visit number.
op = sheet('supplement', '门急诊病历')
if len(op):
    op['visit_id'] = norm_id(op['科研就诊编号']); op['_rec'] = op.groupby('visit_id').cumcount() + 1
    for vid, rec, text in zip(op.visit_id, op._rec, op['初步诊断']):
        for i, (kind, item, trunc) in enumerate(dx_items(text), 1):
            rows.append((vid, 'outpatient_record_preliminary_dx', rec, i, kind, item, trunc))
# Front-page discharge diagnoses as entered before coding (病案出院诊断编目前; the names are exported in the column 出院诊断备注).
# 'dh' replaces the whole list of some admissions and is a placeholder, not a diagnosis.
pc = sheet('supplement', '病案出院诊断编目前')
n_dh = 0
if len(pc):
    pc = pd.DataFrame({'visit_id': norm_id(pc['科研就诊编号']), 'text': norm_text(pc['出院诊断备注'])})
    is_dh = pc.text.fillna('').str.lower().eq('dh')
    n_dh = int(pc[is_dh].visit_id.nunique())
    pc = pc[~is_dh & pc.text.notna()]
    pc['_i'] = pc.groupby('visit_id').cumcount() + 1
    rows += [(v, 'front_page_discharge_dx_uncoded', 1, i, 'diagnosis', t, False) for v, i, t in zip(pc.visit_id, pc._i, pc.text)]
DXT = pd.DataFrame(rows, columns=['visit_id', 'source', 'record_seq', 'item_seq', 'item_type', 'text_cn', 'narrative_cut'])
qc('4 Diagnoses', 'Uncoded front-page discharge diagnoses: admissions whose list is the placeholder "dh" (not used)', n_dh, 'info')
qc('4 Diagnoses', 'Items followed by narrative text in the same field (narrative removed)', int(DXT.narrative_cut.sum()), 'info')
qc('4 Diagnoses', 'Longest diagnosis item (characters)', int(DXT.text_cn.str.len().max()), 'info')
qc('4 Diagnoses', 'Diagnosis-list items by source', DXT.source.value_counts().to_dict())
qc('4 Diagnoses', 'Admissions with an admission-diagnosis list (discharge record or admission note)',
   int(DXT[DXT.source.isin(['discharge_record_admission_dx', 'admission_note_preliminary_dx'])].visit_id.nunique()))

In [ ]:
# Procedures (ICD-9-CM-3, supplement). Times at exactly 00:00:00 carry the date only.
pr = sheet('supplement', '病案手术操作归一')
PR = pd.DataFrame({'visit_id': norm_id(pr['科研就诊编号']), 'proc_time': to_dt(pr['手术操作日期']),
                   'proc_code': norm_text(pr['手术操作编码']), 'proc_name_cn': norm_text(pr['手术操作名称'])})
n0 = len(PR)
PR = collapse_identical(PR, ['visit_id', 'proc_time', 'proc_code', 'proc_name_cn'])
PR['time_precision'] = np.where(PR.proc_time.isna(), None, np.where(midnight(PR.proc_time), 'date', 'minute'))
PROC_RULES = [   # category, code pattern, name pattern
    ('intubation', r'^96\.04', r'气管内插管'),
    ('imv', r'^96\.7[0-2]', r'侵入性机械性通气|呼吸机治疗'),
    ('extubation', r'^97\.37', r'气管插管去除'),
    ('niv', r'^93\.90', r'无创'),
    ('oxygen', r'^93\.96', r'^吸氧$'),
    ('hyperbaric_oxygen', r'^93\.95', r'高压给氧'),
    ('tracheostomy', r'^31\.[12]', r'气管切开'),
    ('plasma_exchange', r'^39\.95x00[45]|^99\.0702', r'血浆置换'),
    ('rrt', r'^39\.95|^54\.98', r'透析|CRRT|肾脏替代|血液滤过'),
    ('hemoperfusion', r'^39\.95x00[0-3]', r'血液灌流'),
    ('ecmo', r'^39\.65', r'体外膜'),
    ('cpr', r'^99\.6[03]', r'心肺复苏|心外按压'),
    ('defibrillation_cardioversion', r'^99\.62', r'除颤|电复律'),
    ('exchange_transfusion', r'^99\.01', r'换血|血置换'),
    ('transfusion', r'^99\.0[2-9]', r'输血|输入'),
    ('central_venous_catheter', r'^38\.93|^38\.97', r'中心静脉|深静脉'),
    ('arterial_catheter', r'^38\.91|^38\.98', r'动脉'),
    ('lumbar_puncture', r'^03\.31', r'腰椎穿刺|腰穿'),
    ('video_eeg', r'^89\.19', r'脑电'),
]
def proc_category(code, name):
    code, name = code or '', name or ''
    for cat, cp, npat in PROC_RULES:
        if re.search(cp, code):
            return cat
    for cat, cp, npat in PROC_RULES:
        if npat and re.search(npat, name):
            return cat
    return 'other'
PR['proc_category'] = [proc_category(c, n) for c, n in zip(PR.proc_code, PR.proc_name_cn)]
PR = PR.sort_values(['visit_id', 'proc_time', 'proc_code'], na_position='last').reset_index(drop=True)
PR['proc_seq'] = PR.groupby('visit_id').cumcount() + 1
qc('5 Procedures', 'Procedure records (raw / after collapsing identical rows)', f'{n0} / {len(PR)}')
qc('5 Procedures', 'Procedure times at 00:00:00 (date only)', f"{int(PR.time_precision.eq('date').sum())} of {len(PR)}", 'info',
   'Date-only times are not used to place an event before or after a clock time.')
qc('5 Procedures', 'Procedures by support category', PR.proc_category.value_counts().head(14).to_dict())

In [ ]:
# Drug orders. The supplementary export repeats the main export's orders for the admissions it covers (no start time);
# the main export (all admissions) is used and the overlap is verified.
dm_ = sheet('main', '住院用药医嘱'); ds_ = sheet('supplement', '住院用药医嘱')
def drug_frame(d, has_start):
    return pd.DataFrame({'visit_id': norm_id(d['科研就诊编号']), 'order_time': to_dt(d['住院用药医嘱开立时间']),
                         'start_time': to_dt(d['住院用药医嘱开始时间']) if has_start else pd.NaT,
                         'end_time': to_dt(d['住院用药医嘱结束时间']), 'drug_name_cn': norm_text(d['住院用药医嘱药品名称']),
                         'route_raw': norm_text(d['住院用药医嘱用法'])})
DM, DS = drug_frame(dm_, True), drug_frame(ds_, False)
KEY = ['visit_id', 'order_time', 'drug_name_cn', 'route_raw', 'end_time']
kM = set(row_keys(DM, KEY)); kS = row_keys(DS, KEY)
supp_only = DS[~kS.isin(kM)]
qc('6 Drug orders', 'Supplementary-export orders not present in the main export', len(supp_only), 'pass' if len(supp_only) == 0 else 'warn',
   'When > 0 these rows are added.')
qc('6 Drug orders', 'Main-export start time equals order time',
   f"{int((DM.start_time == DM.order_time).sum())} of {int(DM.start_time.notna().sum())}", 'info')
D = pd.concat([DM, supp_only.assign(start_time=pd.NaT)], ignore_index=True)
n0 = len(D)
D = collapse_identical(D, KEY + ['start_time'])
qc('6 Drug orders', 'Drug-order rows (main export / unique orders)', f'{n0} / {len(D)}', 'info',
   'Fully identical rows (same admission, order, start and end time, drug and route) are collapsed; n_source_rows keeps the count.')

ROUTES = [
    ('iv_infusion', r'^静脉输液|^静脉滴注|^静脉输血'), ('iv_push', r'^静脉注射|^静推'), ('intraarterial', r'^动脉'),
    ('im', r'^肌肉注射|^肌注|^肌内注射'), ('sc', r'^皮下注射'), ('intradermal', r'^皮内注射|^皮试'),
    ('enteral', r'^口服|^鼻饲|^胃管注入|^鼻肠管注入|^冲服|^嚼服|^含服|^舌下含服|^煎服|^代茶饮|^按说明服'),
    ('inhaled', r'雾化|^经口吸入|^呼吸机用'), ('endotracheal', r'^气管内'),
    ('rectal', r'^塞肛|^挤入肛门|^灌肠|^结肠水疗'), ('intrathecal', r'^鞘内'),
    ('topical_local', r'^外用|^涂|^滴|^喷|^漱口|^外敷|^外贴|^冲洗|^局部|^阴道|^眼内'),
    ('dialysis_related', r'透析|血透'), ('flush_lock', r'^冲管|^封管'), ('discharge_medication', r'^出院带药'),
    ('procedure_related', r'穿刺|麻醉|神经阻滞|腹腔|胸腔|体外循环|洗胃|口腔内'),
    ('unspecified', r'^无$|^遵医嘱$|^科内治疗$|^领药$'),
]
def route_class(r):
    if r is None:
        return 'unspecified'
    for cls, pat in ROUTES:
        if re.search(pat, r):
            return cls
    return 'other'
route_map = {r: route_class(r) for r in D.route_raw.dropna().unique()}
D['route_class'] = D.route_raw.map(route_map).fillna('unspecified')
AGENTS = [   # order matters: specific catecholamine names before epinephrine, dobutamine before dopamine
    ('norepinephrine', r'去甲肾上腺素|正肾'), ('isoproterenol', r'异丙肾上腺素'), ('phenylephrine', r'去氧肾上腺素|苯肾上腺素|新福林'),
    ('epinephrine', r'肾上腺素|付肾'), ('dobutamine', r'多巴酚丁胺'), ('dopamine', r'多巴胺'), ('milrinone', r'米力农'),
    ('levosimendan', r'左西孟旦'), ('vasopressin', r'精氨酸加压素|血管加压素'), ('pituitrin', r'垂体后叶'),
    ('terlipressin', r'特利加压素'), ('desmopressin', r'去氨加压素'), ('metaraminol', r'间羟胺'),
]
NOT_AGENT = r'卡因|左旋多巴|多巴丝肼|肾上腺皮质'
def agent(name):
    if name is None or re.search(NOT_AGENT, name):
        return None
    for a, pat in AGENTS:
        if re.search(pat, name):
            return a
    return None
agent_map = {n: agent(n) for n in D.drug_name_cn.dropna().unique()}
D['vasoactive_agent'] = D.drug_name_cn.map(agent_map)
VASOACTIVE_AGENTS = ['epinephrine', 'norepinephrine', 'dopamine', 'dobutamine', 'milrinone', 'vasopressin']
# Compound infusion orders: a line with route "无" counts as an infusion only when a diluent line (sodium chloride,
# glucose or water for injection) is ordered in the same admission at the same minute.
DIL = D.drug_name_cn.fillna('').str.contains(r'氯化钠|葡萄糖|注射用水') & D.route_class.isin(['iv_infusion', 'unspecified'])
dil_keys = set(zip(D.visit_id[DIL], D.order_time[DIL]))
D['compound_with_diluent'] = [(v, t) in dil_keys for v, t in zip(D.visit_id, D.order_time)]
D['vasoactive_infusion'] = D.vasoactive_agent.isin(VASOACTIVE_AGENTS) & (
    D.route_class.eq('iv_infusion') | (D.route_raw.eq('无') & D.compound_with_diluent))
D['pituitrin_infusion'] = D.vasoactive_agent.eq('pituitrin') & (
    D.route_class.eq('iv_infusion') | (D.route_raw.eq('无') & D.compound_with_diluent))
CLASSES = [
    ('sedative_analgesic', r'咪达唑仑|力月西|右美托咪定|丙泊酚|异丙酚|芬太尼|舒芬太尼|瑞芬太尼|吗啡|氯胺酮|水合氯醛', None),
    ('neuromuscular_blocker', r'维库溴铵|罗库溴铵|顺阿曲库铵|阿曲库铵|泮库溴铵|琥珀酰?胆碱|米库氯铵', None),
    ('antiseizure', r'左乙拉西坦|丙戊酸|苯巴比妥|苯妥英|拉考沙胺|托吡酯|奥卡西平|卡马西平|吡仑帕奈|地西泮|劳拉西泮|氯硝西泮', None),
    ('corticosteroid', r'甲泼尼龙|甲强龙|甲基强的松龙|地塞米松|氢化可的松|泼尼松|强的松', None),
    ('immunoglobulin', r'免疫球蛋白|丙种球蛋白', r'乙型肝炎|乙肝|破伤风|狂犬|抗D|胸腺|抗人T'),
    ('il6_receptor_blocker', r'托珠单抗', None),
    ('osmotherapy', r'甘露醇|甘油果糖|3%氯化钠', None),   # 10% and concentrated sodium chloride are electrolyte additives; excluded
    ('albumin', r'白蛋白', None),
]
def classes(name):
    if name is None:
        return None
    out = [c for c, inc, exc in CLASSES if re.search(inc, name) and not (exc and re.search(exc, name))]
    return '|'.join(out) if out else None
cls_map = {n: classes(n) for n in D.drug_name_cn.dropna().unique()}
D['drug_classes'] = D.drug_name_cn.map(cls_map)
D = D.sort_values(['visit_id', 'order_time', 'drug_name_cn', 'route_raw'], na_position='last').reset_index(drop=True)
D['order_seq'] = D.groupby('visit_id').cumcount() + 1
qc('6 Drug orders', 'Route classes', D.route_class.value_counts().to_dict())
qc('6 Drug orders', 'Orders of recognized vasoactive and related agents (any route), by agent', D.vasoactive_agent.value_counts().to_dict(), 'info',
   'Recognition only; the vasoactive-infusion definition uses six agents and continuous intravenous routes (see definitions).')
qc('6 Drug orders', 'Orders meeting the vasoactive-infusion definition / admissions',
   f"{int(D.vasoactive_infusion.sum())} / {D[D.vasoactive_infusion].visit_id.nunique()}")
qc('6 Drug orders', 'Vasoactive-agent orders excluded by route (bolus, endotracheal, inhaled, topical, enteral, other)',
   D[D.vasoactive_agent.isin(VASOACTIVE_AGENTS) & ~D.vasoactive_infusion].route_class.value_counts().to_dict(), 'info')
qc('6 Drug orders', 'Drug-class matches (orders)', {c: int(D.drug_classes.fillna('').str.contains(c).sum()) for c, _, _ in CLASSES})

In [ ]:
# Non-drug orders (supplement) with support categories.
nd = sheet('supplement', '住院非用药医嘱')
N = pd.DataFrame({'visit_id': norm_id(nd['科研就诊编号']), 'start_time': to_dt(nd['住院非用药医嘱开始时间']),
                  'end_time': to_dt(nd['住院非用药医嘱结束时间']), 'order_name_cn': norm_text(nd['住院非用药医嘱名称'])})
n0 = len(N)
N = collapse_identical(N, ['visit_id', 'start_time', 'end_time', 'order_name_cn'])
N['is_instruction'] = N.order_name_cn.fillna('').str.startswith('\\')
ND_RULES = [   # first match wins
    ('planned_procedure', r'^\\*拟'),
    ('extubation', r'拔(?:除|出)?气管(?:插管|导管)|气管插管去除'),
    ('imaging_or_endoscopy', r'(?-i:CT|MRI)|磁共振|超声|X线|摄片|支气管检查|支气管镜|气管镜|内镜'),   # CT/MRI case-sensitive (hs-cTnI is a lab test)
    ('airway_suction_supply', r'抽吸导管'),
    ('ecmo', r'ECMO|ECM0|ECOM|体外人工膜肺|体外膜'),
    ('niv', r'无创|CPAP|BiPAP|双水平|鼻塞'),
    ('hfnc', r'高流量'),
    ('imv_ett', r'气管插管|气管导管|插管术|气囊监测|气囊压力'),   # incl. cuff-pressure monitoring of the artificial airway
    ('ventilator', r'呼吸机|呼吸回路|SIMV|IPPV|HFOV|高频振荡|潮气量|PEEP'),   # ventilator orders, circuits and settings
    ('tracheostomy', r'气管切开'),
    ('prone_positioning', r'俯卧位'),
    ('surfactant', r'固尔苏|珂立苏|猪肺磷脂|肺表面活性|牛肺'),
    ('oxygen', r'吸氧|氧气|输氧|供氧|氧浓度|FiO2|吸入氧'),   # incl. oxygen-concentration settings
    ('blood_purification_supply', r'血液净化.*血浆置换|血浆置换.*血液净化'),
    ('plasma_exchange', r'血浆置换'),
    ('hemoperfusion', r'血液灌流|灌流器'),
    ('rrt_chronic_home_pd', r'家庭腹膜透析'),
    ('rrt', r'血液净化|血液透析|血液滤过|CRRT|CVVH|腹膜透析|腹透|透析液|透析速度|透析管|血透|PD液|APD'),
    ('cpr', r'心肺复苏|胸外按压'),
    ('defibrillation_cardioversion', r'除颤|电复律'),
    ('arterial_bp_monitoring', r'有创性血压|有创血压'),
    ('icp_monitoring', r'颅内压监测'),
]
def nd_category(name):
    if name is None:
        return 'other'
    for cat, pat in ND_RULES:
        if re.search(pat, name, flags=re.I):
            return cat
    return 'other'
nd_map = {n: nd_category(n) for n in N.order_name_cn.dropna().unique()}
N['category'] = N.order_name_cn.map(nd_map).fillna('other')
N.loc[N.category.eq('rrt') & N.order_name_cn.fillna('').str.contains('结肠透析'), 'category'] = 'other'
N = N.sort_values(['visit_id', 'start_time', 'order_name_cn'], na_position='last').reset_index(drop=True)
N['order_seq'] = N.groupby('visit_id').cumcount() + 1
qc('7 Non-drug orders', 'Non-drug order rows (raw / unique)', f'{n0} / {len(N)}')
qc('7 Non-drug orders', 'Support-related categories (orders)', N[N.category.ne('other')].category.value_counts().to_dict())
# Troponin test orders by name and period (evidence on the hs-cTn assay; the laboratory sheet does not name it).
_tp = N[N.order_name_cn.fillna('').str.contains('肌钙蛋白')]
TROP = (_tp.groupby('order_name_cn').agg(n=('visit_id', 'size'), first=('start_time', 'min'), last=('start_time', 'max'))
        .sort_values('first').reset_index())
def _span(a, b, fmt='%Y-%m'):
    a_, b_ = a.strftime(fmt), b.strftime(fmt)
    return a_ if a_ == b_ else f'{a_} to {b_}'
TROP_TEXT = '; '.join(f"{r.order_name_cn} ({_span(r.first, r.last)}, {r.n} order{'s' if r.n != 1 else ''})" for r in TROP.itertuples())
qc('7 Non-drug orders', 'Troponin test orders by name (first to last order month, number of orders)', TROP_TEXT or 'none', 'info',
   'The laboratory sheet names the result 超敏心肌钙蛋白定量 without troponin I or T.')

In [ ]:
# Charted vital signs. For supplement admissions the supplementary rows are used (they add heart rate, SpO2 and stool
# count to the same rows as the main export); other admissions use the main export.
vm = sheet('main', '患者生命体征'); vs = sheet('supplement', '患者生命体征')
VCOLS = {'呼吸(次/分)': 'resp_rate', '脉搏(次/分)': 'pulse', '心率(次/分)': 'heart_rate_monitor', '血氧': 'spo2', '体温(°C)': 'temp_c',
         '收缩压(mmHg)': 'sbp', '舒张压(mmHg)': 'dbp', '身高(cm)': 'height_cm', '体重(kg)': 'weight_kg', '大便次数': 'stool_count',
         '尿量(ml)': 'urine_ml', '出量总量(ml)': 'output_total_ml', '入量(ml)': 'intake_ml'}
def vital_frame(d, src):
    out = pd.DataFrame({'visit_id': norm_id(d['科研就诊编号']), 'measure_time': to_dt(d['测量时间'])})
    for c, n in VCOLS.items():
        out[n] = to_num(d[c]) if c in d else np.nan
    out['source'] = src
    return out
VM, VS = vital_frame(vm, 'main'), vital_frame(vs, 'supplement')
supp_vis = set(V.visit_id[V.in_supplement])
shared = ['visit_id', 'measure_time', 'resp_rate', 'pulse', 'temp_c', 'sbp', 'dbp', 'height_cm', 'weight_kg', 'urine_ml', 'output_total_ml', 'intake_ml']
km = row_keys(VM[VM.visit_id.isin(supp_vis)], shared).value_counts()
ks = row_keys(VS, shared).value_counts()
same = km.reindex(ks.index).fillna(0).astype(int).eq(ks).all() and len(km) == len(ks)
qc('8 Vital signs', 'Main and supplementary rows identical for supplement admissions (shared columns, as multisets)', bool(same),
   'pass' if same else 'warn')
VT = pd.concat([VM[~VM.visit_id.isin(set(VS.visit_id))], VS], ignore_index=True)
n0 = len(VT)
VT = VT[VT.measure_time.notna()]
n1 = len(VT)
VT = collapse_identical(VT, list(VT.columns))      # fully identical rows (same admission, time and values) are one record
VT = VT.sort_values(['visit_id', 'measure_time', 'source']).reset_index(drop=True)
VT['row_seq'] = VT.groupby(['visit_id', 'measure_time']).cumcount() + 1
LIMITS = {'resp_rate': (4, 150), 'pulse': (20, 300), 'heart_rate_monitor': (20, 300), 'spo2': (20, 100), 'temp_c': (30, 43.5),
          'sbp': (30, 250), 'dbp': (10, 200), 'height_cm': (30, 220), 'weight_kg': (0.3, 200), 'stool_count': (0, 50),
          'urine_ml': (0, 20000), 'output_total_ml': (0, 30000), 'intake_ml': (0, 30000)}
logrows = []
for c, (lo, hi) in LIMITS.items():
    x = VT[c]
    bad = x.notna() & ((x < lo) | (x > hi))
    if c in ('resp_rate', 'temp_c', 'spo2', 'heart_rate_monitor', 'pulse', 'sbp', 'dbp', 'weight_kg', 'height_cm'):
        bad |= x.eq(0)          # 0 is the platform's code for "not recorded" for physiological measurements
    if c == 'dbp':
        bad |= x.notna() & VT.sbp.notna() & (x >= VT.sbp) & ~VT.sbp.isna()
    if bad.any():
        lg = VT.loc[bad, ['visit_id', 'measure_time', 'row_seq']].copy(); lg['variable'] = c; lg['raw_value'] = x[bad].values
        lg['reason'] = np.where(x[bad].eq(0), 'zero (not recorded)', 'outside plausible range' if c != 'dbp' else 'outside range or >= SBP')
        logrows.append(lg)
        VT.loc[bad, c] = np.nan
VLOG = (pd.concat(logrows, ignore_index=True) if logrows else
        pd.DataFrame(columns=['visit_id', 'measure_time', 'row_seq', 'variable', 'raw_value', 'reason']))
VLOG = VLOG.sort_values(['visit_id', 'measure_time', 'row_seq', 'variable']).reset_index(drop=True)
VT['heart_rate'] = VT.heart_rate_monitor.where(VT.heart_rate_monitor.notna(), VT.pulse)
VT['heart_rate_source'] = np.where(VT.heart_rate_monitor.notna(), 'monitor_heart_rate', np.where(VT.pulse.notna(), 'pulse', None))
qc('8 Vital signs', 'Charted vital-sign rows (with time / without time, dropped)', f'{n1} / {n0 - n1}')
qc('8 Vital signs', 'Fully identical rows collapsed (n_source_rows)', int(n1 - len(VT)), 'info')
qc('8 Vital signs', 'Values set to missing, by variable: zero (= not recorded) / outside plausible range',
   {v: f"{int(g.reason.eq('zero (not recorded)').sum())} / {int(g.reason.ne('zero (not recorded)').sum())}" for v, g in VLOG.groupby('variable')}, 'info',
   'Ranges: ' + '; '.join(f'{k} {v[0]}-{v[1]}' for k, v in LIMITS.items()) + '; diastolic pressure at or above systolic is also set to missing.')
qc('8 Vital signs', 'Rows sharing a timestamp with another row (kept as separate measurements)', int((VT.row_seq > 1).sum()), 'info')
log('Drug orders:', len(D), '| non-drug orders:', len(N), '| vital-sign rows:', len(VT))

In [ ]:
# Laboratory tests: one row per report and analyte, English analyte codes, standard units, specimen separation,
# censoring, and linkage of results filed under emergency or outpatient visit numbers.
PANELS = {
    '凝血功能检查': 'coagulation', '血常规': 'cbc', '血气分析': 'blood_gas', '血液乳酸测定': 'lactate', '血液电解质测定': 'electrolytes',
    '血液肝功能测定': 'liver', '血液肾功能测定': 'renal', '血液心肌酶谱测定': 'cardiac_enzymes', '血液N末端B型钠尿肽前体测定': 'nt_probnp',
    '血C反应蛋白（CRP）测定': 'crp', '血液降钙素原测定': 'procalcitonin', '血氨AMMO': 'ammonia', '细胞因子检测': 'cytokines',
    '血液免疫学测定': 'immunoglobulins_complement', '血液淋巴细胞亚群检测': 'lymphocyte_subsets', '血液铁蛋白测定': 'ferritin',
    '血红细胞沉降率测定': 'esr', '血脂测定': 'lipids', '血液淀粉样蛋白A测定': 'serum_amyloid_a', '血清25-羟基维生素D测定': 'vitamin_d',
    '胰岛素测定': 'insulin', '尿（沉渣）分析检查': 'urinalysis', '脑脊液常规': 'csf_routine', '脑脊液生化': 'csf_chemistry',
    '血检常规(血细胞形态)': 'blood_smear',
    # qualitative and pathogen tests
    '13种呼吸道病原体核酸联合检测': 'resp_pathogen_pcr13', '上呼吸道病原微生物多重联合检测': 'resp_pathogen_multiplex_ngs',
    '咽拭核酸检测': 'influenza_pcr', '咽拭流感病毒AB型核酸检测': 'influenza_pcr', '咽拭病毒抗原测定': 'influenza_antigen',
    '咽拭肺炎支原体肺炎衣原体检测': 'mycoplasma_chlamydia_pcr', '粪便诺如病毒抗原': 'norovirus_antigen', '脑脊液病毒DNA测定': 'csf_virus_pcr',
    '血呼吸道感染病原体抗体检测': 'resp_pathogen_igm', '血清肺炎支原体抗体测定': 'mycoplasma_antibody', '血抗人球蛋白分型试验': 'direct_antiglobulin_test',
}
PANEL_SPECIMEN = {'urinalysis': 'urine', 'csf_routine': 'csf', 'csf_chemistry': 'csf'}   # default: blood
# (panel, analyte_cn) -> (code, standard unit, {normalized raw unit: factor to standard unit})
Q = {}
def q(panel, cn, code, unit, conv=None):
    Q[(panel, cn)] = (code, unit, conv or {})
for cn, code, unit, conv in [('D-二聚体定量', 'd_dimer', 'mg/L FEU', None), ('凝血酶原时间测定定量', 'pt', 's', {'秒': 1}),
        ('凝血酶时间测定定量', 'thrombin_time', 's', {'秒': 1}), ('国际标准比率定量', 'inr', 'ratio', {'秒': 1, '': 1}),
        ('抗凝血酶定量', 'antithrombin', '%', None), ('活化部分凝血活酶时间定量', 'aptt', 's', {'秒': 1}), ('纤维蛋白原检测定量', 'fibrinogen', 'g/L', None)]:
    q('coagulation', cn, code, unit, conv)
for cn, code, unit in [('中性粒细胞总数定量', 'neutrophils_abs', '10^9/L'), ('中性粒细胞百分数定量', 'neutrophils_pct', '%'),
        ('单核细胞总数定量', 'monocytes_abs', '10^9/L'), ('单核细胞百分数定量', 'monocytes_pct', '%'), ('嗜酸性粒细胞总数定量', 'eosinophils_abs', '10^9/L'),
        ('嗜酸性粒细胞百分数定量', 'eosinophils_pct', '%'), ('平均红细胞体积定量', 'mcv', 'fL'), ('淋巴细胞百分数定量', 'lymphocytes_pct', '%'),
        ('白细胞计数定量', 'wbc', '10^9/L'), ('红细胞分布宽度定量', 'rdw', '%'), ('红细胞压积定量', 'hematocrit', '%'), ('血小板分布宽度定量', 'pdw', '%'),
        ('血小板计数定量', 'platelets', '10^9/L'), ('血红蛋白浓度定量', 'hemoglobin', 'g/L')]:
    q('cbc', cn, code, unit)
for cn, code, unit, conv in [('二氧化碳分压定量', 'pco2', 'mmHg', None), ('二氧化碳总量定量', 'tco2', 'mmol/L', None), ('吸氧浓度定量', 'fio2', '%', None),
        ('实际碳酸氢根定量', 'hco3_actual', 'mmol/L', None), ('标准碱剩余定量', 'base_excess_standard', 'mmol/L', None),
        ('标准碳酸氢根定量', 'hco3_standard', 'mmol/L', None), ('氧分压定量', 'po2', 'mmHg', None), ('氧饱和度定量', 'so2', '%', None),
        ('碱剩余定量', 'base_excess', 'mmol/L', None), ('血红蛋白浓度定量', 'hemoglobin_blood_gas', 'g/L', {'g/dl': 10}), ('酸碱度定量', 'ph', '', None)]:
    q('blood_gas', cn, code, unit, conv)
q('lactate', '乳酸定量', 'lactate', 'mmol/L'); q('ammonia', '血氨AMMO定量', 'ammonia', 'umol/L')
q('nt_probnp', 'N末端B型利钠肽原定量', 'nt_probnp', 'pg/mL')
for cn, code, unit in [('免疫球蛋白A定量', 'iga', 'g/L'), ('免疫球蛋白E定量', 'ige', 'IU/mL'), ('免疫球蛋白G定量', 'igg', 'g/L'),
        ('免疫球蛋白M定量', 'igm', 'g/L'), ('补体C3c定量', 'c3', 'g/L'), ('补体C4定量', 'c4', 'g/L')]:
    q('immunoglobulins_complement', cn, code, unit)
for cn, code, unit, conv in [('乳酸脱氢酶同工酶-1定量', 'ldh1', 'U/L', None), ('乳酸脱氢酶定量', 'ldh', 'U/L', None), ('肌酸激酶同工酶-ＭＢ定量', 'ck_mb', 'U/L', None),
        ('肌酸激酶定量', 'ck', 'U/L', None), ('超敏心肌钙蛋白定量', 'hs_ctn', 'ng/L', {'ng/ml': 1000})]:
    q('cardiac_enzymes', cn, code, unit, conv)
q('serum_amyloid_a', '血清淀粉样蛋白A定性', 'serum_amyloid_a', 'mg/L'); q('serum_amyloid_a', '血清淀粉样蛋白A定量', 'serum_amyloid_a', 'mg/L')
for cn, code in [('CD19+B淋巴细胞百分比定量', 'cd19_b_pct'), ('CD19+B淋巴细胞绝对值定量', 'cd19_b_abs'), ('CD3+CD4+CD8+T淋巴细胞百分比定量', 'cd4cd8_dp_t_pct'),
        ('CD3+CD4+CD8+T淋巴细胞绝对值定量', 'cd4cd8_dp_t_abs'), ('CD3+CD4+T淋巴细胞百分比定量', 'cd4_t_pct'), ('CD3+CD4+T淋巴细胞绝对值定量', 'cd4_t_abs'),
        ('CD3+CD8+T淋巴细胞百分比定量', 'cd8_t_pct'), ('CD3+CD8+T淋巴细胞绝对值定量', 'cd8_t_abs'), ('CD3+T淋巴细胞百分比定量', 'cd3_t_pct'),
        ('CD3+T淋巴细胞绝对值定量', 'cd3_t_abs'), ('CD4+/CD8+T淋巴细胞比定量', 'cd4_cd8_ratio'), ('CD4+CD25+淋巴细胞百分比定量', 'cd4cd25_pct'),
        ('Th细胞表达第二信号受体(CD3+CD4+CD28+)定量', 'th_cd28_pct'), ('Treg占CD4+T淋巴细胞百分比定量', 'treg_pct_of_cd4'),
        ('Ts细胞表达第二信号受体(CD3+CD8+CD28+)定量', 'ts_cd28_pct'), ('天然调节性T细胞(CD45RA+CD3+CD4+CD25+CD127low)定量', 'natural_treg_pct'),
        ('活化Ts细胞(CD3+CD8+HLA-DR+)/Ts定量', 'activated_ts_pct'), ('活化T淋巴细胞(CD3+HLA-DR+)定量', 'activated_t_pct'),
        ('纯真Th细胞(CD3+CD4+CD45RA+)/Th定量', 'naive_th_pct'), ('自然杀伤细胞百分比定量', 'nk_pct'), ('自然杀伤细胞绝对值定量', 'nk_abs'),
        ('记忆性Th细胞(CD3+CD4+CD45RO+)/Th定量', 'memory_th_pct'), ('诱导调节性T细胞(CD45RO+CD3+CD4+CD25+CD127low)定量', 'induced_treg_pct'),
        ('调节性T淋巴细胞(CD3+CD4+CD25+CD127low)定量', 'treg_cd25_cd127low_pct')]:
    q('lymphocyte_subsets', cn, code, 'cells/uL' if code.endswith('_abs') else ('ratio' if code.endswith('ratio') else '%'), {'个/ul': 1})
for cn, code in [('氯定量', 'chloride'), ('磷定量', 'phosphate'), ('钙定量', 'calcium'), ('钠定量', 'sodium'), ('钾定量', 'potassium'), ('镁定量', 'magnesium')]:
    q('electrolytes', cn, code, 'mmol/L')
for cn, code, unit in [('r-谷氨酰转肽酶定量', 'ggt', 'U/L'), ('前白蛋白定量', 'prealbumin', 'mg/L'), ('总胆汁酸定量', 'bile_acids_total', 'umol/L'),
        ('总胆红素定量', 'bilirubin_total', 'umol/L'), ('总蛋白定量', 'protein_total', 'g/L'), ('球蛋白定量', 'globulin', 'g/L'),
        ('白球比定量', 'albumin_globulin_ratio', 'ratio'), ('白蛋白定量', 'albumin', 'g/L'), ('直接胆红素定量', 'bilirubin_direct', 'umol/L'),
        ('碱性磷酸酶定量', 'alp', 'U/L'), ('胆碱酯酶定量', 'cholinesterase', 'U/L'), ('谷丙转氨酶定量', 'alt', 'U/L'), ('谷草转氨酶定量', 'ast', 'U/L'),
        ('非结合胆红素定量', 'bilirubin_unconjugated', 'umol/L')]:
    q('liver', cn, code, unit, {'': 1} if unit == 'ratio' else None)
for cn, code, unit, conv in [('尿素氮定量', 'urea', 'mmol/L', None), ('尿酸定量', 'uric_acid', 'umol/L', None), ('肌酐定量', 'creatinine', 'umol/L', None),
        ('胱抑素C定量', 'cystatin_c', 'mg/L', None), ('视黄醇结合蛋白定量', 'retinol_binding_protein', 'mg/L', {'ug/ml': 1})]:
    q('renal', cn, code, unit, conv)
q('ferritin', '铁蛋白定量', 'ferritin', 'ng/mL'); q('procalcitonin', '降钙素原定量', 'procalcitonin', 'ng/mL')
q('vitamin_d', '25-羟基维生素D定量', 'vitamin_d_25oh', 'ng/mL'); q('esr', '血沉定量', 'esr', 'mm/h')
for cn, code, unit in [('低密度脂蛋白胆固醇定量', 'ldl_c', 'mmol/L'), ('总胆固醇定量', 'cholesterol_total', 'mmol/L'), ('甘油三酯定量', 'triglycerides', 'mmol/L'),
        ('脂蛋白a定量', 'lipoprotein_a', 'nmol/L'), ('载脂蛋白A1定量', 'apo_a1', 'g/L'), ('载脂蛋白B定量', 'apo_b', 'g/L'), ('高密度脂蛋白胆固醇定量', 'hdl_c', 'mmol/L')]:
    q('lipids', cn, code, unit)
for cn, code in [('HumanIL-10定性', 'il10'), ('HumanIL-2定性', 'il2'), ('HumanIL-4定性', 'il4'), ('HumanIL-6定性', 'il6'), ('HumanTNF-α定性', 'tnf_alpha'),
        ('humanIFN-γ定性', 'ifn_gamma'), ('humanIL-17A定性', 'il17a')]:
    q('cytokines', cn, code, 'pg/mL')
q('insulin', '胰岛素测定定量', 'insulin', 'uIU/mL'); q('crp', 'C反应蛋白定量', 'crp', 'mg/L'); q('crp', '超敏C反应蛋白定量', 'hs_crp', 'mg/L')
q('urinalysis', '尿比重定量', 'specific_gravity', '', {'': 1}); q('csf_routine', '白细胞计数定量', 'wbc', '10^6/L')
q('csf_chemistry', '脑脊液氯离子定量', 'chloride', 'mmol/L'); q('csf_chemistry', '脑脊液葡萄糖定量', 'glucose', 'mmol/L')
q('csf_chemistry', '脑脊液蛋白定量', 'protein', 'g/L', {'mg/l': 0.001})
QUAL_CN = {   # qualitative analytes inside quantitative panels
    ('csf_routine', '占多数细胞定性'): 'predominant_cell', ('csf_routine', '糖定性定性'): 'glucose_qualitative',
    ('csf_routine', '脑脊液浊度定性'): 'turbidity', ('csf_routine', '脑脊液颜色定性'): 'color', ('csf_routine', '蛋白定性定性'): 'protein_pandy',
    ('blood_smear', '血细胞形态'): 'blood_cell_morphology'}
QUAL_PANELS = {'resp_pathogen_pcr13', 'resp_pathogen_multiplex_ngs', 'influenza_pcr', 'influenza_antigen', 'mycoplasma_chlamydia_pcr',
               'norovirus_antigen', 'csf_virus_pcr', 'resp_pathogen_igm', 'mycoplasma_antibody', 'direct_antiglobulin_test'}

def norm_unit(u):
    if is_missing(u):
        return ''
    return re.sub(r'\s+', '', str(u)).replace('μ', 'u').replace('µ', 'u').lower()

SPECIMEN = {'血清': 'serum', '血浆': 'plasma', '静脉血': 'venous_blood', '全血': 'whole_blood', '末梢血': 'capillary_blood', '末稍血': 'capillary_blood',
            '毛细血管血': 'capillary_blood', '外周血': 'peripheral_blood', '外周血或末梢血': 'peripheral_blood', '动脉血': 'arterial_blood', '末稍血（门）': 'capillary_blood', '末梢血（门）': 'capillary_blood',
            '尿液': 'urine', '尿液（门）': 'urine', '脑脊液': 'csf', '咽拭子': 'throat_swab', '痰液': 'sputum', '肺泡灌洗液': 'bronchoalveolar_lavage', '粪便': 'stool'}
def specimen_class(sp):
    if is_missing(sp):
        return None
    if sp.endswith('_blood') or sp in ('serum', 'plasma'):
        return 'blood'
    return {'urine': 'urine', 'csf': 'csf', 'throat_swab': 'respiratory', 'sputum': 'respiratory', 'bronchoalveolar_lavage': 'respiratory', 'stool': 'stool'}.get(sp, 'other')

NUM_RX = re.compile(r'^(?P<sign><=|>=|≤|≥|<|>|≦|≧|〈|〉)?\s*(?P<num>[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?)\s*(?P<tail>.*)$')
def parse_result(raw):
    """Return (value, censor, flag) for one raw result string."""
    if is_missing(raw):
        return (np.nan, '', 'empty')
    if isinstance(raw, (int, float, np.integer, np.floating)):
        return (float(raw), '', '')
    s = str(raw).translate(FULLWIDTH).replace(' ', '').replace('　', '').strip()
    m = NUM_RX.match(s)
    if m:
        sign, tail = m.group('sign') or '', m.group('tail')
        cens = '<' if sign in ('<', '<=', '≤', '≦', '〈') else ('>' if sign in ('>', '>=', '≥', '≧', '〉') else '')
        if tail == '':
            return (float(m.group('num')), cens, '')
        if re.fullmatch(r'[\(（\[][^\)）\]]*[\)）\]]', tail):
            return (float(m.group('num')), cens, 'annotated')
        if re.fullmatch(r'-\d+(?:\.\d+)?', tail) and not cens:
            return (np.nan, '', 'range_text')
    if re.search(r'未凝|无凝固|不凝', s):
        return (np.nan, '>', 'no_clot')
    if re.search(r'超出检测限|超出线性|OVER|超过', s, flags=re.I):
        return (np.nan, '>', 'above_range_no_value')
    return (np.nan, '', 'non_numeric')

lab_parts, qual_parts = [], []
for role in ('main', 'supplement'):
    for sh, parts in RAW[role].items():
        name = sh.replace('实验室检查_', '')
        if not sh.startswith('实验室检查_') or name not in PANELS:
            continue
        panel = PANELS[name]
        SHEETS_READ.add((role, sh))
        df = pd.concat(parts, ignore_index=True, sort=False)
        if df.empty:
            continue
        base = pd.DataFrame({'visit_id': norm_id(df['就诊编号']), 'lab_patient_id': norm_id(df['患者编号']),
                             'report_id': norm_id(df['报告编号']), 'report_time': to_dt(df['报告时间']),
                             'specimen_raw': norm_text(df['样本类型']) if '样本类型' in df else None})
        analytes = {}
        for c in df.columns:
            for suf in ['-定量结果单位', '-定量结果', '-正常高值', '-正常低值', '-定性结果显示', '-定性结果', '-危机高值']:
                if c.endswith(suf):
                    analytes.setdefault(c[:-len(suf)], {})[suf] = c
                    break
        for a, cols in analytes.items():
            get = lambda suf: df[cols[suf]] if suf in cols else pd.Series([None] * len(df))
            rec = base.copy()
            rec['panel'], rec['analyte_cn'], rec['source_role'] = panel, a, role
            rec['quant_raw'], rec['display_raw'], rec['qual_raw'] = get('-定量结果').values, get('-定性结果显示').values, get('-定性结果').values
            rec['unit_raw'] = norm_text(get('-定量结果单位')).values
            rec['ref_low_raw'], rec['ref_high_raw'] = get('-正常低值').values, get('-正常高值').values
            has = ~(rec.quant_raw.map(is_missing) & rec.display_raw.map(is_missing) & rec.qual_raw.map(is_missing))
            rec = rec[has]
            if (panel, a) in Q:
                lab_parts.append(rec)
            elif panel in QUAL_PANELS or (panel, a) in QUAL_CN:
                qual_parts.append(rec)
            else:
                qc('9 Laboratory', f'Unmapped analyte {panel}/{a}', int(has.sum()), 'warn')
L = pd.concat(lab_parts, ignore_index=True, sort=False)
n_raw = len(L)
# The two exports carry copies of the same report; keep one per report and analyte after checking that the copies agree.
_k = row_keys(L, ['report_id', 'panel', 'analyte_cn'])
_multi = _k.duplicated(keep=False)
_v = row_keys(L[_multi], ['visit_id', 'report_time', 'quant_raw', 'display_raw', 'qual_raw', 'unit_raw', 'ref_low_raw', 'ref_high_raw'])
LAB_CONFLICTS = pd.DataFrame({'k': _k[_multi], 'v': _v}).groupby('k').v.nunique()
LAB_CONFLICTS = LAB_CONFLICTS[LAB_CONFLICTS > 1]
qc('9 Laboratory', 'Report-analyte pairs exported more than once whose copies differ (visit, time, value, unit or reference range)',
   f'{len(LAB_CONFLICTS)} of {int(_k[_multi].nunique())}', 'pass' if len(LAB_CONFLICTS) == 0 else 'warn')
L['_q'] = ~L.quant_raw.map(is_missing)
L = L.sort_values(['report_id', 'analyte_cn', '_q', 'source_role'], ascending=[True, True, False, True])
L = L.drop_duplicates(['report_id', 'panel', 'analyte_cn']).drop(columns='_q')
qc('9 Laboratory', 'Quantitative result rows (both exports / unique report-analyte)', f'{n_raw} / {len(L)}')
parsed = {}
RANK = {'no_clot': 4, 'above_range_no_value': 3, 'range_text': 2, 'non_numeric': 1, 'empty': 0}
def parse_row(qv, dv, lv):
    out = parse_result(qv)
    if not np.isnan(out[0]) or out[2] in ('no_clot', 'above_range_no_value'):
        return out
    best = out
    for alt in (dv, lv):
        o2 = parse_result(alt)
        if not np.isnan(o2[0]):
            return (o2[0], o2[1], 'value_from_display' if not o2[2] else o2[2] + '|value_from_display')
        if RANK.get(o2[2], 0) > RANK.get(best[2], 0):
            best = o2
    return best if best[2] != 'empty' else (np.nan, '', 'non_numeric')
res = [parse_row(a, b, c) for a, b, c in zip(L.quant_raw, L.display_raw, L.qual_raw)]
L['value'] = [r[0] for r in res]
L['censor'] = [r[1] for r in res]
L['result_flag'] = [r[2] for r in res]
L['value_raw'] = [str(a).strip() if not is_missing(a) else (str(b).strip() if not is_missing(b) else None) for a, b in zip(L.quant_raw, L.display_raw)]
L['specimen'] = L.specimen_raw.map(lambda s: None if is_missing(s) else SPECIMEN.get(s, 'other'))
L['specimen_class'] = L.specimen.map(specimen_class)
# Analyte code, standard unit and unit conversion. A result whose specimen is cerebrospinal fluid or urine takes the
# definition of the matching CSF or urine panel when one exists (e.g. a CSF cell count filed in the blood-count panel).
ALT_PANELS = {'csf': ['csf_routine', 'csf_chemistry'], 'urine': ['urinalysis']}
def spec_for(panel, cn, sc):
    for alt in ALT_PANELS.get(sc, []):
        if PANEL_SPECIMEN.get(panel, 'blood') != sc and (alt, cn) in Q:
            return Q[(alt, cn)]
    return Q[(panel, cn)]
spec = pd.Series([spec_for(pn, cn, sc) for pn, cn, sc in zip(L.panel, L.analyte_cn, L.specimen_class)], index=L.index)
L['analyte'] = spec.map(lambda t: t[0])
L['unit'] = spec.map(lambda t: t[1])
L['_unorm'] = L.unit_raw.map(norm_unit)
L['_std'] = L.unit.map(norm_unit)
conv = [t[2].get(u, 1.0 if (u == s or u == '') else None) for t, u, s in zip(spec, L._unorm, L._std)]
L['_factor'] = conv
nolab = L._unorm.eq('') & L._std.ne('') & ~L.unit.isin(['ratio'])
L.loc[nolab, 'result_flag'] = (L.loc[nolab, 'result_flag'].fillna('') + '|unit_missing_assumed_standard').str.strip('|')
ref_hi = to_num(L.ref_high_raw); ref_lo = to_num(L.ref_low_raw)
# Rows whose unit label disagrees with the standard unit and has no defined conversion: relabel when the row's own
# reference range sits on the standard-unit scale (within a factor of 2 of the analyte's typical upper reference limit);
# otherwise the value is set to missing.
typ_hi = pd.Series(ref_hi.values, index=L.index).where(L._factor.notna()).groupby(L.analyte).median()
unres = L._factor.isna()
ratio = ref_hi[unres.values].values / L.loc[unres, 'analyte'].map(typ_hi).values if unres.any() else np.array([])
relabel_idx = L.index[unres][(ratio >= 0.5) & (ratio <= 2)] if unres.any() else []
L.loc[relabel_idx, '_factor'] = 1.0
L.loc[relabel_idx, 'result_flag'] = (L.loc[relabel_idx, 'result_flag'].fillna('') + '|unit_relabeled').str.strip('|')
still = L._factor.isna()
L.loc[still, 'result_flag'] = (L.loc[still, 'result_flag'].fillna('') + '|unit_unresolved').str.strip('|')
f = L._factor.astype(float)
L['value'] = (L.value * f).where(~still)
L['ref_low'] = (ref_lo.values * f.values)
L['ref_high'] = (ref_hi.values * f.values)
known_conv = L._factor.notna() & (L._factor != 1.0)
L.loc[known_conv, 'result_flag'] = (L.loc[known_conv, 'result_flag'].fillna('') + '|unit_converted').str.strip('|')
# Specimen: results from a specimen other than the panel's expected specimen get a prefixed analyte code.
expected = L.panel.map(PANEL_SPECIMEN).fillna('blood')
other_spec = L.specimen_class.notna() & (L.specimen_class != expected)
L.loc[other_spec, 'analyte'] = L.loc[other_spec, 'specimen_class'] + '_' + L.loc[other_spec, 'analyte']
L.loc[other_spec, 'specimen_note'] = 'specimen differs from panel default'
L.loc[expected.eq('csf') & ~other_spec, 'analyte'] = 'csf_' + L.loc[expected.eq('csf') & ~other_spec, 'analyte']
L.loc[expected.eq('urine') & ~other_spec, 'analyte'] = 'urine_' + L.loc[expected.eq('urine') & ~other_spec, 'analyte']
L['result_flag'] = L.result_flag.replace('', None)
# Hard physiological limits (flag only; laboratory results are kept as reported)
HARD = {'ph': (6.5, 8.0), 'sodium': (90, 200), 'potassium': (1.0, 12.0), 'chloride': (50, 160), 'calcium': (0.5, 5.0),
        'hemoglobin': (10, 260), 'platelets': (0, 3000), 'wbc': (0, 500), 'lactate': (0, 35), 'creatinine': (5, 3000)}
L['outside_physiological_limits'] = False
for a, (lo, hi) in HARD.items():
    mk = L.analyte.eq(a) & L.value.notna() & ((L.value < lo) | (L.value > hi))
    L.loc[mk, 'outside_physiological_limits'] = True
qc('9 Laboratory', 'Analytes (codes)', int(L.analyte.nunique()))
qc('9 Laboratory', 'Result flags', L.result_flag.value_counts().to_dict(), 'info',
   'value_from_display: number read from the display column; unit_converted/relabeled/unresolved: see definitions.')
qc('9 Laboratory', 'Censored results (< or >)', L.censor.value_counts().to_dict(), 'info', 'value holds the reporting limit.')
qc('9 Laboratory', 'Rows from a specimen other than the panel default (prefixed codes)', L[other_spec].analyte.value_counts().to_dict(), 'info')
qc('9 Laboratory', 'Values outside hard physiological limits (flagged, kept)', L[L.outside_physiological_limits].analyte.value_counts().to_dict(), 'info')

# Visit linkage: results filed under an inpatient admission, a listed outpatient visit, or an unlisted visit number.
VIS = V.set_index('visit_id')
def link_visits(df, time_col):
    df = df.copy()
    inpatient = df.visit_id.isin(VIS.index)
    df['patient_id'] = df.visit_id.map(VIS.patient_id)
    df['patient_id'] = df.patient_id.where(df.patient_id.notna(), df.lab_patient_id)
    df['visit_link'] = np.where(inpatient, 'same_admission', np.where(df.visit_id.isin(set(VO.visit_id)), 'outpatient_visit', 'unlisted_visit'))
    df['linked_admission_id'] = df.visit_id.where(inpatient)
    cand = df[~inpatient & df[time_col].notna()][['patient_id', time_col]].reset_index()
    adm = V[['patient_id', 'visit_id', 'admit_time', 'discharge_time']].dropna(subset=['admit_time'])
    j = cand.merge(adm, on='patient_id', how='inner')
    j['h'] = (j.admit_time - j[time_col]).dt.total_seconds() / 3600
    pre = j[(j.h > 0) & (j.h <= 24)].sort_values('h').drop_duplicates('index')
    during = j[(j[time_col] >= j.admit_time) & ((j[time_col] <= j.discharge_time) | j.discharge_time.isna())].drop_duplicates('index')
    df.loc[during['index'], 'linked_admission_id'] = during.visit_id.values
    df.loc[during['index'], 'visit_link'] = df.loc[during['index'], 'visit_link'] + '_during_admission'
    pre = pre[~pre['index'].isin(during['index'])]
    df.loc[pre['index'], 'linked_admission_id'] = pre.visit_id.values
    df.loc[pre['index'], 'visit_link'] = df.loc[pre['index'], 'visit_link'] + '_within_24h_before_admission'
    df['patient_id_mismatch'] = inpatient & df.lab_patient_id.notna() & (df.lab_patient_id != df.visit_id.map(VIS.patient_id))
    return df
L = link_visits(L, 'report_time')
L = L.sort_values(['patient_id', 'report_time', 'report_id', 'analyte'], na_position='last').reset_index(drop=True)
qc('9 Laboratory', 'Result rows by visit linkage', L.visit_link.value_counts().to_dict(), 'info',
   'Results filed under an emergency or outpatient number are linked to an admission when reported in the 24 h before it.')
qc('9 Laboratory', 'Rows whose patient number differs from the admission\'s patient', int(L.patient_id_mismatch.sum()), 'info',
   'patient_id is taken from the admission; lab_patient_id keeps the number on the report.')
qc('9 Laboratory', 'Report time missing', int(L.report_time.isna().sum()), 'pass' if L.report_time.notna().all() else 'warn')
log('Lab rows:', len(L), '| analytes:', L.analyte.nunique())

In [ ]:
# Qualitative and pathogen tests (incl. the 7-virus antigen panel and the multiplex sequencing panel).
G = pd.concat(qual_parts, ignore_index=True, sort=False)
G = G.sort_values(['report_id', 'analyte_cn', 'source_role']).drop_duplicates(['report_id', 'panel', 'analyte_cn'])
seven = []
for role in ('main', 'supplement'):
    d = sheet(role, '七项呼吸道病毒的检测')
    if len(d):
        for c in ['副流感病毒1型', '副流感病毒2型', '副流感病毒3型', '流感病毒A型', '流感病毒B型', '腺病毒', '呼吸道合胞病毒']:
            if c in d:
                seven.append(pd.DataFrame({'visit_id': norm_id(d['科研就诊编号']), 'lab_patient_id': norm_id(d['科研患者编号']),
                    'report_id': norm_id(d['病理号']) if '病理号' in d else None, 'report_time': to_dt(d['检查时间']), 'specimen_raw': None,
                    'panel': 'resp_virus_antigen7', 'analyte_cn': c, 'source_role': role, 'qual_raw': d[c].values, 'display_raw': None, 'quant_raw': None}))
if seven:
    S7 = pd.concat(seven, ignore_index=True)
    S7 = S7[~S7.qual_raw.map(is_missing)]
    S7 = S7.sort_values('source_role').drop_duplicates(['visit_id', 'report_time', 'analyte_cn'])
    G = pd.concat([G, S7], ignore_index=True, sort=False)
TEST_CODES = {'人鼻病毒核酸定性': 'rhinovirus', '偏肺病毒RNA定性': 'metapneumovirus', '冠状病毒核酸检测定性': 'coronavirus', '副流感病毒(PIV)RNA定性': 'parainfluenza',
    '博卡病毒DNA定性': 'bocavirus', '呼吸道合胞病毒(RSV)RNA定性': 'rsv', '流感病毒A型RNA定性': 'influenza_a', '流感病毒A型（H1N1）RNA定性': 'influenza_a_h1n1',
    '流感病毒A型（H3N2）RNA定性': 'influenza_a_h3n2', '流感病毒B型RNA定性': 'influenza_b', '肺炎支原体DNA定性': 'mycoplasma_pneumoniae',
    '腺病毒(AdV)RNA定性': 'adenovirus', '衣原体DNA定性': 'chlamydia', '流感病毒A型定性': 'influenza_a', '流感病毒B型定性': 'influenza_b',
    '肺炎支原体RNA定性': 'mycoplasma_pneumoniae', '肺炎衣原体RNA定性': 'chlamydia_pneumoniae', '粪便诺如病毒抗原(GI)定性': 'norovirus_gi',
    '粪便诺如病毒抗原(GII)定性': 'norovirus_gii', '乙型脑炎病毒核酸定性': 'japanese_encephalitis_virus', '单纯疱疹病毒Ⅰ+Ⅱ型DNA（定性）定性': 'hsv1_2',
    '巨细胞病毒DNA（定性）定性': 'cmv', '水痘-带状疱疹病毒核酸（定性）定性': 'vzv', '肺炎支原体抗体IgM定性': 'mycoplasma_pneumoniae_igm',
    '肺炎支原体抗体MP-Ab定性': 'mycoplasma_pneumoniae_antibody', '抗C3d-IgG定性': 'dat_c3d_igg', '抗C3d定性': 'dat_c3d',
    '副流感病毒1型': 'parainfluenza_1', '副流感病毒2型': 'parainfluenza_2', '副流感病毒3型': 'parainfluenza_3', '流感病毒A型': 'influenza_a',
    '流感病毒B型': 'influenza_b', '腺病毒': 'adenovirus', '呼吸道合胞病毒': 'rsv', '呼吸道病毒': 'multiplex_panel'}
CSF_VALUES = {'无色': 'colorless', '淡黄色': 'light_yellow', '微黄色': 'slightly_yellow', '黄色': 'yellow', '红色': 'red', '淡红色': 'light_red',
              '透明': 'clear', '微混': 'slightly_turbid', '浑浊': 'turbid', '单个核细胞': 'mononuclear', '多个核细胞': 'polymorphonuclear'}
def qual_result(raw, test=None):
    if is_missing(raw):
        return None
    s = str(raw).translate(FULLWIDTH).replace(' ', '').strip()
    if s in ('/', '-', '--'):
        return 'no_result'
    if s in CSF_VALUES:
        return CSF_VALUES[s]
    if test == 'turbidity':
        if '凝块' in s:
            return 'turbid_with_clot'
        if re.search(r'微[浑混]', s):
            return 'slightly_turbid'
        if re.search(r'[浑混]浊', s):
            return 'turbid'
        if '透明' in s:
            return 'clear'
    if test == 'predominant_cell':
        if '多个核' in s:
            return 'polymorphonuclear'
        if '单个核' in s:
            return 'mononuclear'
    if test == 'color':
        return 'bloody' if '血性' in s else 'other_color'
    if re.search(r'弱阳', s):
        return 'weak_positive'
    if re.search(r'阴性|未检出|未发现|^\(?-\)?$|^\(-\)$', s):
        return 'negative'
    if re.search(r'阳性|检出|^\(?\++\)?$|\(\++\)', s):
        return 'positive'
    if s == '已报告':
        return 'reported_without_result'
    if re.fullmatch(r'[<>]?\d+(\.\d+)?(E[+-]?\d+)?', s, flags=re.I):
        return 'numeric'
    return 'other'
G['result_raw'] = [S(b) if not is_missing(b) else S(a) for a, b in zip(G.qual_raw, G.display_raw)]
multi = G.panel.eq('resp_pathogen_multiplex_ngs')
rows = []
for r in G[multi].itertuples(index=False):
    txt = r.result_raw or ''
    # entries are organism:reads:load; empty slots are exported as "-:-:-" and are skipped
    found = [x for x in txt.split(',') if x.strip() and x.split(':')[0].strip() not in ('', '-', '--', '/')]
    if not found or re.search(r'未发现', txt):
        rows.append({**r._asdict(), 'test': 'multiplex_panel', 'organism_cn': None, 'result': 'not_detected', 'reads': np.nan, 'load_raw': None})
    for x in found:
        parts = x.split(':')
        if len(parts) >= 2 and not re.search(r'未发现', parts[0]):
            rows.append({**r._asdict(), 'test': 'multiplex_panel', 'organism_cn': parts[0].strip(), 'result': 'detected',
                         'reads': to_num([parts[1]])[0], 'load_raw': ':'.join(parts[2:]).strip() or None})
GM = pd.DataFrame(rows)
GQ = G[~multi].copy()
GQ['test'] = [QUAL_CN.get((p, a)) or TEST_CODES.get(a, 'other') for p, a in zip(GQ.panel, GQ.analyte_cn)]
GQ['organism_cn'], GQ['reads'], GQ['load_raw'] = None, np.nan, None
GQ['result'] = [qual_result(x, t) for x, t in zip(GQ.result_raw, GQ.test)]
GQ['result_value'] = [parse_result(x)[0] if r == 'numeric' else np.nan for x, r in zip(GQ.result_raw, GQ.result)]
GM['result_value'] = np.nan
LQ = pd.concat([GQ, GM], ignore_index=True, sort=False)
LQ['specimen'] = LQ.specimen_raw.map(lambda s: None if is_missing(s) else SPECIMEN.get(s, 'other'))
LQ['specimen_class'] = LQ.specimen.map(specimen_class)
LQ['time_precision'] = np.where(midnight(LQ.report_time), 'date', 'minute')
LQ = link_visits(LQ, 'report_time')
LQ = LQ.sort_values(['patient_id', 'report_time', 'panel', 'test', 'organism_cn'], na_position='last').reset_index(drop=True)
qc('10 Qualitative tests', 'Rows by panel', LQ.panel.value_counts().to_dict())
qc('10 Qualitative tests', 'Result categories', LQ.result.value_counts().to_dict(), 'info')
qc('10 Qualitative tests', 'Unrecognized result strings (result = other)', int(LQ.result.eq('other').sum()), 'info')

# Point-of-care glucose (no timestamp in the export): kept per admission without time.
gl = sheet('main', '血糖检测')
GL = pd.DataFrame({'visit_id': norm_id(gl['科研就诊编号']), 'value_raw': norm_text(gl['血糖值'])})
pg = [parse_result(x) for x in GL.value_raw]
GL['glucose_mmol_L'] = [p[0] for p in pg]; GL['censor'] = [p[1] for p in pg]; GL['result_flag'] = [p[2] or None for p in pg]
GL['row_seq'] = GL.groupby('visit_id').cumcount() + 1
qc('10 Qualitative tests', 'Point-of-care glucose rows (no timestamp in export)', len(GL), 'info', 'Not usable for time-anchored analyses.')
gls = sheet('supplement', '血糖检测')
if len(gls):
    # The sheet has only patient, admission and value. Equal values within an admission cannot be told apart and are kept
    # as separate measurements; the supplementary copy is checked to hold the same values in the same order and number.
    seq_m = GL.groupby('visit_id').value_raw.apply(list)
    seq_s = pd.DataFrame({'visit_id': norm_id(gls['科研就诊编号']), 'value_raw': norm_text(gls['血糖值'])}).groupby('visit_id').value_raw.apply(list)
    same_gl = all(seq_m.get(v) == seq_s[v] for v in seq_s.index)
    qc('10 Qualitative tests', 'Glucose: supplementary rows identical to the main export (same values, order and number per admission)',
       f'{same_gl} ({len(seq_s)} admissions)', 'pass' if same_gl else 'warn',
       'Equal values within an admission are repeated measurements (no time or report number exists to tell them apart).')

In [ ]:
# Examinations: ECG, echocardiography, EEG and imaging events (times, names and numeric measurements; no narrative text).
EX = []
EXAM_DUP, EXAM_VALID = {}, {}
def exams(role, sh, modality, tcol, extra=None):
    d = sheet(role, sh)
    if d.empty:
        return None
    # One row per report. Rows that agree in every exported column except the item name (报告名称, CT and MRI in the main
    # export) are one report, exported once per examination item or more than once; the item names are joined.
    keycols = [c for c in d.columns if c not in ('报告名称', '_file')]
    k = row_keys(d, keycols)
    nrows = k.map(k.value_counts())
    keep = ~k.duplicated()
    EXAM_DUP[(modality, role)] = int((~keep).sum())
    names = None
    if '报告名称' in d:
        nm = pd.Series(norm_text(d['报告名称']).values, index=k.values).dropna()
        names = nm.groupby(level=0).apply(lambda s: ' | '.join(sorted(set(s))))
    kk, d, nrows = k[keep], d.loc[keep], nrows[keep]
    out = pd.DataFrame({'visit_id': norm_id(d['科研就诊编号']), 'exam_time': to_dt(d[tcol]), 'modality': modality, 'source_role': role})
    out['exam_name_cn'] = kk.map(names).values if names is not None else None
    out['_concl'] = norm_text(d['检查结论']).values if '检查结论' in d else None    # used only to match names across exports
    for k_, v in (extra or {}).items():
        out[k_] = v(d).values
    out['n_source_rows'] = nrows.astype(int).values
    return out
ecg_num = lambda c: (lambda d: to_num(d[c]) if c in d else pd.Series([np.nan] * len(d)))
def ef_parse(d):
    if '射血分数' not in d:
        return pd.Series([np.nan] * len(d))
    x = d['射血分数'].map(lambda s: np.nan if is_missing(s) else (float(s) if isinstance(s, (int, float)) else
                          (float(re.search(r'\d+(?:\.\d+)?', str(s)).group()) if re.search(r'\d+(?:\.\d+)?', str(s)) else np.nan)))
    return x.where((x >= 10) & (x <= 90))
specs = [('心电图报告', 'ECG', '心电检查时间', {'ecg_hr': ecg_num('HR'), 'ecg_pr_ms': ecg_num('PR'), 'ecg_qrs_ms': ecg_num('QRSD'),
                                               'ecg_qt_ms': ecg_num('QT'), 'ecg_qtc_ms': ecg_num('QTc')}),
         ('儿童心脏彩超', 'ECHO', '检查时间', {'echo_lvef_pct': ef_parse}), ('脑电图检查', 'EEG', '检查日期', None),
         ('X线报告', 'XRAY', '检查时间', None), ('CT报告', 'CT', '检查时间', None), ('MRI报告', 'MRI', '检查时间', None),
         ('超声报告', 'ULTRASOUND', '超声报告时间', None)]
supp_vis = set(V.visit_id[V.in_supplement])
for sh, mod, tcol, extra in specs:
    em, es = exams('main', sh, mod, tcol, extra), exams('supplement', sh, mod, tcol, extra)
    if em is not None and es is not None:
        EXAM_VALID[mod] = f'{int(em.visit_id.isin(set(es.visit_id)).sum())} / {len(es)}'
    parts = []
    if es is not None:
        parts.append(es)
    if em is not None:
        parts.append(em[~em.visit_id.isin(set(es.visit_id))] if es is not None else em)
    if parts:
        e = pd.concat(parts, ignore_index=True, sort=False)
        if es is not None and em is not None and em.exam_name_cn.notna().any():
            # supplementary rows replace the main rows of the admissions they cover; the item names (main export only) are
            # matched on admission, time and conclusion text
            nm_ = em.dropna(subset=['exam_name_cn']).drop_duplicates(['visit_id', 'exam_time', '_concl'])
            names = {(v, t, c): n for v, t, c, n in zip(nm_.visit_id, nm_.exam_time, nm_._concl, nm_.exam_name_cn)}
            miss = e.exam_name_cn.isna()
            e.loc[miss, 'exam_name_cn'] = [names.get((v, t, c)) for v, t, c in zip(e.visit_id[miss], e.exam_time[miss], e._concl[miss])]
        EX.append(e)
EXM = pd.concat(EX, ignore_index=True, sort=False)
for c in ['ecg_hr', 'ecg_pr_ms', 'ecg_qrs_ms', 'ecg_qt_ms', 'ecg_qtc_ms', 'echo_lvef_pct']:
    if c not in EXM:
        EXM[c] = np.nan
lim = {'ecg_hr': (20, 350), 'ecg_pr_ms': (40, 400), 'ecg_qrs_ms': (30, 250), 'ecg_qt_ms': (150, 700), 'ecg_qtc_ms': (250, 750)}
for c, (lo, hi) in lim.items():
    EXM.loc[EXM[c].notna() & ((EXM[c] < lo) | (EXM[c] > hi)), c] = np.nan
EXM['time_precision'] = np.where(EXM.exam_time.isna(), None, np.where(midnight(EXM.exam_time), 'date', 'minute'))
EXM['visit_known'] = EXM.visit_id.isin(VIS.index)
EXM = EXM.sort_values(['visit_id', 'exam_time', 'modality', 'exam_name_cn'], na_position='last').reset_index(drop=True)
EXM['exam_seq'] = EXM.groupby('visit_id').cumcount() + 1
qc('11 Examinations', 'Rows collapsed into one row per report (identical in every exported column, incl. the report text, except the item name)',
   {f'{m} {r}': n for (m, r), n in EXAM_DUP.items() if n}, 'info',
   'A report is exported once per examination item (CT and MRI item names are joined in exam_name_cn) or more than once; n_source_rows keeps the count.')
qc('11 Examinations', 'Reports of supplement admissions after collapsing: main export / supplementary export', EXAM_VALID, 'info',
   'The supplementary export has more columns. Fewer reports in the main export are distinct reports that the main export cannot tell apart '
   '(same time and conclusion); the same small undercount applies to admissions outside the supplement, which have only the main export.')
_cm = EXM[EXM.modality.isin(['CT', 'MRI'])]
qc('11 Examinations', 'CT and MRI reports without an item name (supplementary report not matched in the main export)', int(_cm.exam_name_cn.isna().sum()), 'info')
qc('11 Examinations', 'Examination events by modality', EXM.modality.value_counts().to_dict())
qc('11 Examinations', 'Events with date-only time', EXM[EXM.time_precision.eq('date')].modality.value_counts().to_dict(), 'info')
qc('11 Examinations', 'ECG with interval measurements / echo with LVEF', f"{int(EXM.ecg_qtc_ms.notna().sum())} / {int(EXM.echo_lvef_pct.notna().sum())}")
log('Qualitative rows:', len(LQ), '| glucose rows:', len(GL), '| exam events:', len(EXM))

In [ ]:
# Organ-support events (long table) from drug orders, non-drug orders and procedures.
# Drug orders carry no duration in this export (end time = start time, or missing; infusions are re-ordered about daily),
# so drug-order events have no end_time. Non-drug orders are long-term or one-time orders with a start and an end time.
ev = []
vz = D[D.vasoactive_infusion]
ev.append(pd.DataFrame({'visit_id': vz.visit_id, 'support': 'vasoactive_infusion', 'event_time': vz.order_time, 'end_time': pd.NaT,
                        'time_precision': 'minute', 'source': 'drug_order', 'source_seq': vz.order_seq, 'detail': vz.vasoactive_agent}))
pz = D[D.pituitrin_infusion]
ev.append(pd.DataFrame({'visit_id': pz.visit_id, 'support': 'pituitrin_infusion', 'event_time': pz.order_time, 'end_time': pd.NaT,
                        'time_precision': 'minute', 'source': 'drug_order', 'source_seq': pz.order_seq, 'detail': 'pituitrin'}))
ND_SUPPORT = {'imv_ett': 'imv_ett', 'ventilator': 'ventilator', 'niv': 'niv', 'hfnc': 'hfnc', 'oxygen': 'oxygen', 'tracheostomy': 'tracheostomy',
              'rrt': 'rrt', 'rrt_chronic_home_pd': 'rrt_chronic', 'plasma_exchange': 'plasma_exchange', 'hemoperfusion': 'hemoperfusion',
              'ecmo': 'ecmo', 'cpr': 'cpr', 'defibrillation_cardioversion': 'defibrillation_cardioversion', 'arterial_bp_monitoring': 'arterial_line',
              'icp_monitoring': 'icp_monitoring', 'prone_positioning': 'prone_positioning', 'extubation': 'extubation'}
ns = N[N.category.isin(ND_SUPPORT)]
ev.append(pd.DataFrame({'visit_id': ns.visit_id, 'support': ns.category.map(ND_SUPPORT), 'event_time': ns.start_time, 'end_time': ns.end_time,
                        'time_precision': np.where(ns.start_time.isna(), None, 'minute'), 'source': 'nondrug_order', 'source_seq': ns.order_seq,
                        'detail': ns.order_name_cn}))
PR_SUPPORT = {'intubation': 'imv_ett', 'imv': 'imv_procedure', 'niv': 'niv', 'oxygen': 'oxygen', 'tracheostomy': 'tracheostomy', 'rrt': 'rrt',
              'plasma_exchange': 'plasma_exchange', 'hemoperfusion': 'hemoperfusion', 'ecmo': 'ecmo', 'cpr': 'cpr',
              'defibrillation_cardioversion': 'defibrillation_cardioversion', 'extubation': 'extubation'}
ps_ = PR[PR.proc_category.isin(PR_SUPPORT)]
ev.append(pd.DataFrame({'visit_id': ps_.visit_id, 'support': ps_.proc_category.map(PR_SUPPORT), 'event_time': ps_.proc_time, 'end_time': pd.NaT,
                        'time_precision': ps_.time_precision, 'source': 'procedure', 'source_seq': ps_.proc_seq,
                        'detail': ps_.proc_code + ' ' + ps_.proc_name_cn.fillna('')}))
SE = pd.concat(ev, ignore_index=True)
SE['end_time'] = pd.to_datetime(SE.end_time)
_g = vz.sort_values(['visit_id', 'vasoactive_agent', 'order_time']).groupby(['visit_id', 'vasoactive_agent']).order_time.diff().dt.total_seconds() / 3600
VASO_GAP = {'median_h': round(float(_g.median()), 1), 'p90_h': round(float(_g.quantile(0.9)), 1), 'n': int(_g.notna().sum())}
_ve = vz.end_time.notna() & (vz.end_time != vz.start_time)
qc('12 Organ support', 'Vasoactive-infusion orders: end time differs from start time / hours between successive orders of the same agent (median, 90th percentile)',
   f"{int(_ve.sum())} of {len(vz)} / {VASO_GAP['median_h']} h, {VASO_GAP['p90_h']} h (n = {VASO_GAP['n']})", 'info',
   'Drug orders carry no duration in this export; infusions are re-ordered about daily.')
SE = SE.sort_values(['visit_id', 'event_time', 'support', 'source', 'source_seq'], na_position='last').reset_index(drop=True)
qc('12 Organ support', 'Support events by type', SE.support.value_counts().to_dict())

In [ ]:
# First time of each support per admission. Event times with date precision are used only when no minute-precision
# event of the same support exists (then first_time = that date at 00:00 and precision = 'date').
def first_event(df):
    if df.empty:
        return pd.DataFrame(columns=['first_time', 'precision', 'source'])
    mm = df[df.time_precision.eq('minute') & df.event_time.notna()].sort_values('event_time').groupby('visit_id').first()
    dd = df[df.time_precision.eq('date') & df.event_time.notna()].sort_values('event_time').groupby('visit_id').first()
    out = mm[['event_time', 'source']].assign(precision='minute')
    add = dd.loc[dd.index.difference(out.index), ['event_time', 'source']].assign(precision='date')
    out = pd.concat([out, add]).rename(columns={'event_time': 'first_time'})
    return out
SF = V[['visit_id', 'in_supplement']].set_index('visit_id').copy()
supp_known = SF.in_supplement
def put(name, fe, known, present=None):
    present = set(fe.index) if present is None else present
    SF[f'{name}_ever'] = tri(SF.index.isin(present), known).values
    SF[f'{name}_first_time'] = fe.first_time.reindex(SF.index)
    SF[f'{name}_first_time_precision'] = fe.precision.reindex(SF.index)
    SF[f'{name}_first_source'] = fe.source.reindex(SF.index)
all_known = pd.Series(True, index=SF.index)
put('vasoactive', first_event(SE[SE.support.eq('vasoactive_infusion')]), all_known, set(SE.visit_id[SE.support.eq('vasoactive_infusion')]))
_vz = SE[SE.support.eq('vasoactive_infusion') & SE.event_time.notna()]
_t0 = _vz.visit_id.map(_vz.groupby('visit_id').event_time.min())
fa = _vz[_vz.event_time.eq(_t0)].groupby('visit_id').detail.apply(lambda s: '|'.join(sorted(set(s))))   # agents ordered at the first minute
SF['vasoactive_first_agent'] = fa.reindex(SF.index)
SF['vasoactive_n_agents'] = SE[SE.support.eq('vasoactive_infusion')].groupby('visit_id').detail.nunique().reindex(SF.index).fillna(0).astype(int)
SF['pituitrin_infusion_ever'] = SF.index.isin(set(SE.visit_id[SE.support.eq('pituitrin_infusion')]))
# Invasive mechanical ventilation
ett_v = set(SE.visit_id[SE.support.isin(['imv_ett', 'imv_procedure'])])
vent_v = set(SE.visit_id[SE.support.eq('ventilator')])
niv_v = set(SE.visit_id[SE.support.eq('niv')])
SF['imv_evidence'] = None
SF.loc[SF.index.isin(ett_v), 'imv_evidence'] = 'definite'
SF.loc[SF.index.isin(vent_v - ett_v - niv_v), 'imv_evidence'] = 'probable'
SF.loc[SF.index.isin((vent_v - ett_v) & niv_v), 'imv_evidence'] = 'uncertain'
imv_vis = set(SF.index[SF.imv_evidence.isin(['definite', 'probable'])])
use = SE.support.isin(['imv_ett', 'imv_procedure']) | (SE.support.eq('ventilator') & ~SE.visit_id.isin(niv_v))
fe = first_event(SE[use & SE.visit_id.isin(imv_vis)])
put('imv', fe, supp_known, imv_vis)
SF.loc[SF.in_supplement & SF.imv_evidence.isna(), 'imv_evidence'] = 'none'
for s in ['niv', 'hfnc', 'oxygen', 'tracheostomy', 'rrt', 'plasma_exchange', 'hemoperfusion', 'ecmo', 'cpr']:
    put(s, first_event(SE[SE.support.eq(s)]), supp_known, set(SE.visit_id[SE.support.eq(s)]))
SF['rrt_chronic_ever'] = tri(SF.index.isin(set(SE.visit_id[SE.support.eq('rrt_chronic')])), supp_known).values
SF = SF.drop(columns='in_supplement').reset_index()
qc('12 Organ support', 'Admissions with vasoactive infusion (all admissions)', int(SF.vasoactive_ever.eq(True).sum()))
qc('12 Organ support', 'Vasoactive first agent', SF.vasoactive_first_agent.value_counts().to_dict())
_sup = SF.merge(V[['visit_id', 'in_supplement']], on='visit_id')
_sup = _sup[_sup.in_supplement]
qc('12 Organ support', 'IMV evidence (supplement admissions)', _sup.imv_evidence.value_counts().to_dict(), 'info',
   'definite = intubation/ETT order or code 96.04/96.70-96.72; probable = ventilator order without any NIV evidence; '
   'uncertain = ventilator order only, with NIV evidence (not counted as IMV). Missing for admissions outside the supplement.')
qc('12 Organ support', 'IMV first time precision (admissions with IMV)', _sup[_sup.imv_ever.eq(True)].imv_first_time_precision.fillna('no time recorded').value_counts().to_dict(), 'info',
   'no time recorded: evidence without any recorded time (e.g. a procedure code without a procedure time).')
for s in ['vasoactive', 'imv', 'niv', 'rrt', 'ecmo', 'cpr']:
    bad = SF[f'{s}_ever'].eq(True) & SF[f'{s}_first_time'].isna()
    qc('12 Organ support', f'{s}: ever = True but no event time', int(bad.sum()), 'info')
for s in ['niv', 'hfnc', 'rrt', 'plasma_exchange', 'ecmo', 'cpr']:
    qc('12 Organ support', f'Admissions with {s} (supplement admissions)', int(SF[f'{s}_ever'].eq(True).sum()))

In [ ]:
# Outcomes: in-hospital death, discharge against medical advice (DAMA) and a rule-based classification of DAMA in a
# critical condition, read from the final discharge record (text is used here only; excerpts go to the review sheet).
OUTC = V[['visit_id', 'in_supplement', 'disposition', 'outcome', 'discharge_time']].copy()
dead = OUTC.disposition.eq('death') | OUTC.outcome.eq('death')
# Death is known for supplement admissions with a recorded disposition or outcome (2 have neither).
OUTC['death'] = tri(dead.values, (OUTC.in_supplement & (OUTC.disposition.notna() | OUTC.outcome.notna())).values).values
OUTC['death_time'] = OUTC.discharge_time.where(dead)
OUTC['dama'] = tri(OUTC.disposition.eq('dama').values, (OUTC.in_supplement & OUTC.disposition.notna()).values).values
dr2 = DR.copy()
def final_text(g):
    parts = []
    for _, r in g.iterrows():
        course = '' if is_missing(r.get('诊疗经过')) else str(r.get('诊疗经过'))
        parts.append(course[-400:])
        for c in ['出院情况', '出院医嘱']:
            if not is_missing(r.get(c)):
                parts.append(str(r.get(c)))
    return '。'.join(parts)
TXT = dr2.groupby('visit_id').apply(final_text)
NEG = r'(?:无|未|否认|不|没有|未见|非)\S{0,3}$'
P_WITHDRAW = r'放弃(?:治疗|抢救|进一步|一切|积极|继续|有创|呼吸机|气管插管|后续)|拒绝(?:进一步|继续|抢救|气管插管|插管|有创|治疗)'
P_TERMINAL = r'带(?:气管)?(?:插)?管出院|带呼吸机|呼吸机(?:辅助)?(?:通气)?下(?:出院|转运|离院)|拔除气管插管后(?:出院|离院)|撤(?:除)?呼吸机|简易呼吸器'
P_MORIBUND = r'随时(?:可能)?(?:出现)?(?:死亡|心跳骤停|呼吸心跳骤停|生命危险)|临终|濒死|脑死亡|无自主呼吸|瞳孔散大|对光反射消失|深昏迷|血压(?:测不出|难以维持)|病危|病情危重|危重状态'
P_IMPROVED = r'好转|痊愈|治愈|生命体征平稳|生命体征稳定|病情平稳|病情稳定|一般情况(?:可|好)|精神(?:可|好|反应可)'
P_NOT_IMPROVED = r'无好转|未见好转|无明显好转|未好转|未见明显好转|无改善|无明显改善'
def clause_hits(text, pat, negatable=True):
    hits = []
    for cl in re.split(r'[。；;！!？?\n，,]', text or ''):
        for m in re.finditer(pat, cl):
            if negatable and re.search(NEG, cl[:m.start()]):
                continue
            if pat == P_IMPROVED and re.search(P_NOT_IMPROVED, cl):
                continue
            hits.append(cl.strip()[:80])
            break
    return hits
dama_vis = set(OUTC.visit_id[OUTC.dama.eq(True)])
dis_t = V.set_index('visit_id').discharge_time
# Organ support in the 24 h before discharge:
#  * a vasoactive-infusion order started in that window (drug orders carry no duration and are re-ordered about daily);
#  * an invasive-ventilation order active in that window: intubation/ETT orders, and ventilator orders in admissions
#    without NIV evidence (start <= discharge and end >= discharge - 24 h; an order without an end time counts when it
#    starts in the window). Long-term orders end when stopped, at the latest at discharge;
#  * an IMV procedure with a clock time in the window.
sup = SE[SE.time_precision.eq('minute') & (SE.support.isin(['vasoactive_infusion', 'imv_ett', 'imv_procedure'])
         | (SE.support.eq('ventilator') & ~SE.visit_id.isin(niv_v)))].copy()
dc_ = sup.visit_id.map(dis_t)
w0 = dc_ - pd.Timedelta(hours=24)
start_in = (sup.event_time >= w0) & (sup.event_time <= dc_)
running = sup.source.eq('nondrug_order') & (sup.event_time <= dc_) & sup.end_time.notna() & (sup.end_time >= w0)
support_dc = set(sup.visit_id[start_in | running])
support_dc_start = set(sup.visit_id[start_in])
rows = []
for vid in sorted(dama_vis):
    t = TXT.get(vid)
    w, tm, mb, im = (clause_hits(t, P_WITHDRAW, False), clause_hits(t, P_TERMINAL, False), clause_hits(t, P_MORIBUND), clause_hits(t, P_IMPROVED))
    ni = clause_hits(t, P_NOT_IMPROVED, False)
    sup = vid in support_dc
    if w or tm or sup:
        cat = 'A_withdrawal_terminal_or_support_at_discharge'
    elif mb and not im:
        cat = 'B_moribund_without_improvement'
    elif im and not mb and not ni:
        cat = 'D_improved_or_stable'
    else:
        cat = 'C_uncertain'
    rows.append({'visit_id': vid, 'discharge_record_available': t is not None, 'dama_text_withdrawal': bool(w),
                 'dama_text_terminal_support': bool(tm), 'dama_support_last_24h': sup, 'dama_text_moribund': bool(mb),
                 'dama_text_improved': bool(im), 'dama_text_not_improved': bool(ni), 'dama_critical_category': cat,
                 '_excerpts': ' | '.join((w + tm + mb + ni + im)[:4])})
DC = pd.DataFrame(rows)
OUTC = OUTC.merge(DC.drop(columns='_excerpts'), on='visit_id', how='left')
crit = OUTC.dama_critical_category.isin(['A_withdrawal_terminal_or_support_at_discharge', 'B_moribund_without_improvement'])
OUTC['dama_critical'] = tri(crit.values, OUTC.dama.notna().values).values
OUTC['dama_critical_needs_review'] = OUTC.dama_critical_category.eq('C_uncertain').fillna(False).astype(bool)
OUTC['death_or_dama_critical'] = OUTC.death.astype('boolean') | OUTC.dama_critical.astype('boolean')   # Kleene logic: True if either is True
# Clinician adjudication (from v1.2). The workbook is the review sheet of the previous release with a decision in each row; only the
# decision is used here (reviewer notes and excerpts stay in adjudication_INTERNAL). Final label: the decision where it is critical or
# not_critical; the rule label where the clinician was unsure or the admission was not reviewed.
ADJ_VALID = ['critical', 'not_critical', 'unsure']
ADJ_DEC, ADJ_NOTE = {}, {}
if ADJ_INPUTS:
    _a = pd.read_excel(ADJ_INPUTS[0]['path'], sheet_name='dama_review', dtype=object, engine=ENGINE)
    _a.columns = [str(c).strip() for c in _a.columns]
    _need = {'visit_id', 'dama_critical_category', ADJ_DECISION_COL}
    if not _need <= set(_a.columns):
        raise RuntimeError(f'Adjudication workbook lacks columns: {sorted(_need - set(_a.columns))}')
    _a['visit_id'] = norm_id(_a.visit_id)
    _a['_dec'] = _a[ADJ_DECISION_COL].map(S).map(lambda x: x.lower() if x else None)
    _bad = int((_a._dec.notna() & ~_a._dec.isin(ADJ_VALID)).sum())
    _dup = int(_a.visit_id.duplicated().sum())
    _not_dama = len(set(_a.visit_id.dropna()) - dama_vis)
    qc('13 Outcomes', 'Adjudication workbook: rows / duplicated admissions / admissions without DAMA in this build / invalid decisions',
       f'{len(_a)} / {_dup} / {_not_dama} / {_bad}', 'pass' if not (_dup or _not_dama or _bad) else 'fail',
       'Valid decisions: ' + ', '.join(ADJ_VALID) + '.')
    if _dup or _not_dama or _bad:
        raise RuntimeError('The adjudication workbook does not match this build (duplicated admissions, admissions without DAMA, or invalid decisions).')
    _shown = _a.set_index('visit_id').dama_critical_category.map(S)
    _now = DC.set_index('visit_id').dama_critical_category
    _common = _now.index.intersection(_shown.index)
    _mism = int((_shown.loc[_common] != _now.loc[_common]).sum())
    qc('13 Outcomes', 'DAMA admissions whose rule category in the workbook differs from this build', _mism, 'pass' if _mism == 0 else 'warn',
       'The reviewer saw the rule category of the previous release.')
    ADJ_DEC = _a.dropna(subset=['_dec']).set_index('visit_id')._dec.to_dict()
    if 'reviewer_note' in _a.columns:
        ADJ_NOTE = _a.set_index('visit_id').reviewer_note.map(S).dropna().to_dict()
_unrev = len(dama_vis - set(ADJ_DEC))
qc('13 Outcomes', 'DAMA admissions without a clinician decision (the rule label is used)', _unrev, 'pass' if _unrev == 0 else 'warn')
OUTC['dama_review_decision'] = OUTC.visit_id.map(ADJ_DEC)
_dec = OUTC.dama_review_decision
OUTC['dama_critical_final'] = (OUTC.dama_critical.astype('boolean').mask(_dec.eq('critical').fillna(False).astype(bool), True)
                               .mask(_dec.eq('not_critical').fillna(False).astype(bool), False))
OUTC['dama_critical_final_source'] = pd.Series(np.select(
    [_dec.isin(['critical', 'not_critical']).values, _dec.eq('unsure').fillna(False).astype(bool).values, OUTC.dama.eq(True).fillna(False).astype(bool).values],
    ['clinician', 'rule (clinician unsure)', 'rule (not reviewed)'], ''), index=OUTC.index).replace('', np.nan)
OUTC['death_or_dama_critical_final'] = OUTC.death.astype('boolean') | OUTC.dama_critical_final.astype('boolean')
OUTC = OUTC.drop(columns=['in_supplement', 'disposition', 'outcome', 'discharge_time'])
qc('13 Outcomes', 'In-hospital deaths (supplement admissions)', int(OUTC.death.eq(True).sum()))
qc('13 Outcomes', 'Discharges against medical advice (DAMA)', int(OUTC.dama.eq(True).sum()))
qc('13 Outcomes', 'DAMA critical-condition categories', DC.dama_critical_category.value_counts().to_dict(), 'info',
   'A and B are classified as critical by the rule; the rule does not decide C (clinician decisions below).')
_dsup = DC[DC.dama_support_last_24h]
qc('13 Outcomes', 'DAMA with organ support in the 24 h before discharge: order started in the window / ventilation order running into the window only',
   f'{int(_dsup.visit_id.isin(support_dc_start).sum())} / {int((~_dsup.visit_id.isin(support_dc_start)).sum())}', 'info')
_conf = DC.dama_critical_category.str.startswith('A') & DC.dama_text_improved
qc('13 Outcomes', 'Category A with improvement wording in the final record (listed second for review)', int(_conf.sum()), 'info')
qc('13 Outcomes', 'Death or critical DAMA', int(OUTC.death_or_dama_critical.eq(True).sum()))
qc('13 Outcomes', 'Clinician decisions on DAMA records', ({k: int(pd.Series(ADJ_DEC).eq(k).sum()) for k in ADJ_VALID} if ADJ_DEC else 'none'), 'info')
_xt = pd.crosstab(OUTC.dama_critical_category, OUTC.dama_review_decision.fillna('not reviewed'))
qc('13 Outcomes', 'Rule category by clinician decision', {f'{i} / {j}': int(_xt.loc[i, j]) for i in _xt.index for j in _xt.columns if _xt.loc[i, j]}, 'info')
qc('13 Outcomes', 'Critical DAMA: by rule / final label (clinician decision, rule where unsure or not reviewed)',
   f"{int(OUTC.dama_critical.eq(True).sum())} / {int(OUTC.dama_critical_final.eq(True).sum())}", 'info')
qc('13 Outcomes', 'Critical DAMA by the final label, by source', OUTC[OUTC.dama_critical_final.eq(True)].dama_critical_final_source.value_counts().to_dict(), 'info')
qc('13 Outcomes', 'Death or critical DAMA (final label)', int(OUTC.death_or_dama_critical_final.eq(True).sum()))
# Review sheet for DAMA (internal; contains short excerpts of the discharge record)
REV = DC.merge(V[['visit_id', 'patient_id', 'admit_time', 'discharge_time', 'age_years']], on='visit_id', how='left')
REV['any_icu_stay'] = REV.visit_id.isin(set(IS.visit_id))
REV['review_priority'] = np.select(
    [REV.dama_critical_category.eq('C_uncertain'), REV.dama_critical_category.str.startswith('A') & REV.dama_text_improved,
     REV.dama_critical_category.str.startswith('B'), REV.dama_critical_category.str.startswith('A')],
    ['1 rule cannot decide (C)', '2 critical by rule (A) but improvement wording', '3 critical by moribund wording (B)', '4 critical by rule (A)'],
    '5 improved or stable (D)')
REV = REV.sort_values(['review_priority', 'discharge_time']).reset_index(drop=True)
REV = REV.rename(columns={'_excerpts': 'excerpts_from_final_discharge_record'})
REV['reviewer_decision (critical / not_critical / unsure)'] = REV.visit_id.map(ADJ_DEC).fillna('')   # prefilled from the adjudication workbook
REV['reviewer_note'] = REV.visit_id.map(ADJ_NOTE).fillna('')
REV = REV[['review_priority', 'visit_id', 'patient_id', 'admit_time', 'discharge_time', 'age_years', 'any_icu_stay', 'discharge_record_available',
           'dama_critical_category', 'dama_support_last_24h', 'dama_text_withdrawal', 'dama_text_terminal_support', 'dama_text_moribund',
           'dama_text_improved', 'dama_text_not_improved', 'excerpts_from_final_discharge_record',
           'reviewer_decision (critical / not_critical / unsure)', 'reviewer_note']]
log('Support events:', len(SE), '| DAMA:', len(DC), '| categories:', DC.dama_critical_category.value_counts().to_dict())

In [ ]:
# Visit-level summary (one row per inpatient admission).
first_icu = IS[IS.stay_seq.eq(1)].set_index('visit_id')
VSUM = V[['visit_id', 'patient_id', 'in_supplement', 'admit_time', 'discharge_time', 'admit_year', 'age_days', 'age_years', 'sex',
          'admission_source', 'disposition', 'outcome', 'los_hours']].copy()
VSUM['any_icu_stay'] = VSUM.visit_id.isin(set(IS.visit_id))
VSUM['any_picu_stay'] = VSUM.visit_id.isin(set(IS.visit_id[IS.unit.isin(['picu_main', 'picu_west'])]))
VSUM['n_icu_stays'] = VSUM.visit_id.map(IS.groupby('visit_id').size()).fillna(0).astype(int)
for c in ['unit', 'in_time', 'out_time', 'in_source', 'out_source', 'timing_complete', 'los_hours', 'hours_admission_to_icu', 'age_days_at_icu_in']:
    VSUM[f'first_icu_{c}'] = VSUM.visit_id.map(first_icu[c])
VSUM = VSUM.merge(SF, on='visit_id', how='left').merge(OUTC, on='visit_id', how='left')
qc('14 Summary', 'Admissions with any ICU stay / PICU stay', f"{int(VSUM.any_icu_stay.sum())} / {int(VSUM.any_picu_stay.sum())}")
qc('14 Summary', 'Supplement admissions with a PICU stay', int((VSUM.any_picu_stay & VSUM.in_supplement).sum()))

In [ ]:
# Table specifications: every column of every output table is declared here (type, unit, description). The build stops
# if a table has an undeclared column, so the data dictionary is always complete.
ID, TXT_, CAT, DT_, FL, INT, BOOL = 'id', 'text', 'category', 'datetime', 'float', 'int', 'bool'
SPEC = {}
def spec(table, desc, cols):
    SPEC[table] = {'desc': desc, 'cols': cols}

spec('patients', 'One row per patient.', [
    ('patient_id', ID, '', 'Research patient number (科研患者编号).'),
    ('sex', CAT, '', 'male / female (from the admission records; identical across admissions).'),
    ('ethnicity_cn', TXT_, '', 'Ethnicity as recorded (民族), Chinese.'),
    ('home_county_cn', TXT_, '', 'Registered home county or district (户口地址-县), Chinese. Internal use only.'),
    ('residence_wuhan', BOOL, '', 'Home district is one of the 13 districts of Wuhan; missing when the district is not recorded.'),
    ('n_admissions', INT, '', 'Number of inpatient admissions in the export.'),
    ('first_admit_time', DT_, '', 'Earliest admission time in the export.')])
spec('admissions', 'One row per inpatient admission (main export, 病案首页基本信息), with supplementary fields.', [
    ('visit_id', ID, '', 'Research admission number (科研就诊编号).'),
    ('patient_id', ID, '', 'Research patient number.'),
    ('in_supplement', BOOL, '', 'Admission is covered by the supplementary export (the export query\'s index admissions, mostly with an encephalopathy-related diagnosis). Supplement-only variables are missing when False.'),
    ('admit_time', DT_, '', 'Hospital admission time (入院日期).'),
    ('discharge_time', DT_, '', 'Hospital discharge time (出院日期); missing when not recorded (mostly admissions still open at export).'),
    ('discharge_time_missing', BOOL, '', 'Discharge time not recorded.'),
    ('sex', CAT, '', 'male / female.'),
    ('age_years', FL, 'years', 'Age at admission (年龄（岁）, decimal years as exported).'),
    ('age_months', FL, 'months', 'Age at admission (年龄（月）, decimal months as exported).'),
    ('age_days', FL, 'days', 'Age at admission (年龄（天）).'),
    ('admit_ward', TXT_, '', 'Admission ward (入院科别), Chinese ward name.'),
    ('discharge_ward', TXT_, '', 'Discharge ward (出院科别), Chinese ward name.'),
    ('transfer_path', TXT_, '', 'Ward path text (转科科别), e.g. "小儿神经二内科->重症医学科"; no times.'),
    ('admit_unit', CAT, '', 'Unit class of the admission ward: picu_main, picu_west, other_icu, neonatal_ward, ward.'),
    ('discharge_unit', CAT, '', 'Unit class of the discharge ward (same classes).'),
    ('admission_source_raw', TXT_, '', 'Admission route as recorded (入院途径), supplement only.'),
    ('admission_source', CAT, '', 'outpatient_clinic / emergency / transfer_other_hospital / other.'),
    ('disposition_raw', TXT_, '', 'Discharge disposition as recorded (离院方式), supplement only.'),
    ('disposition', CAT, '', 'discharged / dama (discharge against medical advice) / transfer_hospital / transfer_community / death / other.'),
    ('outcome_raw', TXT_, '', 'Outcome at discharge as recorded (转归情况), supplement only.'),
    ('outcome', CAT, '', 'cured / improved / not_improved / death / other.'),
    ('los_days_recorded', FL, 'days', 'Length of stay recorded on the front page (实际住院天数).'),
    ('los_hours', FL, 'hours', 'Discharge time minus admission time.'),
    ('rescue_any', BOOL, '', 'Rescue (抢救) recorded on the front page; supplement only.'),
    ('rescue_count', FL, '', 'Number of rescues (抢救次数).'),
    ('rescue_success_count', FL, '', 'Number of successful rescues (成功次数).'),
    ('admit_year', INT, '', 'Calendar year of admission.')])
spec('outpatient_visits', 'Outpatient and emergency visits listed in the supplementary export (患者就诊信息 rows without an admission time).', [
    ('visit_id', ID, '', 'Research visit number.'), ('patient_id', ID, '', 'Research patient number.'),
    ('visit_time', DT_, '', 'Visit time (门诊就诊时间).'), ('department_cn', TXT_, '', 'Department (门诊就诊科室), Chinese.'),
    ('age_years', FL, 'years', 'Age at the visit.'), ('age_days', FL, 'days', 'Age at the visit.')])
spec('transfers', 'Ward transfers (患者转科信息, supplement), with quality flags.', [
    ('visit_id', ID, '', 'Research admission number.'), ('patient_id', ID, '', 'Research patient number.'),
    ('transfer_seq', INT, '', 'Order of the transfer within the admission (by time).'),
    ('transfer_time', DT_, '', 'Transfer time as recorded (转科时间).'),
    ('from_ward', TXT_, '', 'Ward transferred from (转出科室).'), ('to_ward', TXT_, '', 'Ward transferred to (转入科室).'),
    ('from_unit', CAT, '', 'Unit class of from_ward.'), ('to_unit', CAT, '', 'Unit class of to_ward.'),
    ('n_source_rows', INT, '', 'Number of identical rows collapsed into this row.'),
    ('flag_unknown_visit', BOOL, '', 'Admission not found in the main export (not used).'),
    ('flag_after_discharge', BOOL, '', 'Recorded after hospital discharge (not used in the ward timeline).'),
    ('flag_before_admission', BOOL, '', 'Recorded before the admission time.'),
    ('minutes_before_admission', FL, 'minutes', 'How long before the admission time (transfers up to 24 h before admission are placed at admission).'),
    ('flag_to_ward_missing', BOOL, '', 'Destination ward missing (destination treated as unknown).'),
    ('flag_missing_time', BOOL, '', 'Transfer time missing (not used).'),
    ('used_in_timeline', BOOL, '', 'Transfer used to build the ward timeline and ICU stays.'),
    ('timeline_time', DT_, '', 'Time used in the timeline (transfer time; admission time for transfers recorded before admission).')])
spec('icu_stays', 'ICU stays derived from the ward timeline (admission ward, transfers, discharge ward); one row per stay in one ICU unit.', [
    ('visit_id', ID, '', 'Research admission number.'), ('patient_id', ID, '', 'Research patient number.'),
    ('stay_seq', INT, '', 'Order of the stay within the admission.'),
    ('unit', CAT, '', 'picu_main (重症医学科病区), picu_west (儿童重症监护室(西院)), other_icu (adult general or surgical ICU).'),
    ('ward_cn', TXT_, '', 'ICU ward name as recorded.'),
    ('in_time', DT_, '', 'ICU entry time; missing when unknown.'),
    ('in_source', CAT, '', 'hospital_admission (admitted directly to the ICU), transfer, or unknown.'),
    ('in_time_lower_bound', DT_, '', 'When in_time is unknown: the last known time before ICU entry.'),
    ('out_time', DT_, '', 'ICU exit time; missing when unknown.'),
    ('out_source', CAT, '', 'transfer, hospital_discharge (discharged or died in the ICU), or unknown (no transfer out recorded).'),
    ('out_time_upper_bound', DT_, '', 'When out_time is unknown: hospital discharge time (the exit happened no later than this).'),
    ('source', CAT, '', 'transfer_records (supplement admissions), ward_fields_only (no transfer records available), transfer_path_text_only (ICU in the ward path text but no times).'),
    ('in_supplement', BOOL, '', 'Admission covered by the supplementary export.'),
    ('timing_complete', BOOL, '', 'Both entry and exit times are known.'),
    ('los_hours', FL, 'hours', 'ICU length of stay (when timing is complete).'),
    ('hours_admission_to_icu', FL, 'hours', 'ICU entry minus hospital admission.'),
    ('age_days_at_icu_in', FL, 'days', 'Age at ICU entry.'),
    ('icu_episode', INT, '', 'Contiguous ICU episode number within the admission (a direct transfer between ICUs stays in one episode).')])
spec('diagnoses_coded', 'Coded discharge diagnoses (病案出院诊断归一, supplement), in export order.', [
    ('visit_id', ID, '', 'Research admission number.'), ('diagnosis_cn', TXT_, '', 'Diagnosis name (诊断疾病名称).'),
    ('icd10_code', TXT_, '', 'Local ICD-10 code (诊断疾病编码).'), ('dx_seq', INT, '', 'Order within the admission as exported (the principal diagnosis is not flagged in the source).'),
    ('icd10_3char', TXT_, '', 'ICD-10 three-character category.')])
spec('diagnoses_listed', 'Diagnosis lists as written: discharge records (admission and discharge diagnoses, all admissions); admission notes, outpatient and emergency records (preliminary diagnoses) and front-page discharge diagnoses before coding (supplement).', [
    ('visit_id', ID, '', 'Research admission number; for source outpatient_record_preliminary_dx the outpatient or emergency visit number (outpatient_visits).'),
    ('source', CAT, '', 'discharge_record_admission_dx (入院诊断), discharge_record_discharge_dx (出院诊断), admission_note_preliminary_dx (入院记录 初步诊断), '
     'outpatient_record_preliminary_dx (门急诊病历 初步诊断), front_page_discharge_dx_uncoded (病案出院诊断编目前).'),
    ('record_seq', INT, '', 'Record number when an admission has more than one record.'), ('item_seq', INT, '', 'Item number within the list.'),
    ('item_type', CAT, '', 'diagnosis or remark (诊断备注).'), ('text_cn', TXT_, '', 'Diagnosis text, Chinese.'),
    ('narrative_cut', BOOL, '', 'Narrative text that followed this item in the same field was removed (the rest of that field is not used).')])
spec('procedures', 'Procedures (病案手术操作归一, supplement, ICD-9-CM-3).', [
    ('visit_id', ID, '', 'Research admission number.'), ('proc_time', DT_, '', 'Procedure date/time (手术操作日期).'),
    ('proc_code', TXT_, '', 'ICD-9-CM-3 code (手术操作编码).'), ('proc_name_cn', TXT_, '', 'Procedure name (手术操作名称).'),
    ('n_source_rows', INT, '', 'Number of identical rows collapsed into this row.'),
    ('time_precision', CAT, '', 'minute, or date when the recorded time is exactly 00:00:00 (date only).'),
    ('proc_category', CAT, '', 'Support/procedure category (see definitions.md), other otherwise.'),
    ('proc_seq', INT, '', 'Order within the admission (by time).')])
spec('drug_orders', 'Drug orders (住院用药医嘱; main export for all admissions; supplementary copies verified identical and not duplicated).', [
    ('visit_id', ID, '', 'Research admission number.'), ('order_time', DT_, '', 'Order time (开立时间).'),
    ('start_time', DT_, '', 'Start time (开始时间; equal to the order time in every order of this export).'),
    ('end_time', DT_, '', 'End time (结束时间); often equal to the order time for one-time orders.'),
    ('drug_name_cn', TXT_, '', 'Drug name as ordered (药品名称).'), ('route_raw', TXT_, '', 'Route as ordered (用法).'),
    ('n_source_rows', INT, '', 'Number of fully identical rows (same admission, order, start and end time, drug and route) collapsed into this row.'),
    ('route_class', CAT, '', 'iv_infusion, iv_push, im, sc, intradermal, enteral, inhaled, endotracheal, rectal, intrathecal, topical_local, dialysis_related, flush_lock, discharge_medication, procedure_related, intraarterial, unspecified, other.'),
    ('vasoactive_agent', CAT, '', 'Vasoactive or related agent identified from the drug name (see definitions.md).'),
    ('compound_with_diluent', BOOL, '', 'A diluent line (sodium chloride, glucose, water for injection) was ordered in the same admission at the same minute.'),
    ('vasoactive_infusion', BOOL, '', 'Order meets the vasoactive-infusion definition (epinephrine, norepinephrine, dopamine, dobutamine, milrinone or vasopressin by continuous intravenous route).'),
    ('pituitrin_infusion', BOOL, '', 'Posterior pituitary injection (垂体后叶) by infusion; reported separately (often used for bleeding).'),
    ('drug_classes', TXT_, '', 'Drug classes matched by name lists (pipe-separated): sedative_analgesic, neuromuscular_blocker, antiseizure, corticosteroid, immunoglobulin, il6_receptor_blocker, osmotherapy, albumin.'),
    ('order_seq', INT, '', 'Order within the admission (by time).')])
spec('nondrug_orders', 'Non-drug orders (住院非用药医嘱, supplement) with support categories.', [
    ('visit_id', ID, '', 'Research admission number.'), ('start_time', DT_, '', 'Start time (开始时间).'), ('end_time', DT_, '', 'End time (结束时间).'),
    ('order_name_cn', TXT_, '', 'Order name as recorded; names starting with "\\" are free-text instructions.'),
    ('n_source_rows', INT, '', 'Number of identical rows collapsed into this row.'),
    ('is_instruction', BOOL, '', 'Free-text instruction order (name starts with "\\").'),
    ('category', CAT, '', 'Support category (see definitions.md) or other.'), ('order_seq', INT, '', 'Order within the admission (by time).')])
spec('vital_signs', 'Charted vital signs (患者生命体征). Supplement admissions use the supplementary rows (adds monitor heart rate, SpO2, stool count); others the main export. Implausible values are set to missing (see vital_signs_cleaning_log).', [
    ('visit_id', ID, '', 'Research admission number.'), ('measure_time', DT_, '', 'Measurement time (测量时间).'),
    ('resp_rate', FL, 'breaths/min', 'Respiratory rate.'), ('pulse', FL, 'beats/min', 'Pulse rate (脉搏).'),
    ('heart_rate_monitor', FL, 'beats/min', 'Heart rate (心率, monitor); supplement admissions only.'),
    ('spo2', FL, '%', 'Pulse oximetry (血氧); supplement admissions only.'), ('temp_c', FL, 'degrees C', 'Body temperature.'),
    ('sbp', FL, 'mmHg', 'Systolic blood pressure.'), ('dbp', FL, 'mmHg', 'Diastolic blood pressure.'),
    ('height_cm', FL, 'cm', 'Height.'), ('weight_kg', FL, 'kg', 'Weight.'), ('stool_count', FL, '', 'Stool count (supplement only).'),
    ('urine_ml', FL, 'mL', 'Urine output entry.'), ('output_total_ml', FL, 'mL', 'Total output entry.'), ('intake_ml', FL, 'mL', 'Intake entry.'),
    ('source', CAT, '', 'main or supplement export.'),
    ('heart_rate', FL, 'beats/min', 'Monitor heart rate when recorded, otherwise pulse.'),
    ('heart_rate_source', CAT, '', 'monitor_heart_rate or pulse.'),
    ('n_source_rows', INT, '', 'Number of identical rows (same admission, time and values) collapsed into this row.'),
    ('row_seq', INT, '', 'Row number among rows sharing the same timestamp (rows with different values are kept as separate measurements).')])
spec('vital_signs_cleaning_log', 'Vital-sign values set to missing, with the raw value and the reason.', [
    ('visit_id', ID, '', 'Research admission number.'), ('measure_time', DT_, '', 'Measurement time.'),
    ('row_seq', INT, '', 'row_seq of the vital_signs row (visit_id, measure_time, row_seq identify it).'),
    ('variable', CAT, '', 'Column in vital_signs.'), ('raw_value', FL, '', 'Value as exported.'), ('reason', CAT, '', 'zero (not recorded) / outside plausible range / outside range or >= SBP.')])
spec('labs', 'Quantitative laboratory results: one row per report and analyte, standard units, report time only (the platform has no specimen collection time).', [
    ('visit_id', ID, '', 'Visit number on the report (就诊编号); may be an emergency or outpatient number.'),
    ('lab_patient_id', ID, '', 'Patient number on the report (患者编号).'),
    ('report_id', ID, '', 'Report number (报告编号).'), ('report_time', DT_, '', 'Report time (报告时间).'),
    ('specimen_raw', TXT_, '', 'Specimen as recorded (样本类型).'), ('panel', CAT, '', 'Laboratory panel (source sheet), English code.'),
    ('analyte_cn', TXT_, '', 'Analyte name as exported.'), ('source_role', CAT, '', 'Export the row was taken from (main or supplement; copies are identical).'),
    ('unit_raw', TXT_, '', 'Unit as exported.'),
    ('value', FL, 'unit column', 'Result in the standard unit; for censored results the reporting limit.'),
    ('censor', CAT, '', '"<" or ">" when the result was reported beyond a limit; empty otherwise.'),
    ('result_flag', TXT_, '', 'Parsing and unit flags (pipe-separated): value_from_display, annotated, non_numeric, range_text, no_clot, above_range_no_value, unit_converted, unit_relabeled, unit_missing_assumed_standard.'),
    ('value_raw', TXT_, '', 'Raw result string (quantitative cell, else displayed result).'),
    ('specimen', CAT, '', 'Standard specimen (serum, plasma, venous_blood, whole_blood, capillary_blood, peripheral_blood, arterial_blood, urine, csf, ...).'),
    ('specimen_class', CAT, '', 'blood, urine, csf, respiratory, stool, other.'),
    ('analyte', CAT, '', 'English analyte code; prefixed with csf_ or urine_ for those specimens.'),
    ('unit', CAT, '', 'Standard unit of value.'),
    ('ref_low', FL, 'unit column', 'Lower reference limit in the standard unit.'), ('ref_high', FL, 'unit column', 'Upper reference limit in the standard unit.'),
    ('specimen_note', TXT_, '', 'Note when the specimen differs from the panel default.'),
    ('outside_physiological_limits', BOOL, '', 'Value outside hard physiological limits (flag only; value kept as reported).'),
    ('patient_id', ID, '', 'Patient number from the linked admission (else lab_patient_id).'),
    ('visit_link', CAT, '', 'same_admission, outpatient_visit, unlisted_visit, with suffix _within_24h_before_admission or _during_admission when linked by patient and time.'),
    ('linked_admission_id', ID, '', 'Admission the result belongs to (same admission, or linked by patient and time).'),
    ('patient_id_mismatch', BOOL, '', 'Patient number on the report differs from the admission\'s patient.')])
spec('lab_qualitative', 'Qualitative and pathogen tests (one row per report and test; the multiplex sequencing panel has one row per organism).', [
    ('visit_id', ID, '', 'Visit number on the report.'), ('lab_patient_id', ID, '', 'Patient number on the report.'),
    ('report_id', ID, '', 'Report number (病理号 for the 7-virus antigen panel).'), ('report_time', DT_, '', 'Report time (date only for the 7-virus panel).'),
    ('specimen_raw', TXT_, '', 'Specimen as recorded.'), ('panel', CAT, '', 'Panel (source sheet), English code.'),
    ('analyte_cn', TXT_, '', 'Test name as exported.'), ('source_role', CAT, '', 'Export the row was taken from.'),
    ('result_raw', TXT_, '', 'Result string as exported (displayed result, else qualitative cell).'),
    ('test', CAT, '', 'English test code.'), ('organism_cn', TXT_, '', 'Organism reported by the multiplex sequencing panel.'),
    ('result', CAT, '', 'negative, positive, weak_positive, detected, not_detected, numeric, reported_without_result, no_result, or CSF categories.'),
    ('reads', FL, 'reads', 'Sequence reads (multiplex panel).'), ('load_raw', TXT_, '', 'Load as reported (multiplex panel).'),
    ('result_value', FL, '', 'Numeric result when reported (e.g. copies).'),
    ('specimen', CAT, '', 'Standard specimen.'), ('specimen_class', CAT, '', 'Specimen class.'),
    ('time_precision', CAT, '', 'minute or date.'), ('patient_id', ID, '', 'Patient number (linked admission, else report).'),
    ('visit_link', CAT, '', 'As in labs.'), ('linked_admission_id', ID, '', 'As in labs.'), ('patient_id_mismatch', BOOL, '', 'As in labs.')])
spec('glucose_untimed', 'Point-of-care glucose (血糖检测): the export has no time for these values.', [
    ('visit_id', ID, '', 'Research admission number.'), ('value_raw', TXT_, '', 'Value as exported.'),
    ('glucose_mmol_L', FL, 'mmol/L', 'Parsed glucose.'), ('censor', CAT, '', '"<" or ">" for censored values.'),
    ('result_flag', TXT_, '', 'Parsing flag.'),
    ('row_seq', INT, '', 'Row number within the admission (export order). Every exported row is kept: the sheet has no time or report number, so equal values are separate measurements.')])
spec('exams', 'Examination reports: ECG, echocardiography, EEG, X-ray, CT, MRI, ultrasound (one row per report: time, item names and numeric measurements; narrative text not included).', [
    ('visit_id', ID, '', 'Research visit number on the report.'), ('exam_time', DT_, '', 'Examination time as recorded.'),
    ('modality', CAT, '', 'ECG, ECHO, EEG, XRAY, CT, MRI, ULTRASOUND.'), ('source_role', CAT, '', 'Export the row was taken from (supplementary rows replace the main rows of the admissions they cover).'),
    ('exam_name_cn', TXT_, '', 'Examination item names (报告名称; CT and MRI only), joined with " | " when one report covers several items.'),
    ('ecg_hr', FL, 'beats/min', 'ECG heart rate (supplement only).'), ('ecg_pr_ms', FL, 'ms', 'PR interval.'), ('ecg_qrs_ms', FL, 'ms', 'QRS duration.'),
    ('ecg_qt_ms', FL, 'ms', 'QT interval.'), ('ecg_qtc_ms', FL, 'ms', 'QTc as reported by the machine.'),
    ('echo_lvef_pct', FL, '%', 'Left ventricular ejection fraction (values outside 10-90 set to missing).'),
    ('n_source_rows', INT, '', 'Exported rows collapsed into this report (one per examination item, or repeated exports of the same report).'),
    ('time_precision', CAT, '', 'minute or date.'), ('visit_known', BOOL, '', 'Visit number is an inpatient admission in the export.'),
    ('exam_seq', INT, '', 'Report number within the visit (by time).')])
spec('support_events', 'Organ-support and related events from drug orders, non-drug orders and procedures (long format).', [
    ('visit_id', ID, '', 'Research admission number.'),
    ('support', CAT, '', 'vasoactive_infusion, pituitrin_infusion, imv_ett, imv_procedure, ventilator, niv, hfnc, oxygen, tracheostomy, rrt, rrt_chronic, plasma_exchange, hemoperfusion, ecmo, cpr, defibrillation_cardioversion, arterial_line, icp_monitoring, prone_positioning, extubation.'),
    ('event_time', DT_, '', 'Order start time or procedure time.'),
    ('end_time', DT_, '', 'Order end time (non-drug orders only; long-term orders end when stopped, at the latest at discharge). Missing for drug orders, which carry no duration in this export, and for procedures.'),
    ('time_precision', CAT, '', 'minute or date.'),
    ('source', CAT, '', 'drug_order, nondrug_order, procedure.'),
    ('source_seq', INT, '', 'order_seq (drug_orders, nondrug_orders) or proc_seq (procedures) of the source row; with visit_id and source it identifies the row.'),
    ('detail', TXT_, '', 'Agent, order name or procedure code and name.')])
SUPPORTS = ['vasoactive', 'imv', 'niv', 'hfnc', 'oxygen', 'tracheostomy', 'rrt', 'plasma_exchange', 'hemoperfusion', 'ecmo', 'cpr']
sum_cols = [
    ('visit_id', ID, '', 'Research admission number.'), ('patient_id', ID, '', 'Research patient number.'),
    ('in_supplement', BOOL, '', 'Covered by the supplementary export (variables from supplement-only sources are missing when False).'),
    ('admit_time', DT_, '', 'Hospital admission time.'), ('discharge_time', DT_, '', 'Hospital discharge time.'), ('admit_year', INT, '', 'Year of admission.'),
    ('age_days', FL, 'days', 'Age at admission.'), ('age_years', FL, 'years', 'Age at admission.'), ('sex', CAT, '', 'male / female.'),
    ('admission_source', CAT, '', 'See admissions.'), ('disposition', CAT, '', 'See admissions.'), ('outcome', CAT, '', 'See admissions.'),
    ('los_hours', FL, 'hours', 'Hospital length of stay.'),
    ('any_icu_stay', BOOL, '', 'Any ICU stay (all ICU units, incl. stays known only from the ward path text).'),
    ('any_picu_stay', BOOL, '', 'Any stay in picu_main or picu_west.'), ('n_icu_stays', INT, '', 'Number of ICU stays.'),
    ('first_icu_unit', CAT, '', 'Unit of the first ICU stay.'), ('first_icu_in_time', DT_, '', 'Entry time of the first ICU stay.'),
    ('first_icu_out_time', DT_, '', 'Exit time of the first ICU stay.'), ('first_icu_in_source', CAT, '', 'See icu_stays.'),
    ('first_icu_out_source', CAT, '', 'See icu_stays.'), ('first_icu_timing_complete', BOOL, '', 'Entry and exit known for the first stay.'),
    ('first_icu_los_hours', FL, 'hours', 'Length of the first ICU stay.'), ('first_icu_hours_admission_to_icu', FL, 'hours', 'Hospital admission to first ICU entry.'),
    ('first_icu_age_days_at_icu_in', FL, 'days', 'Age at first ICU entry.')]
for s in SUPPORTS:
    known = 'all admissions (drug orders)' if s == 'vasoactive' else 'supplement admissions only'
    sum_cols += [(f'{s}_ever', BOOL, '', f'Any {s} event during the admission; known for {known}.'),
                 (f'{s}_first_time', DT_, '', f'First {s} event (minute precision preferred; date-only time when no timed event exists).'),
                 (f'{s}_first_time_precision', CAT, '', 'minute or date.'), (f'{s}_first_source', CAT, '', 'drug_order, nondrug_order or procedure.')]
    if s == 'vasoactive':
        sum_cols += [('vasoactive_first_agent', CAT, '', 'Agent(s) of the first vasoactive-infusion order; agents ordered at the same minute are joined with "|" (alphabetical).'),
                     ('vasoactive_n_agents', INT, '', 'Number of distinct vasoactive agents infused during the admission.'),
                     ('pituitrin_infusion_ever', BOOL, '', 'Posterior pituitary injection by infusion during the admission (not part of the vasoactive definition).')]
    if s == 'imv':
        sum_cols.insert(len(sum_cols) - 4, ('imv_evidence', CAT, '', 'definite, probable, uncertain (not counted), none; missing for non-supplement admissions. See definitions.md.'))
sum_cols += [('rrt_chronic_ever', BOOL, '', 'Home (chronic) peritoneal dialysis order.'),
    ('death', BOOL, '', 'In-hospital death (disposition or outcome = death); known for supplement admissions with a recorded disposition or outcome.'),
    ('death_time', DT_, '', 'Discharge time of admissions ending in death (no separate death time exists).'),
    ('dama', BOOL, '', 'Discharge against medical advice (离院方式 = 非医嘱离院); supplement only.'),
    ('discharge_record_available', BOOL, '', 'A discharge record was found (DAMA admissions).'),
    ('dama_text_withdrawal', BOOL, '', 'Final discharge record documents refusal or withdrawal of treatment.'),
    ('dama_text_terminal_support', BOOL, '', 'Final discharge record documents discharge on or after removal of life support.'),
    ('dama_support_last_24h', BOOL, '', 'Organ support in the 24 h before discharge: a vasoactive-infusion order started in that window, or an invasive-ventilation order active in it (see definitions.md).'),
    ('dama_text_moribund', BOOL, '', 'Final record documents a moribund or critical condition (negated mentions ignored).'),
    ('dama_text_improved', BOOL, '', 'Final record documents improvement or stability.'),
    ('dama_text_not_improved', BOOL, '', 'Final record documents no improvement.'),
    ('dama_critical_category', CAT, '', 'A_withdrawal_terminal_or_support_at_discharge, B_moribund_without_improvement, C_uncertain, D_improved_or_stable (DAMA only).'),
    ('dama_critical', BOOL, '', 'DAMA in a critical condition by rule (categories A or B); False for other known dispositions. The adjudicated label is dama_critical_final.'),
    ('dama_critical_needs_review', BOOL, '', 'Category C: the rule cannot decide; the clinician decision is in dama_review_decision.'),
    ('death_or_dama_critical', BOOL, '', 'Death or DAMA in a critical condition by rule: True when either is True, False when both are known to be False, otherwise missing.'),
    ('dama_review_decision', CAT, '', 'Clinician review of the DAMA record: critical, not_critical or unsure; missing for admissions without DAMA or not reviewed (definitions.md, section 4).'),
    ('dama_critical_final', BOOL, '', 'DAMA in a critical condition, final label: the clinician decision where it is critical or not_critical, otherwise the rule (dama_critical); False for other known dispositions.'),
    ('dama_critical_final_source', CAT, '', 'clinician, rule (clinician unsure) or rule (not reviewed); DAMA admissions only.'),
    ('death_or_dama_critical_final', BOOL, '', 'Death or DAMA in a critical condition (final label): True when either is True, False when both are known to be False, otherwise missing.')]
spec('admission_summary', 'One row per inpatient admission with the derived variables (ICU, organ support, outcomes).', sum_cols)

RAWCOLS = ['quant_raw', 'display_raw', 'qual_raw', 'ref_low_raw', 'ref_high_raw']
L = L.drop(columns=[c for c in RAWCOLS if c in L])
LQ = LQ.drop(columns=[c for c in RAWCOLS + ['unit_raw'] if c in LQ])
OUTPUT = {
    'patients': P, 'admissions': V, 'outpatient_visits': VO,
    'transfers': TR[[c for c, *_ in SPEC['transfers']['cols']]],
    'icu_stays': IS, 'diagnoses_coded': DXC, 'diagnoses_listed': DXT, 'procedures': PR, 'drug_orders': D, 'nondrug_orders': N,
    'vital_signs': VT, 'vital_signs_cleaning_log': VLOG, 'labs': L, 'lab_qualitative': LQ, 'glucose_untimed': GL, 'exams': EXM,
    'support_events': SE, 'admission_summary': VSUM}
for name, df in OUTPUT.items():
    declared = [c for c, *_ in SPEC[name]['cols']]
    present = [c for c in df.columns if not c.startswith('_')]
    undeclared = [c for c in present if c not in declared]
    missing = [c for c in declared if c not in present]
    if undeclared or missing:
        raise RuntimeError(f'{name}: undeclared columns {undeclared}; declared but missing {missing}')
    OUTPUT[name] = df[declared]
log('All', len(OUTPUT), 'tables match their specifications.')
# Rows of admission-level tables whose visit number is not an inpatient admission of the main front page.
_adm_ids = set(OUTPUT['admissions'].visit_id)
_tcol = {'drug_orders': 'order_time', 'nondrug_orders': 'start_time', 'vital_signs': 'measure_time', 'vital_signs_cleaning_log': 'measure_time',
         'transfers': 'transfer_time', 'procedures': 'proc_time'}
ORPHANS = {}
for t in ['drug_orders', 'nondrug_orders', 'vital_signs', 'vital_signs_cleaning_log', 'procedures', 'transfers', 'icu_stays', 'support_events',
          'diagnoses_coded', 'diagnoses_listed', 'glucose_untimed']:
    d = OUTPUT[t]
    if t == 'diagnoses_listed':
        d = d[d.source.ne('outpatient_record_preliminary_dx')]
    o = d[~d.visit_id.isin(_adm_ids)]
    if len(o):
        yrs = ''
        if t in _tcol and o[_tcol[t]].notna().any():
            y0, y1 = int(o[_tcol[t]].dt.year.min()), int(o[_tcol[t]].dt.year.max())
            yrs = f', {y0}' if y0 == y1 else f', {y0}-{y1}'
        nv = o.visit_id.nunique()
        ORPHANS[t] = f"{len(o)} row{'s' if len(o) != 1 else ''}, {nv} visit{'s' if nv != 1 else ''}{yrs}"
qc('15 Final checks', 'Rows whose visit number is not an inpatient admission of the main front page (kept; no admission-level variables)',
   ORPHANS or 0, 'info', 'They drop out of joins with admissions. labs, lab_qualitative and exams carry their own visit linkage.')

In [ ]:
# Write the tables (UTF-8 CSV, deterministic row order), the data dictionary, the review sheet and the documents.
def fmt(df, name):
    out = df.copy()
    for c, typ, *_ in SPEC[name]['cols']:
        s = out[c]
        if typ == DT_:
            s = pd.to_datetime(s, errors='coerce')
            out[c] = s.dt.strftime('%Y-%m-%d %H:%M:%S').where(s.notna(), '')
        elif typ == BOOL:
            out[c] = s.map(lambda x: '' if is_missing(x) or x is pd.NA else ('True' if bool(x) else 'False'))
        elif typ == INT:
            out[c] = pd.to_numeric(s, errors='coerce').map(lambda x: '' if pd.isna(x) else str(int(x)))
        elif typ == FL:
            out[c] = pd.to_numeric(s, errors='coerce').map(lambda x: '' if pd.isna(x) else repr(float(x)))
        else:
            out[c] = s.map(lambda x: '' if is_missing(x) else str(x))
    return out
MANIFEST = {'dataset': 'Acute encephalopathy export, Wuhan Children\'s Hospital research platform', 'freeze_version': FREEZE_VERSION,
            'built': time.strftime('%Y-%m-%dT%H:%M:%S'), 'software': {'python': platform.python_version(), 'pandas': pd.__version__,
            'numpy': np.__version__, 'excel_engine': ENGINE}, 'sources': [{k: w[k] for k in ('file', 'role', 'bytes', 'md5')} for w in WORKBOOKS + ADJ_INPUTS],
            'tables': {}, 'documents': {}}
for name, df in OUTPUT.items():
    path = TAB / f'{name}.csv'
    fmt(df, name).to_csv(path, index=False, encoding='utf-8-sig', lineterminator='\n')
    MANIFEST['tables'][name] = {'file': f'tables/{name}.csv', 'rows': int(len(df)), 'columns': int(df.shape[1]), 'md5': md5_file(path)}
    log(f'  {name}: {len(df)} rows')
dd = pd.DataFrame([{'table': t, 'table_description': SPEC[t]['desc'], 'column': c, 'type': typ, 'unit': u, 'description': d}
                   for t in OUTPUT for c, typ, u, d in SPEC[t]['cols']])
dd.to_csv(DOC / 'data_dictionary.csv', index=False, encoding='utf-8-sig', lineterminator='\n')
# Source coverage: every sheet of both exports, with the number of rows and whether the build reads it.
NOT_USED = {'病案入院诊断': 'Admission-condition codes (入院病情) exported without the diagnoses they refer to; they cannot be linked to a diagnosis.'}
SRC = pd.DataFrame([{'export': role, 'sheet': sh, 'rows': int(sum(len(p) for p in parts)), 'used': (role, sh) in SHEETS_READ}
                    for role in ('main', 'supplement') for sh, parts in RAW[role].items()])
SRC['note'] = [('' if u else NOT_USED.get(s, 'not used')) for s, u in zip(SRC.sheet, SRC.used)]
qc('0 Sources', 'Sheets read by the build: main / supplementary export',
   ' / '.join(f"{int(g.used.sum())} of {len(g)}" for _, g in SRC.groupby('export', sort=False)), 'info', 'Every sheet is listed in data_dictionary.xlsx (sheet source_sheets).')
for r in SRC[~SRC.used].itertuples():
    qc('0 Sources', f'Sheet not used: {r.export} {r.sheet} ({r.rows} rows)', r.note, 'info' if r.sheet in NOT_USED else 'warn')
lab_dict = L.groupby(['analyte', 'unit', 'panel']).agg(analyte_cn=('analyte_cn', lambda s: ' / '.join(sorted(set(s)))), n_results=('value', 'size'),
                                                     n_numeric=('value', lambda s: int(s.notna().sum())), median=('value', 'median')).reset_index()
lab_dict['median'] = lab_dict['median'].round(4)
qual_dict = LQ.groupby(['panel', 'test']).agg(test_cn=('analyte_cn', lambda s: ' / '.join(sorted(set(map(str, s))))[:200]), n=('result', 'size')).reset_index()
with pd.ExcelWriter(DOC / 'data_dictionary.xlsx', engine='openpyxl') as xw:
    pd.DataFrame([{'table': t, 'rows': MANIFEST['tables'][t]['rows'], 'description': SPEC[t]['desc']} for t in OUTPUT]).to_excel(xw, sheet_name='tables', index=False)
    dd.to_excel(xw, sheet_name='columns', index=False)
    SRC.to_excel(xw, sheet_name='source_sheets', index=False)
    lab_dict.to_excel(xw, sheet_name='lab_analytes', index=False)
    qual_dict.to_excel(xw, sheet_name='qualitative_tests', index=False)
    pd.DataFrame({'route_raw': list(route_map), 'route_class': list(route_map.values())}).sort_values('route_class').to_excel(xw, sheet_name='drug_routes', index=False)
    pd.DataFrame(sorted(nd_map.items(), key=lambda kv: (kv[1], kv[0])), columns=['order_name_cn', 'category']).query('category != "other"').to_excel(xw, sheet_name='nondrug_categories', index=False)
    pd.DataFrame([{'drug_name_cn': k, 'vasoactive_agent': v} for k, v in agent_map.items() if v]).sort_values('vasoactive_agent').to_excel(xw, sheet_name='vasoactive_names', index=False)
    pd.DataFrame([{'drug_name_cn': k, 'drug_classes': v} for k, v in cls_map.items() if v]).sort_values('drug_classes').to_excel(xw, sheet_name='drug_class_names', index=False)
with pd.ExcelWriter(ADJ / 'dama_review_INTERNAL.xlsx', engine='openpyxl') as xw:
    REV.to_excel(xw, sheet_name='dama_review', index=False)
    ws = xw.sheets['dama_review']
    for col, width in zip('ABCDEFGHIJKLMNOPQR', [34, 12, 10, 18, 18, 8, 10, 10, 34, 10, 10, 10, 10, 10, 10, 90, 22, 30]):
        ws.column_dimensions[col].width = width
    ws.freeze_panes = 'C2'
log('Data dictionary and review sheet written.')

In [ ]:
def md_table(df):
    df = df.astype(object).where(df.notna(), '')
    esc = lambda x: str(x).replace('|', '/').replace('\n', ' ')
    lines = ['| ' + ' | '.join(esc(c) for c in df.columns) + ' |', '|' + '---|' * df.shape[1]]
    lines += ['| ' + ' | '.join(esc(x) for x in r) + ' |' for r in df.itertuples(index=False)]
    return '\n'.join(lines)

# Final checks on the written tables: no narrative free text, no masked names, no undeclared columns.
long_text = {}
for name, df in OUTPUT.items():
    for c, typ, *_ in SPEC[name]['cols']:
        if typ == TXT_:
            s = df[c].dropna().astype(str)
            if len(s):
                long_text[f'{name}.{c}'] = int(s.str.len().max())
too_long = {k: v for k, v in long_text.items() if v > 300}
qc('15 Final checks', 'Timestamps outside 1990-2027 set to missing (data-entry errors)', int(INVALID_DATES['out_of_range']), 'info')
qc('15 Final checks', 'Longest value in each text column (characters)', dict(sorted(long_text.items(), key=lambda kv: -kv[1])[:8]), 'info')
qc('15 Final checks', 'Text columns with values over 300 characters (narrative text)', too_long or 0, 'pass' if not too_long else 'fail')
masked = {}
for name, df in OUTPUT.items():
    for c, typ, *_ in SPEC[name]['cols']:
        if typ == TXT_:
            n = int(df[c].dropna().astype(str).str.contains(r'患儿[一-龥]{1,2}\*|[一-龥]\*\*').sum())
            if n:
                masked[f'{name}.{c}'] = n
qc('15 Final checks', 'Text values containing masked personal names (e.g. 傅**)', masked or 0, 'pass' if not masked else 'fail')
if too_long or masked:
    raise RuntimeError(f'Free-text or name check failed: {too_long} {masked}')

In [ ]:
# Reconciliation with freeze v1 (runs when the v1 folder `cleaned_data/` is present next to this notebook).
V1 = HERE / 'cleaned_data'
rec_md = ['# Reconciliation: freeze v1 versus v1.1', '']
def v1csv(name, **kw):
    return pd.read_csv(V1 / name, dtype=str, keep_default_na=False, na_values=[''], encoding='utf-8-sig', **kw)
def counts_text(s):
    s = pd.Series(s)
    return '; '.join(f'{k}: {v}' for k, v in s.value_counts().items()) if len(s) else 'none'
if (V1 / 'visits_INTERNAL.csv').exists() and (V1 / 'outcomes_derived_INTERNAL.csv').exists():
    same_src = None
    if (V1 / 'freeze_manifest_v1.json').exists():
        m1 = json.loads((V1 / 'freeze_manifest_v1.json').read_bytes().decode('utf-8', errors='replace'))
        same_src = sorted(v.get('md5', '') for v in m1.get('source_files', {}).values()) == sorted(w['md5'] for w in WORKBOOKS)
    rec_md += [('Freeze v1 (`cleaned_data/`) was built from the same two exports (identical MD5 checksums). ' if same_src else
                'The source checksums of freeze v1 could not be matched to these exports. ') +
               'Every number below is computed by the build notebook from both versions.', '']
    v1v, v1o = v1csv('visits_INTERNAL.csv'), v1csv('outcomes_derived_INTERNAL.csv')
    v1v['visit_id'], v1o['visit_id'] = norm_id(v1v.visit_id), norm_id(v1o.visit_id)
    B = lambda s: s.astype(str).str.lower().isin(['true', '1', '1.0'])
    X = VSUM.set_index('visit_id')
    o1 = v1o.set_index('visit_id').reindex(X.index); w1 = v1v.set_index('visit_id').reindex(X.index)
    rows = []
    def as_bool(s):
        s = s.reindex(X.index)
        return s.astype(object).where(s.notna(), False).astype(bool)
    def add(item, a, b, note=''):
        a, b = as_bool(a), as_bool(b)
        rows.append({'item (admissions)': item, 'v1': int(a.sum()), 'v1.1': int(b.sum()), 'both': int((a & b).sum()),
                     'v1 only': int((a & ~b).sum()), 'v1.1 only': int((~a & b).sum()), 'explanation': note})
        return a, b
    # PICU stays
    PS = IS[IS.unit.isin(['picu_main', 'picu_west'])]
    def picu_reason(vid):
        g = PS[PS.visit_id == vid]
        if g.source.eq('transfer_path_text_only').all():
            return 'PICU only in the ward path text (no transfer record; times unknown)'
        if g.unit.eq('picu_west').any():
            return 'West-campus PICU ward name (half-width brackets) not matched by v1'
        if g.in_source.eq('unknown').any():
            return 'discharged from the PICU without a recorded transfer into it (entry time unknown)'
        return 'other'
    p1, p11 = add('PICU stay', B(w1.ever_picu), X.any_picu_stay)
    rows[-1]['explanation'] = 'v1.1 only: ' + counts_text([picu_reason(v) for v in X.index[p11 & ~p1]]) + '.'
    both_p = p1 & p11
    first_p = PS.sort_values('stay_seq').groupby('visit_id').first().reindex(X.index)
    first_main = PS[PS.unit.eq('picu_main')].sort_values('stay_seq').groupby('visit_id').first().reindex(X.index)
    v1_in = pd.to_datetime(w1.picu_admit_time, errors='coerce'); v1_out = pd.to_datetime(w1.picu_discharge_time, errors='coerce')
    west_first = both_p & first_p.unit.eq('picu_west') & (v1_in == first_main.in_time)
    same_in = both_p & (v1_in == first_p.in_time)
    rows.append({'item (admissions)': 'Entry time of the first PICU stay (PICU stay in both)', 'v1': int(both_p.sum()), 'v1.1': int(both_p.sum()),
                 'both': int(same_in.sum()), 'v1 only': '', 'v1.1 only': '',
                 'explanation': f'Identical: {int(same_in.sum())}. v1.1 finds an earlier West-campus PICU stay before the main PICU stay: {int(west_first.sum())}.'
                                + (f' Other differences: {int((both_p & ~same_in & ~west_first).sum())}.' if (both_p & ~same_in & ~west_first).any() else '')})
    same_out = both_p & ((v1_out == first_p.out_time) | (v1_out.isna() & first_p.out_time.isna()))
    still_in = same_out & v1_out.isna()
    unk = both_p & first_p.out_time.isna() & v1_out.notna()
    other_out = both_p & ~same_out & ~unk & ~west_first
    rows.append({'item (admissions)': 'Exit time of the first PICU stay (PICU stay in both)', 'v1': int(both_p.sum()), 'v1.1': int(both_p.sum()),
                 'both': int(same_out.sum()), 'v1 only': '', 'v1.1 only': '',
                 'explanation': f'Identical: {int(same_out.sum())} (incl. {int(still_in.sum())} still in hospital at export: exit missing in both). '
                                f'v1 set the exit to hospital discharge although no transfer out of the PICU was recorded and the child was discharged '
                                f'from another ward; v1.1 leaves it unknown (upper bound = discharge): {int(unk.sum())}. First PICU stay is the West-campus '
                                f'stay in v1.1: {int((both_p & west_first & ~same_out).sum())}.' + (f' Other: {int(other_out.sum())}.' if other_out.any() else '')})
    # Outcomes
    add('In-hospital death', B(o1.in_hospital_death), X.death,
        'Same definition (disposition or outcome = death). Unknown for admissions outside the supplementary export in v1.1 (v1 coded them as no).')
    c1, c11 = add('Critical discharge against medical advice', B(o1.critical_discharge), X.dama_critical)
    cat = X.dama_critical_category
    comp1 = o1[['c2_tail_critical', 'c3_support_at_dc']].apply(B)[c1 & ~c11].sum()
    rows[-1]['explanation'] = (
        'v1.1 rule (definitions.md): A = withdrawal or terminal-support wording, or organ support in the 24 h before discharge (a vasoactive-infusion order '
        'started, or an invasive-ventilation order running, in that window); B = moribund wording without improvement wording; C = conflicting or no '
        'evidence (not counted; listed first for clinician review); D = improvement or stability wording. '
        f'v1 only, by v1.1 category: {counts_text(cat[c1 & ~c11])} (v1 reasons: critical wording at the end of the record {int(comp1.c2_tail_critical)}, '
        f'support at discharge {int(comp1.c3_support_at_dc)}; v1 counted any vasoactive route and any ventilation including NIV). '
        f'v1.1 only, by category: {counts_text(cat[~c1 & c11])}.' + ('' if FREEZE_VERSION == 'v1.1' else
        ' This row compares the rule; the clinician-adjudicated label (dama_critical_final) is compared with the rule in reconciliation_@@PREV@@.md.'))
    # Vasoactive infusion
    a, b = add('Vasoactive infusion', B(o1.vaso_ever), X.vasoactive_ever)
    fl1 = D[D.visit_id.isin(set(X.index[a & ~b])) & D.vasoactive_agent.notna()]
    def vreason(g):
        if not g.vasoactive_agent.isin(VASOACTIVE_AGENTS).any():
            return 'only agents outside the definition (' + ', '.join(sorted(set(g.vasoactive_agent))) + ')'
        g = g[g.vasoactive_agent.isin(VASOACTIVE_AGENTS)]
        if not g.route_class.isin(['iv_infusion', 'unspecified']).any():
            return 'non-infusion routes only (IV push, endotracheal, inhaled, topical, enteral, IM)'
        return 'route not specified and no diluent line at the same minute'
    vr = fl1.groupby('visit_id').apply(vreason) if len(fl1) else pd.Series(dtype=object)
    no_order = int((a & ~b).sum()) - len(vr)
    rows[-1]['explanation'] = ('v1 counted any catecholamine order (incl. isoproterenol) by any route and did not count milrinone or vasopressin. '
                               f'v1 only: {counts_text(vr)}' + (f'; no matching order found: {no_order}' if no_order else '') +
                               f'. v1.1 only: {int((~a & b).sum())} (first agent: {counts_text(X.loc[~a & b, "vasoactive_first_agent"])}).')
    # Invasive mechanical ventilation
    a, b = add('Invasive mechanical ventilation (v1: any ventilation procedure or order)', B(o1.vent_op_ever) | o1.vent_order_first_time.notna(), X.imv_ever)
    r11 = ['IMV procedure code without a procedure time (not counted by v1)' if pd.isna(X.loc[v, 'imv_first_time']) else 'other'
           for v in X.index[~a & b]]
    rows[-1]['explanation'] = ('v1 included non-invasive ventilation, nasal prongs, extubation and pre-operative anesthesia orders and NIV procedure codes; '
                               'v1.1 counts intubation/ETT orders, codes 96.04 and 96.70-96.72, and ventilator orders without NIV evidence. '
                               f'v1 only, by v1.1 IMV evidence: {counts_text(X.loc[a & ~b, "imv_evidence"].fillna("missing"))}. v1.1 only: {counts_text(r11)}.')
    REC = pd.DataFrame(rows)

    # Row counts of corresponding tables, with the source of every difference.
    T = lambda t: MANIFEST['tables'][t]['rows']
    def nrows1(f):
        return len(v1csv(f, usecols=[0])) if (V1 / f).exists() else None
    cnt = []
    def addc(t1, t11, note=''):
        cnt.append({'v1 table': t1, 'v1 rows': nrows1(t1), 'v1.1 table': t11, 'v1.1 rows': T(t11), 'note': note})
    addc('visits_INTERNAL.csv', 'admissions')
    addc('visits_outpatient_INTERNAL.csv', 'outpatient_visits')
    addc('patients_INTERNAL.csv', 'patients', 'v1.1 does not keep native place (籍贯).')
    addc('transfers_INTERNAL.csv', 'transfers')
    addc('operations_INTERNAL.csv', 'procedures', f'{int((PR.n_source_rows - 1).sum())} fully identical rows collapsed (n_source_rows).')
    addc('diagnoses_discharge_INTERNAL.csv', 'diagnoses_coded')
    d1 = v1csv('orders_drug_INTERNAL.csv', usecols=['source_file']).source_file.value_counts()
    k3 = len(D.drop_duplicates(['visit_id', 'start_time', 'drug_name_cn']))
    addc('orders_drug_INTERNAL.csv', 'drug_orders',
         f"v1 = {int(d1.get('main', 0)):,} main-export rows + {int(d1.get('supp', 0)):,} supplementary copies. "
         + (f"The v1 main-export rows equal the number of distinct (admission, minute, drug name) combinations ({k3:,}): v1 merged orders of the same "
            f"drug at the same minute that differed in route or end time. " if int(d1.get('main', 0)) == k3 else '')
         + f"v1.1 keeps one row per distinct order ({int(D.n_source_rows.sum()):,} main-export rows, identical rows collapsed); the supplementary rows are verified copies.")
    k3n = len(N.drop_duplicates(['visit_id', 'start_time', 'order_name_cn']))
    addc('orders_nondrug_INTERNAL.csv', 'nondrug_orders',
         (f'The v1 rows equal the number of distinct (admission, start time, name) combinations ({k3n:,}): v1 merged orders that differed only in end time. '
          if nrows1('orders_nondrug_INTERNAL.csv') == k3n else '') + f'v1.1 collapses only identical rows ({int(N.n_source_rows.sum()):,} rows before collapsing).')
    vv = v1csv('vitals_charted_INTERNAL.csv', usecols=['visit_id', 'source_file'])
    vv_supp = norm_id(vv.visit_id).isin(set(V.visit_id[V.in_supplement]))
    vmain = vv.source_file.eq('main')
    addc('vitals_charted_INTERNAL.csv', 'vital_signs',
         f'v1 = {int((vmain & ~vv_supp).sum()):,} main-export rows of admissions outside the supplement + {int((vmain & vv_supp).sum()):,} main-export rows '
         f'and {int((~vmain).sum()):,} supplementary rows of supplement admissions (two copies of the same measurements). v1.1 keeps one copy (the supplementary '
         f'rows, verified identical on the shared columns) and collapses {int((VT.n_source_rows - 1).sum())} fully identical rows.')
    m1 = v1csv('measurements_long_INTERNAL.csv', usecols=['report_id', 'analyte', 'value', 'qual_result'])
    mb1 = v1csv('microbiology_long_INTERNAL.csv', usecols=['report_id', 'pathogen'])
    key1 = norm_id(m1.report_id).fillna('') + '|' + m1.analyte.fillna('')
    txt1 = m1.value.isna() & m1.qual_result.notna()
    K_num, K_txt = set(key1[~txt1]), set(key1[txt1])
    K_mb = set(norm_id(mb1.report_id).fillna('') + '|' + mb1.pathogen.fillna(''))
    lkey = L.report_id.fillna('') + '|' + L.analyte_cn.fillna('')
    qkey = LQ.report_id.fillna('') + '|' + LQ.analyte_cn.fillna('')
    K_L, K_Q = set(lkey), set(qkey)
    only_m_qual = (K_txt - K_mb) & K_Q
    new_q = K_Q - K_mb - K_txt
    addc('measurements_long_INTERNAL.csv', 'labs',
         f'v1 = {int((~txt1).sum()):,} numeric rows (incl. {int(key1[~txt1].duplicated().sum())} repeated report-analyte pairs) + {int(txt1.sum()):,} rows with a text result and no number ({len(K_txt & K_mb):,} duplicating '
         f'microbiology_long rows, {len(only_m_qual):,} qualitative items, {len(K_txt & K_L):,} results without a number). v1.1 labs = '
         f'{len(K_num & K_L):,} of the v1 numeric results + the {len(K_txt & K_L):,} results without a number (value missing, flagged); '
         f'{len(K_num - K_L):,} v1 numeric rows are qualitative tests reporting a number and are in lab_qualitative. Other v1 numeric rows not in v1.1: '
         f'{len(K_num - K_L - K_Q):,}.')
    addc('microbiology_long_INTERNAL.csv', 'lab_qualitative',
         f'v1.1 = {len(K_mb & K_Q):,} report-test pairs of v1 microbiology_long + {len(only_m_qual):,} qualitative items that v1 kept only in '
         f'measurements_long ({counts_text(LQ.panel[qkey.isin(only_m_qual)])}) + {len(new_q):,} pairs not in v1 ({counts_text(LQ.panel[qkey.isin(new_q)])}); '
         f'{len(LQ) - len(K_Q):,} more rows than pairs because the multiplex sequencing panel has one row per organism. v1 microbiology pairs not in v1.1: {len(K_mb - K_Q):,}.')
    g1 = v1csv('glucose_INTERNAL.csv', usecols=['visit_id', 'value', 'source_file'])
    g1m = ~g1.source_file.fillna('').str.contains('补充')
    nd_gl = len(GL.assign(_v=GL.glucose_mmol_L.round(3)).drop_duplicates(['visit_id', '_v']))
    addc('glucose_INTERNAL.csv', 'glucose_untimed',
         f'v1 = {int(g1m.sum()):,} main-export rows (equal values within an admission merged; distinct values in v1.1: {nd_gl:,}) + {int((~g1m).sum()):,} '
         f'supplementary copies. v1.1 keeps every main-export row: the sheet has no time, so equal values are separate measurements; the supplementary '
         f'rows are verified copies.')
    ex1 = {f: nrows1(f) for f in ['ecg_INTERNAL.csv', 'eeg_INTERNAL.csv', 'imaging_structured_INTERNAL.csv']}
    raw_s = {'ecg_INTERNAL.csv': len(sheet('supplement', '心电图报告')), 'eeg_INTERNAL.csv': len(sheet('supplement', '脑电图检查')),
             'imaging_structured_INTERNAL.csv': sum(len(sheet('supplement', x)) for x in ['超声报告', 'X线报告', 'MRI报告', 'CT报告', '儿童心脏彩超'])}
    every_row = all(ex1[f] == raw_s[f] for f in ex1)
    cnt.append({'v1 table': 'ecg, eeg, imaging_structured (incl. echo)', 'v1 rows': sum(v or 0 for v in ex1.values()), 'v1.1 table': 'exams', 'v1.1 rows': T('exams'),
                'note': ('v1: supplement admissions only, ' + ('every exported row of the supplementary export' if every_row else 'rows of the supplementary export')
                         + ', with report text and text-derived flags (' + ', '.join(f"{k.replace('_INTERNAL.csv', '')} {v:,}" for k, v in ex1.items())
                         + '). v1.1: all admissions, one row per report (repeated rows collapsed), no report text.')})
    CNT = pd.DataFrame(cnt)
    gone = []
    for f, what in [('vitals_admission_INTERNAL.csv', 'admission physical examination read from the admission note (vital signs, consciousness, pupils, muscle tone, meningeal signs)'),
                    ('onset_history_INTERNAL.csv', 'symptom onset read from the history text'),
                    ('diagnoses_admit_raw_INTERNAL.csv', 'admission-condition codes (入院病情) without the diagnoses they refer to'),
                    ('ecg_INTERNAL.csv', 'ECG text flags (tachycardia, bradycardia, arrhythmia, block, ST-T, low voltage); the intervals are in exams'),
                    ('echo_INTERNAL.csv', 'echocardiography text flags and findings text; LVEF is in exams'),
                    ('eeg_INTERNAL.csv', 'EEG text flags (background, epileptiform activity, status)'),
                    ('imaging_structured_INTERNAL.csv', 'imaging conclusions (text) and lesion flags read from them')]:
        if (V1 / f).exists():
            gone.append({'v1 table': f, 'v1 rows': nrows1(f), 'content not in v1.1': what})
    GONE = pd.DataFrame(gone)
    rec_md += ['## Admission-level comparison of key derived variables', '', f'All {len(X):,} admissions.', '', md_table(REC), '',
               '## Row counts of corresponding tables', '', md_table(CNT), '',
               '## v1 content not carried into v1.1', '',
               'These variables were read from narrative text (admission notes, history, reports) by rules that depend on the research question and '
               'need validation against chart review. v1.1 holds source data and generic derivations only; text-derived variables can be built as a '
               'separate, versioned feature layer from the raw exports, whose checksums are in manifest.json. Report text itself is not part of the '
               'frozen tables (v1 echo and imaging tables contained findings and conclusions).', '', md_table(GONE) if len(GONE) else '(none)', '']
    _cur = lambda s: s.replace('v1.1', FREEZE_VERSION).replace('@@PREV@@', 'v1.1')   # labels of this build; v1.1 is the previous release
    (REC.rename(columns=_cur).replace(r'v1\.1', FREEZE_VERSION, regex=True).replace('@@PREV@@', 'v1.1', regex=True)
        .to_csv(DOC / 'reconciliation_v1.csv', index=False, encoding='utf-8-sig', lineterminator='\n'))
    qc('16 Reconciliation', 'Compared with freeze v1', 'see docs/reconciliation_v1.md', 'info')
else:
    rec_md += ['Freeze v1 (`cleaned_data/`) was not found next to the notebook; no comparison was made.']
(DOC / 'reconciliation_v1.md').write_text('\n'.join(rec_md).replace('v1.1', FREEZE_VERSION).replace('@@PREV@@', 'v1.1'), encoding='utf-8')

In [ ]:
# Reconciliation with the previous release (runs when its folder is next to this notebook): v1.2 may change only the adjudicated
# outcome labels, so every other table and every previous column of admission_summary must be identical.
PREV_VERSION = 'v1.1'
PREV = HERE / f'AE_freeze_{PREV_VERSION}'
REC2_OK = None
rec2 = [f'# Reconciliation: freeze {PREV_VERSION} versus {FREEZE_VERSION}', '']
if (PREV / 'manifest.json').exists():
    PM = json.loads((PREV / 'manifest.json').read_text(encoding='utf-8'))
    same_raw = sorted(s['md5'] for s in PM.get('sources', []) if s.get('role') in ('main', 'supplement')) == sorted(w['md5'] for w in WORKBOOKS)
    rec2 += [f'Raw exports identical to those of {PREV_VERSION} (MD5): {same_raw}. '
             + (f"New input: the clinician adjudication workbook `{ADJ_INPUTS[0]['file']}` (MD5 {ADJ_INPUTS[0]['md5']})." if ADJ_INPUTS
                else 'No adjudication workbook was supplied.'), '', 'Every number below is computed by the build notebook from both versions.', '']
    trows = []
    for t, info in MANIFEST['tables'].items():
        pt = PM['tables'].get(t)
        trows.append({'table': t, f'rows {PREV_VERSION}': pt['rows'] if pt else '', f'rows {FREEZE_VERSION}': info['rows'],
                      f'columns {PREV_VERSION}': pt['columns'] if pt else '', f'columns {FREEZE_VERSION}': info['columns'],
                      'identical file (MD5)': bool(pt) and pt['md5'] == info['md5']})
    TREC = pd.DataFrame(trows)
    gone_t = [t for t in PM['tables'] if t not in MANIFEST['tables']]
    a_old = pd.read_csv(PREV / 'tables' / 'admission_summary.csv', dtype=str, keep_default_na=False, encoding='utf-8-sig')
    a_new = pd.read_csv(TAB / 'admission_summary.csv', dtype=str, keep_default_na=False, encoding='utf-8-sig')
    shared = [c for c in a_old.columns if c in a_new.columns]
    same_rows = a_old.visit_id.tolist() == a_new.visit_id.tolist()
    diff_cols = [c for c in shared if not a_old[c].equals(a_new[c])] if same_rows else shared
    new_cols = [c for c in a_new.columns if c not in a_old.columns]
    gone_cols = [c for c in a_old.columns if c not in a_new.columns]
    other_diff = [r['table'] for r in trows if not r['identical file (MD5)'] and r['table'] != 'admission_summary']
    REC2_OK = same_raw and same_rows and not diff_cols and not gone_cols and not other_diff and not gone_t
    rec2 += ['## Tables', '', md_table(TREC), '',
             f'`admission_summary`: same admissions in the same order: {same_rows}. Of its {len(shared)} {PREV_VERSION} columns, '
             f'{len(shared) - len(diff_cols)} are identical value for value' + (f'; these differ: {", ".join(diff_cols)}' if diff_cols else '')
             + (f'; removed: {", ".join(gone_cols)}' if gone_cols else '') + f'. New columns: {", ".join(new_cols) or "none"}.', '',
             ('Every other table is byte-identical to ' + PREV_VERSION + '.' if not other_diff and not gone_t else
              f'Other tables that differ: {other_diff or "none"}; tables of {PREV_VERSION} missing here: {gone_t or "none"}.'), '']
    XD = VSUM[VSUM.dama.eq(True)]
    ct1 = pd.crosstab(XD.dama_critical_category, XD.dama_review_decision.fillna('not reviewed'), margins=True, margins_name='total')
    ct1.index.name = 'rule category'
    lab = lambda s: s.map({True: 'critical', False: 'not critical'}).fillna('unknown')
    ct2 = pd.crosstab(lab(XD.dama_critical).rename('rule label (dama_critical)'), lab(XD.dama_critical_final).rename('final label'), margins=True, margins_name='total')
    d_old, d_new = VSUM.death_or_dama_critical, VSUM.death_or_dama_critical_final
    rec2 += ['## The adjudicated label', '',
             f'{len(XD)} admissions ended in discharge against medical advice. A clinician recorded critical, not_critical or unsure for '
             f'{int(XD.dama_review_decision.notna().sum())} of them. Rule category by decision:', '',
             md_table(ct1.reset_index()), '',
             'Rule label versus final label (the decision where critical or not_critical; the rule where unsure or not reviewed):', '',
             md_table(ct2.reset_index()), '',
             f'Death or critical DAMA: {int(d_old.eq(True).sum())} admissions by rule, {int(d_new.eq(True).sum())} with the final label '
             f'({int((d_old.eq(True) & d_new.eq(False)).sum())} change from yes to no, {int((d_old.eq(False) & d_new.eq(True)).sum())} from no to yes).', '',
             '## Documents', '',
             f'definitions.md (section 4, clinician adjudication), the data dictionary (new columns; descriptions of dama_critical, '
             f'dama_critical_needs_review and death_or_dama_critical), qc_report.md (sections 0, 13 and 16), README.md, the review sheet (decisions '
             f'and notes prefilled), and the build notebook ({NB_NAME}).', '']
    qc('16 Reconciliation', f'Compared with the released freeze {PREV_VERSION}: raw exports, other tables and previous admission_summary columns identical',
       f'raw {same_raw}; other tables {not other_diff and not gone_t}; admission_summary rows {same_rows}, previous columns {not diff_cols and not gone_cols}',
       'pass' if REC2_OK else 'fail', f'see docs/reconciliation_{PREV_VERSION}.md')
else:
    rec2 += [f'The released freeze {PREV_VERSION} was not found next to the notebook; no comparison was made.']
    qc('16 Reconciliation', f'Compared with the released freeze {PREV_VERSION}', 'previous release not found', 'warn')
(DOC / f'reconciliation_{PREV_VERSION}.md').write_text('\n'.join(rec2), encoding='utf-8')

In [ ]:
# QC report, definitions, README, loader, notebook copy and manifest.
qc_df = pd.DataFrame(QC)
qc_md = [f'# QC report, frozen dataset {FREEZE_VERSION}', '', f"Built {MANIFEST['built']} with pandas {pd.__version__} ({ENGINE}). "
         'Every number below is computed by the build notebook from the raw exports.', '',
         f"Checks: {int(qc_df.status.eq('pass').sum())} pass, {int(qc_df.status.eq('warn').sum())} warn, "
         f"{int(qc_df.status.eq('fail').sum())} fail, {int(qc_df.status.eq('info').sum())} informational.", '']
for sec, g in qc_df.groupby('section', sort=False):
    md = g[['check', 'value', 'status', 'note']].copy()
    md['value'] = md.value.map(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (dict, list)) else str(x))
    qc_md += [f'## {sec}', '', md_table(md), '']
(DOC / 'qc_report.md').write_text('\n'.join(qc_md), encoding='utf-8')
if qc_df.status.eq('fail').any():
    raise RuntimeError('QC failures: ' + '; '.join(qc_df[qc_df.status.eq('fail')].check))

n = lambda t: MANIFEST['tables'][t]['rows']
X = VSUM
_ev = [tuple(int(x) for x in v.split(' / ')) for v in EXAM_VALID.values()]
EXAM_UNDER = max([(b_ - a_) / b_ * 100 for a_, b_ in _ev if b_] + [0.0])
_ndec = {k: int(pd.Series(ADJ_DEC).eq(k).sum()) for k in ADJ_VALID} if ADJ_DEC else {}
ADJ_DEFS = ((f"A pediatric clinician of the study team reviewed every DAMA admission in that sheet, seeing the rule category, the flags and "
             f"the excerpts, and recorded critical, not_critical or unsure (`dama_review_decision`: critical {_ndec['critical']}, not_critical "
             f"{_ndec['not_critical']}, unsure {_ndec['unsure']}; not reviewed {_unrev}). `dama_critical_final` takes the decision where it is critical "
             f"or not_critical and the rule label otherwise (`dama_critical_final_source`); `death_or_dama_critical_final` combines it with death. "
             f"Critical by the final label: {int(X.dama_critical_final.eq(True).sum())} admissions (by rule: {int(X.dama_critical.eq(True).sum())}). "
             f"The rule columns are unchanged. The adjudication workbook is an input of the build (file name and MD5 in manifest.json); reviewer notes "
             f"and excerpts stay in adjudication_INTERNAL, whose review sheet is prefilled with the decisions.")
            if ADJ_DEC else "No adjudication workbook was supplied: `dama_critical_final` equals the rule label.")
import textwrap
ADJ_BULLET = textwrap.fill('* **Clinician adjudication** (from v1.2). ' + ADJ_DEFS, width=150, subsequent_indent='  ', break_on_hyphens=False)
DEFS = f"""# Definitions, frozen dataset {FREEZE_VERSION}

All rules are implemented in `code/{NB_NAME}`; the name-to-category mappings actually applied are listed in
`docs/data_dictionary.xlsx` (sheets drug_routes, vasoactive_names, drug_class_names, nondrug_categories, lab_analytes, qualitative_tests), and every
sheet of the two exports is listed there with its use (sheet source_sheets).

## 1. Sources and coverage
* **Main export** (all {n('admissions'):,} inpatient admissions of {n('patients'):,} patients, {int(X.admit_year.min())}-{int(X.admit_year.max())}):
  front page, discharge records, drug orders, charted vital signs, laboratory panels, ECG, echocardiography, EEG and imaging reports.
* **Supplementary export** ({int(X.in_supplement.sum()):,} admissions, the export query's index admissions,
  {int(X[X.in_supplement].admit_year.min())}-{int(X[X.in_supplement].admit_year.max())}): admission route, discharge disposition and outcome,
  procedures, non-drug orders, ward transfers with times, admission notes, outpatient and emergency records, further laboratory panels
  (NT-proBNP, pathogen panels and others), ECG intervals, echocardiographic LVEF, monitor heart rate and SpO2.
* `in_supplement` marks coverage. A variable that comes only from the supplementary export is **missing (not "no")** for the other admissions.
* All times are local clock times as recorded (China Standard Time, UTC+8, no daylight saving time).
* Copies of the same record in both exports were verified identical (drug orders, laboratory results, vital-sign rows, point-of-care glucose,
  discharge records) and are kept once. Discharge records that exist only in the supplementary export ({DR_EXTRA}) are added.
* **Repeated rows.** Rows identical in every exported column are one record exported more than once: they are collapsed and the multiplicity is
  kept in `n_source_rows` (drug orders, non-drug orders, procedures, vital signs, examination reports). Point-of-care glucose is the exception:
  its sheet holds only the admission and the value, so equal values are separate measurements and every row is kept.
* Not used: `病案入院诊断` (admission-condition codes, 入院病情, exported without the diagnoses they refer to).
* A few drug-order, non-drug-order and vital-sign rows carry a visit number that is not an inpatient admission of the main front page (QC report,
  section 15). They are kept; they have no admission-level variables and drop out of joins with `admissions`.

## 2. Wards and ICU stays
* Units: `picu_main` = 重症医学科病区 (also 重症医学科 in the ward path text); `picu_west` = 儿童重症监护室(西院) (half- or full-width brackets);
  `other_icu` = adult general ICU (成人GICU/综合病区) or surgical ICU (外科重症监护, SICU); `neonatal_ward` = ward names containing 新生儿 (the export does not
  distinguish neonatal intensive-care beds within these wards); `ward` = all others.
* Ward timeline per admission: admission ward at the admission time, then each transfer in time order. Transfers recorded after hospital discharge
  ({int(TR.flag_after_discharge.sum())}) are not used; transfers recorded up to 24 h before the admission time are placed at admission; a transfer with a
  missing destination ends the current stay and leaves the location unknown until the next transfer.
* An ICU stay is continuous time in one ICU unit. Entry = admission time (admitted directly) or transfer time. Exit = transfer time out; or hospital
  discharge when the discharge ward is the same ICU (discharged or died in the ICU); otherwise **unknown**, with `out_time_upper_bound` = discharge.
  If the discharge ward is an ICU but no transfer into it was recorded, the entry time is unknown (`in_time_lower_bound` = last known time).
* Admissions without an ICU stay from the ward fields or transfer records whose ward path text (转科科别) shows a PICU get one stay with unknown
  times (`source = transfer_path_text_only`); this applies in and outside the supplement.
* Admissions outside the supplement have no transfer records: an ICU admission ward gives a stay from admission, whose exit is the discharge time
  only when the child was discharged from the same ICU and the ward path text does not show leaving an ICU (otherwise unknown); an ICU discharge
  ward reached without a recorded transfer gives a stay that ends at discharge and has an unknown entry.
* The origin ward (转出科室) recorded on a transfer is not used to build the timeline; transfers whose origin disagrees with the timeline are counted
  in the QC report (section 2).
* `icu_episode` joins stays in different ICUs that follow each other directly.

## 3. Organ support (`support_events`, first times in `admission_summary`)
* Times: order times are clock times (to the minute, some to the second); the start time equals the order time for every drug order in this
  export. Procedure times of exactly 00:00:00 carry the date only. `time_precision` is `minute` for a clock time and `date` for a date-only time;
  date-only times are used for a first time only when no timed event exists (`*_first_time_precision = date`) and must not be used to order
  events within a day.
* **Durations.** Drug orders carry no duration in this export: the end time equals the start time or is missing, and infusions are re-ordered about
  daily (hours between successive orders of the same agent: median {VASO_GAP['median_h']}, 90th percentile {VASO_GAP['p90_h']}). Non-drug orders
  (ventilator, NIV, oxygen, RRT and others) are long-term or one-time orders with start and end times; long-term orders end when stopped, at the
  latest at discharge. `support_events.end_time` holds the end time of non-drug orders.
* **Vasoactive infusion**: an order of epinephrine, norepinephrine, dopamine, dobutamine, milrinone or vasopressin by continuous intravenous route
  (routes 静脉输液 or 静脉滴注 and their pump variants), or with route 无 when the order is a compound infusion (a sodium chloride, glucose or water-for-
  injection line was ordered in the same admission at the same minute). Not counted: intravenous push (静脉注射, 静推), endotracheal, inhaled, topical,
  enteral and intramuscular routes, and the unspecified routes 科内治疗 and 遵医嘱; isoproterenol, levosimendan, phenylephrine, terlipressin,
  metaraminol and desmopressin. Posterior pituitary injection (垂体后叶, pituitrin) by infusion is reported separately (`pituitrin_infusion`): it is
  often used to control bleeding. Need for vasoactive medication is one of the cardiovascular items of the Phoenix sepsis criteria (Schlapbach LJ et
  al. JAMA 2024;331:665-674, doi:10.1001/jama.2024.0179, PMID 38245889); the agent list above is this dataset's operational definition.
* **Invasive mechanical ventilation (IMV)**. *Definite*: an intubation or endotracheal-tube order (气管插管, 气管插管术, 特殊方法气管插管术, tube supplies,
  tube care, position adjustments, cuff-pressure monitoring) or procedure code 96.04 or 96.70-96.72. *Probable*: ventilator orders (呼吸机辅助呼吸,
  ventilator settings such as SIMV, tidal volume or PEEP, breathing circuits) in an admission without any non-invasive ventilation evidence.
  *Uncertain* (not counted): ventilator orders only, with NIV evidence.
  First time = earliest timed evidence; ventilator orders count toward the first time only in admissions without NIV evidence. Not IMV:
  non-invasive ventilation, CPAP, nasal prongs, planned procedures (拟…全麻), extubation, suction catheters, tracheal CT and bronchoscopy.
* **NIV**: 无创辅助通气, CPAP, BiPAP, nasal prongs (鼻塞), codes 93.90x. **HFNC**: 高流量. **Oxygen**: oxygen orders, supplies and
  oxygen-concentration settings, code 93.96.
* **RRT**: continuous blood purification, hemodialysis, hemofiltration and peritoneal-dialysis orders and supplies; codes 39.95 (except 39.95x004 and
  39.95x005) and 54.98. Home peritoneal dialysis is `rrt_chronic`; colon dialysis is excluded. **Plasma exchange**: 血浆置换 orders (planned orders
  excluded), codes 39.95x004, 39.95x005, 99.0702; supplies shared by CRRT and plasma exchange are not counted. **Hemoperfusion**: 血液灌流.
  **ECMO**: ECMO orders, code 39.65. **CPR**: 心肺复苏 orders, codes 99.60 and 99.63.

## 4. Outcomes
* **Death**: disposition (离院方式) or outcome (转归情况) = 死亡. The export has no separate death time; `death_time` = discharge time.
* **DAMA**: disposition = 非医嘱离院 (discharge against medical advice).
* **DAMA in a critical condition** (rule-based; wording is read from the final discharge record: the last 400 characters of the hospital course plus
  the discharge condition and discharge instructions of each discharge record; matching within clauses; mentions preceded by
  无/未/否认/不/没有/未见/非 are ignored):
  * A: refusal or withdrawal of treatment (放弃治疗/抢救/进一步…, 拒绝进一步治疗…), discharge on or after removal of life support (带管出院, 呼吸机下出院,
    拔除气管插管后出院, 撤呼吸机, 简易呼吸器), or organ support in the 24 h before discharge (`dama_support_last_24h`): a vasoactive-infusion order
    started in that window; an invasive-ventilation order active in it (intubation/ETT orders, and ventilator orders in admissions without NIV
    evidence, that started before discharge and ended at or after discharge minus 24 h; an order without an end time counts when it started in the
    window); or an IMV procedure with a clock time in the window;
  * B: moribund wording (随时可能死亡或心跳骤停, 临终, 濒死, 脑死亡, 无自主呼吸, 瞳孔散大, 对光反射消失, 深昏迷, 血压测不出或难以维持, 病危, 病情危重,
    危重状态) without improvement wording;
  * D: improvement or stability wording without moribund or no-improvement wording;
  * C: anything else (conflicting or no evidence).
  `dama_critical` = A or B (rule). Category C is not counted by the rule. All DAMA admissions are listed with short excerpts in
  `adjudication_INTERNAL/dama_review_INTERNAL.xlsx`, in this order: C; A with improvement wording; B; other A; D.
{ADJ_BULLET}

## 5. Diagnoses
* `diagnoses_coded`: coded discharge diagnoses (病案出院诊断归一, local ICD-10), unique per admission, in export order (the principal diagnosis is not
  flagged in the source).
* `diagnoses_listed`: diagnosis lists as written, split into items: admission and discharge diagnoses of the discharge records (all admissions),
  preliminary diagnoses of the admission notes and of the outpatient and emergency records, and the front-page discharge diagnoses before coding
  (supplement). Narrative text that sometimes follows a list in the same field is cut off (`narrative_cut`). In the uncoded front-page list, "dh"
  replaces the whole list of some admissions; it is a placeholder and is not kept.

## 6. Laboratory results
* **Timing**: report time only. The research platform holds no specimen collection time (confirmed with the hospital information department).
* One row per report number and analyte. Results are read from the quantitative cell, otherwise from the displayed result. Censored results
  ("<", ">", "≤", "≥", full-width and bracket forms) keep the reporting limit in `value` and "<" or ">" in `censor`. Full-width digits are normalized.
  "No clot" results (未凝集, 无凝固) are missing with `censor` ">" and flag `no_clot`. Placeholders (****, ----, 已报告 and similar) are missing
  (`non_numeric`).
* **Units**: one standard unit per analyte. Conversions: hs-cTn ng/mL x 1000 = ng/L; blood-gas hemoglobin g/dL x 10 = g/L; CSF protein mg/L x 0.001 =
  g/L. A missing unit label is taken as the standard unit (flagged). Any other label that disagrees with the standard unit is relabeled only when the
  row's own reference range lies on the standard-unit scale (within a factor of 2 of the analyte's typical upper reference limit), otherwise the
  value is set to missing.
* **Specimens**: urine or CSF results inside blood panels get prefixed codes (e.g. `urine_ph`, `csf_il10`); a CSF cell count filed in the
  blood-count panel takes the CSF definition (`csf_wbc`, 10^6/L). Blood gases are mostly capillary samples: PO2 does not reflect arterial
  oxygenation.
* Results filed under emergency or outpatient visit numbers are linked to an admission (`linked_admission_id`) when reported during it or in the
  24 h before it.
* hs-cTn: reported in ng/mL with an upper reference limit of 0.01 ng/mL (10 ng/L) throughout. The laboratory sheet does not say troponin I or T;
  the test orders are named: {TROP_TEXT}. Analyte, assay and 99th-percentile URL for each period remain to be confirmed with the laboratory
  [CONFIRM]. CK-MB is an activity assay (U/L), not a mass assay.
* Values outside hard physiological limits are flagged (`outside_physiological_limits`) and kept as reported.
* `lab_qualitative`: qualitative and pathogen tests, one row per report and test; the multiplex sequencing panel has one row per organism
  (organism:reads:load entries; empty "-:-:-" slots are skipped).
* Point-of-care glucose has no time in the export (`glucose_untimed`).

## 7. Vital signs
* Values outside these ranges are set to missing and logged in `vital_signs_cleaning_log`: {'; '.join(f'{k} {v[0]}-{v[1]}' for k, v in LIMITS.items())}.
  A value of 0 is the platform's code for "not recorded". Diastolic pressure at or above systolic pressure is set to missing.
* `heart_rate` = monitor heart rate (supplement admissions) when recorded, otherwise pulse. Rows sharing a timestamp but differing in values are
  separate measurements (`row_seq`); fully identical rows are collapsed.

## 8. Examinations and free text
* `exams` has one row per report: rows that agree in every exported column, including the report text, except the item name are one report (the
  main export lists a CT or MRI report once per examination item; the item names are joined in `exam_name_cn`). It holds times, item names (CT,
  MRI) and numeric measurements (ECG intervals, LVEF). Echocardiography times are dates only; most EEG times are dates only.
* The main export has fewer columns than the supplementary export, so a few distinct reports with the same time and conclusion cannot be told
  apart there. On the supplement admissions, where both exports exist, the main export yields at most {EXAM_UNDER:.1f}% fewer reports per modality
  (QC report, section 11); admissions outside the supplement carry this small undercount.
* Narrative text (discharge summaries, admission notes, outpatient records, imaging, EEG, ECG and echo reports) is not part of the frozen tables; it
  stays in the raw exports, whose checksums are recorded in `manifest.json`. Variables read from narrative text (for example the admission physical
  examination or imaging lesion patterns) belong in a separate, versioned feature layer built from the raw exports.
"""
(DOC / 'definitions.md').write_text(DEFS, encoding='utf-8')

LOADER = '''"""Load and verify the frozen dataset.

Usage (from a notebook in the folder that contains AE_freeze_v1.1/):
    import sys; sys.dont_write_bytecode = True     # keeps the frozen folder free of __pycache__
    sys.path.insert(0, 'AE_freeze_v1.1/code')
    from load_frozen import load, tables, verify
    verify()                                      # every file against manifest.json
    adm = load('admission_summary')               # typed: datetimes, nullable booleans, numbers
The first load of a table writes a typed copy to ./cache_frozen_tables/ (outside the frozen folder); later loads take about a second.
"""
import hashlib
import json
from pathlib import Path
import pandas as pd

ROOT = Path(__file__).resolve().parent.parent
_DICT = None
_MANIFEST = None

def _dictionary():
    global _DICT
    if _DICT is None:
        _DICT = pd.read_csv(ROOT / 'docs' / 'data_dictionary.csv', dtype=str, keep_default_na=False, encoding='utf-8-sig')
    return _DICT

def _manifest():
    global _MANIFEST
    if _MANIFEST is None:
        _MANIFEST = json.loads((ROOT / 'manifest.json').read_text(encoding='utf-8'))
    return _MANIFEST

def _md5(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

def tables():
    return sorted(_dictionary().table.unique())

def verify(verbose=True):
    """Check every file listed in manifest.json against its MD5 checksum and list files the manifest does not know.
    Returns the list of problems (empty when the folder is intact)."""
    man = _manifest()
    files = {v['file']: v['md5'] for v in man['tables'].values()}
    files.update({k: v['md5'] for k, v in man['documents'].items()})
    problems = []
    for rel, md5 in sorted(files.items()):
        p = ROOT / rel
        if not p.exists():
            problems.append(f'missing: {rel}')
        elif _md5(p) != md5:
            problems.append(f'changed: {rel}')
    known = set(files) | {'manifest.json'}
    for p in sorted(ROOT.rglob('*')):
        rel = p.relative_to(ROOT).as_posix()
        if p.is_file() and '__pycache__' not in p.parts and rel not in known:
            problems.append(f'not in manifest: {rel}')
    if verbose:
        print('Frozen dataset intact: every file matches manifest.json.' if not problems else 'Problems:\\n  ' + '\\n  '.join(problems))
    return problems

def load(name, cache=True, cache_dir=None):
    spec = _dictionary()[_dictionary().table == name]
    if spec.empty:
        raise KeyError(f'Unknown table {name!r}; available: {tables()}')
    path = ROOT / 'tables' / f'{name}.csv'
    digest = _manifest()['tables'][name]['md5'][:12]
    cdir = Path(cache_dir) if cache_dir else Path.cwd() / 'cache_frozen_tables'
    cfile = cdir / f'{ROOT.name}_{name}_{digest}_pd{pd.__version__}.pkl'
    if cache and cfile.exists():
        try:
            return pd.read_pickle(cfile)
        except Exception:
            pass
    df = pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[''], encoding='utf-8-sig')
    for col, typ in zip(spec['column'], spec['type']):
        if typ == 'datetime':
            df[col] = pd.to_datetime(df[col], format='%Y-%m-%d %H:%M:%S', errors='coerce')
        elif typ == 'float':
            df[col] = pd.to_numeric(df[col], errors='coerce')
        elif typ == 'int':
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
        elif typ == 'bool':
            df[col] = df[col].map({'True': True, 'False': False}).astype('boolean')
    if cache:
        cdir.mkdir(parents=True, exist_ok=True)
        pd.to_pickle(df, cfile)
    return df
'''
LOADER = LOADER.replace('AE_freeze_v1.1', f'AE_freeze_{FREEZE_VERSION}')
(CODE / 'load_frozen.py').write_text(LOADER, encoding='utf-8')
nb = HERE / NB_NAME
if nb.exists():
    shutil.copy2(nb, CODE / nb.name)

tab_rows = '\n'.join(f"| `{t}` | {MANIFEST['tables'][t]['rows']:,} | {SPEC[t]['desc']} |" for t in OUTPUT)
README = f"""# Acute encephalopathy export: frozen dataset {FREEZE_VERSION}

Frozen research dataset built from the Wuhan Children's Hospital research-platform exports (main export and supplementary export; checksums in
`manifest.json`). It is the single source for analyses of this export: do not edit files in this folder. A correction is made in the build notebook
and released as a new version with its own folder, manifest and reconciliation.

**Internal use only.** The tables carry research patient and admission numbers and real dates. Outputs for publication must not contain them.

## Contents
| Table | Rows | Content |
|---|---|---|
{tab_rows}

* `docs/data_dictionary.xlsx` (and `.csv`): every column with type, unit and description; laboratory analyte list; the mappings used for
  routes, vasoactive agents, drug classes and non-drug order categories.
* `docs/definitions.md`: all derivation rules (ICU stays, organ support, outcomes, laboratory conventions).
* `docs/qc_report.md`: every check run during the build. `docs/reconciliation_v1.md`: differences from freeze v1, explained.
  `docs/reconciliation_@@PREV@@.md`: what changed from the previous release.
* `adjudication_INTERNAL/dama_review_INTERNAL.xlsx`: discharges against medical advice with the rule category, short excerpts of the discharge
  record and the clinician decisions and notes. It contains free text: never share it.
* `code/`: the build notebook and `load_frozen.py`.

## Loading
```python
import sys; sys.dont_write_bytecode = True     # keeps the frozen folder free of __pycache__
sys.path.insert(0, 'AE_freeze_v1.1/code')
from load_frozen import load, verify
verify()                                      # every file against manifest.json
adm = load('admission_summary')               # typed: datetimes, nullable booleans, numbers
labs = load('labs')
```
Plain pandas also works: `pd.read_csv('AE_freeze_v1.1/tables/labs.csv', encoding='utf-8-sig')` (types are then not applied).

## Rebuilding
Put `code/AE_freeze_build_v1_1.ipynb` in a folder with the two raw exports and the adjudication workbook (checksums in `manifest.json`) and
run all cells. If this folder is present there, the notebook leaves it untouched, builds into `AE_freeze_v1.1_rebuild_<date-time>/` and writes
`REBUILD_CHECK.md`, which states whether every table reproduces the released checksum. The review sheet in `adjudication_INTERNAL/` carries the
same decisions and can serve as the adjudication workbook (its checksum differs from the original workbook; the tables do not).

## Rules applied everywhere
1. Nothing is imputed. Unreadable or impossible values are missing with the reason recorded; raw values are kept (`*_raw` columns,
   `vital_signs_cleaning_log`).
2. Unknown is missing, never "no": supplement-only variables are missing when `in_supplement` is False.
3. Laboratory times are report times; the platform has no specimen collection time.
4. Procedure times at 00:00:00 are dates only (`time_precision`).
5. No narrative free text in the tables.

## Key counts
* Admissions {n('admissions'):,} ({int(X.in_supplement.sum()):,} in the supplementary export); patients {n('patients'):,}.
* Admissions with a PICU stay {int(X.any_picu_stay.sum()):,} ({int((X.any_picu_stay & X.in_supplement).sum()):,} in the supplement);
  ICU stays with known entry and exit times {int(IS.timing_complete.sum()):,} of {len(IS):,}.
* Vasoactive infusion {int(X.vasoactive_ever.fillna(False).sum()):,} admissions; invasive ventilation {int(X.imv_ever.fillna(False).sum()):,}
  (supplement admissions); deaths {int(X.death.fillna(False).sum()):,}; discharges against medical advice {int(X.dama.fillna(False).sum()):,}, of which
  critical by rule {int(X.dama_critical.fillna(False).sum()):,} (rule category C {int(X.dama_critical_needs_review.fillna(False).sum()):,}).
* Critical DAMA by the final, clinician-adjudicated label {int(X.dama_critical_final.fillna(False).sum()):,}; death or critical DAMA (final label)
  {int(X.death_or_dama_critical_final.fillna(False).sum()):,}.

## Open items
* hs-cTn analyte, assay and 99th-percentile upper reference limit [CONFIRM with the laboratory]; the test-order names listed in
  `docs/definitions.md` (section 6) are the first evidence.
* DAMA records marked unsure by the reviewer ({_ndec.get('unsure', 0)}) keep the rule label; a second reviewer can resolve them by filling the decision
  column of the review sheet, which is then released as a new version.
* Point-of-care glucose has no time; blood gases are mostly capillary.
* Variables read from narrative text (admission physical examination, symptom onset, imaging, EEG, ECG and echo findings) are not part of the
  freeze; build them as a separate, versioned feature layer when a study needs them (see `docs/reconciliation_v1.md`).
"""
README = README.replace('AE_freeze_build_v1_1.ipynb', NB_NAME).replace('AE_freeze_v1.1', f'AE_freeze_{FREEZE_VERSION}').replace('@@PREV@@', 'v1.1')
README += f"""
## Changes from v1.1
* New input: the clinician adjudication of discharges against medical advice (workbook name and MD5 in `manifest.json`).
* `admission_summary`: new columns `dama_review_decision`, `dama_critical_final`, `dama_critical_final_source` and
  `death_or_dama_critical_final`; the rule columns are unchanged.
* {('Every other table, and every v1.1 column of admission_summary, is identical to v1.1 (`docs/reconciliation_v1.1.md`).' if REC2_OK else
   'The comparison with v1.1 could not confirm that the other tables are unchanged: see `docs/reconciliation_v1.1.md`.')}
* Documents: definitions (section 4), data dictionary, QC report (sections 0, 13 and 16), this README, and the review sheet (decisions prefilled).
"""
(OUT / 'README.md').write_text(README, encoding='utf-8')

for f in sorted(list(DOC.glob('*')) + list(CODE.glob('*')) + list(ADJ.glob('*')) + [OUT / 'README.md']):
    if f.is_file():
        MANIFEST['documents'][f.relative_to(OUT).as_posix()] = {'bytes': f.stat().st_size, 'md5': md5_file(f)}
MANIFEST['build_seconds'] = round(time.time() - T0, 1)
(OUT / 'manifest.json').write_text(json.dumps(MANIFEST, ensure_ascii=False, indent=1), encoding='utf-8')
if REFERENCE:
    ref_t = REFERENCE.get('tables', {})
    differ = [t for t in MANIFEST['tables'] if ref_t.get(t, {}).get('md5') != MANIFEST['tables'][t]['md5']]
    missing = [t for t in ref_t if t not in MANIFEST['tables']]
    same_src = sorted(x.get('md5') for x in REFERENCE.get('sources', [])) == sorted(w['md5'] for w in WORKBOOKS + ADJ_INPUTS)
    verdict = ('Every table reproduces the checksum of the released freeze.' if not differ and not missing else
               f'Tables whose checksum differs: {differ or "none"}; tables of the release missing here: {missing or "none"}.')
    (OUT / 'REBUILD_CHECK.md').write_text('\n'.join([
        f'# Rebuild check against the released freeze {FREEZE_VERSION}', '',
        f"Released freeze built {REFERENCE.get('built')} with {REFERENCE.get('software')}; this rebuild {MANIFEST['built']} with {MANIFEST['software']}.", '',
        f'Inputs (raw exports and adjudication workbook) identical to those of the release (MD5): {same_src}.', '', verdict, '']), encoding='utf-8')
    log('Rebuild check:', verdict)
log(f'Frozen dataset {FREEZE_VERSION} written to {OUT} in {MANIFEST["build_seconds"]} s.')
log('QC:', qc_df.status.value_counts().to_dict())